# CD4 annotation
- Load data
- Introduce catpat annotations
- Select CD4 T cells
- Remove doublets by conventional QC
- Removing CD8 T cells (Mostly CD4/CD8 T cells)
- Select highly variable genes
- PCR, neighours and **over**clustering

In [ ]:
import anndata as ad
import scanpy as sc
import scanpy.external as sce
import scirpy as ir
import pandas as pd
import numpy as np
import seaborn as sb
import os
import muon as mu
from scipy.stats import binomtest
from scipy.stats import mannwhitneyu

# Helps to define plotting area
import matplotlib.pyplot as plt
from matplotlib import rcParams 
from matplotlib.pyplot import rc_context
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors
from matplotlib import colors
# from matplotlib_venn import venn2
from scipy.stats import spearmanr
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap

from bioservices import KEGG
from gprofiler import GProfiler

## For making text editable in .pdfs once exported use cairo backend instead of agg
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

# set seed
np.random.seed(44)

#display options
pd.options.display.max_rows = 999
pd.options.display.max_columns = 999

In [ ]:
# # Set figure params
sc.settings.set_figure_params(figsize=(4.3, 3))

# Loading data

In [ ]:
sdata = mu.read(filename="../../04_data_objects/06_preprocessed_v2/2026_6_main_dataset_clustered_annotated.h5mu")

# Fix catpat (assign NIC, IC and MS annotations)

In [ ]:
#Group annotation
patient_list = ["Pt11", "Pt12", "Pt14", "Pt17", "Pt20"]
control_list = ["Pt1", "Pt8", "Pt18", "Pt19", "Pt21"]
IC_list = ["Pt5", "Pt4"]

#Match the lists defined above to define "RRMS" and "Control"
sdata["gex"].obs["catpat"] = pd.Categorical(np.where(sdata["gex"].obs["donor"].isin(patient_list), "MS", np.where(sdata["gex"].obs["donor"].isin(control_list), "NIC", np.where(sdata["gex"].obs["donor"].isin(IC_list), "IC", "Error"))))


# Select CD4 T cells

In [ ]:
mu.pp.filter_obs(sdata["gex"], "final_ann", lambda x: (x.isin(["CD4 T cells"])))  
sdata.update()

In [ ]:
sdata

# Removing doublets based on conventional QC

In [ ]:
# Plot the QC metrics, and decide a cutoff %%%%%%%%%%%%
# Number of Axes & plot size
ncols = 4
nrows = 1
figsize = 4
wspace = 1

fig, axs = plt.subplots(
    nrows=nrows,
    ncols=ncols,
    figsize=(ncols * figsize + figsize * wspace * (ncols - 1), nrows * figsize),
)

# Plot individual values  
sc.pl.violin(
    sdata["gex"], 
    keys="n_genes_by_counts",
    groupby="Compartment",
    jitter=0.4,
    multi_panel=False,
    show=False,
    ax=axs[0]
)
sc.pl.violin(
    sdata["gex"], 
    keys="total_counts",
    groupby="Compartment",
    jitter=0.4,
    multi_panel=False,
    show=False,
    ax=axs[1]
)
sc.pl.violin(
    sdata["gex"], 
    keys="pct_counts_mt",
    groupby="Compartment",
    jitter=0.4,
    multi_panel=False,
    show=False,
    ax=axs[2]
)

sc.pl.violin(
    sdata["gex"], 
    keys="scrublet_score",
    groupby="Compartment",
    jitter=0.4,
    multi_panel=False,
    show=False,
    ax=axs[3]
)


#n_genes_total (i.e. number of genes per cell)
axs[0].axhline(y=200, linewidth = 1.5, color = "black", linestyle = "--")
axs[0].axhline(y=3000, linewidth = 1.5, color = "black", linestyle = "--")

#number of total reads 
axs[1].axhline(y=400, linewidth = 1.5, color = "black", linestyle = "--")
axs[1].axhline(y=4000, linewidth = 1.5, color = "black", linestyle = "--")

#mitochondrial gene expression
axs[2].axhline(y=10, linewidth = 1.5, color = "black", linestyle = "--")

#mitochondrial gene expression
axs[3].axhline(y=0.5, linewidth = 1.5, color = "black", linestyle = "--")

# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%

In [ ]:
# ## filter to remove doublets
mu.pp.filter_obs(sdata["gex"], 'n_genes_by_counts', lambda x: (x < 3000)) 
mu.pp.filter_obs(sdata["gex"], 'total_counts', lambda x: (x < 4000))
mu.pp.filter_obs(sdata["gex"], 'scrublet_score', lambda x: (x < 0.5))
sdata.update()

In [ ]:
# Compute medians per group
medians = sdata["gex"].obs.groupby("chain_pairing")["scrublet_score"].median()

# Sort groups by median (ascending)
sorted_groups = medians.sort_values().index.tolist()

# --- Plot violin with small boxplot inside ---
with plt.rc_context({"figure.figsize": (6, 4)}):
    ax = sc.pl.violin(
        sdata["gex"],
        keys="scrublet_score",
        groupby="chain_pairing",
        order=sorted_groups,  # sorted by median
        multi_panel=False,
        rotation=45,
        show=False,
        stripplot=True,      # no individual dots
        inner="box"           # small boxplot inside violin
    )

# Rotate x-axis labels
plt.xticks(rotation=45, ha='right')
plt.ylabel("scrublet_score")
plt.legend(loc='upper right', bbox_to_anchor=(1.35, 1))

# Rasterize violin and box shapes for PDF
for col in ax.collections:
    col.set_rasterized(True)

# Save figure
plt.tight_layout()
plt.savefig(
    "../../05_plots/11_CD8annotation/1_Violin_chain_pairing.pdf",
    bbox_inches="tight",
    dpi=300
)
plt.show()


#Cluster 28 do have more counts - but this cluster has a very low fraction of two-chain TCRs. So I think it's okay. 

## Summary

In [ ]:
df_total = sdata["gex"].obs.groupby(["Compartment", "catpat", "donor"], observed=True).size()
df_total

## Move CITEseq annotations into obs

In [ ]:
# Proteins to transfer
proteins = [
    "CD45RA_adt",
    "CD4_adt",
    "CD8_adt",
    "CD62L_adt",
    "CD95_adt",
]

# Extract protein expression matrix
prot_df = pd.DataFrame(
    sdata["prot"][:, proteins].X.toarray()
    if hasattr(sdata["prot"].X, "toarray") else sdata["prot"][:, proteins].X,
    index=sdata["prot"].obs_names,
    columns=[f"prot_{p}" for p in proteins]
)

# Transfer into gex.obs
sdata["gex"].obs = sdata["gex"].obs.join(prot_df)

In [ ]:
#Also transfer the donor-dependent assignments of surface differentiation status. 
sdata["gex"].obs["Differentiation"] = sdata["prot"].obs["Differentiation"].copy()

# Removing CD8 T cells (Mostly CD4/CD8 T cells) 
While voting, some CD4 T cells expressing prot_CD8 or CD8A/B were included as CD4 T cells, because their clonotype exists primarily among clusters with strong CD4 expression. However, in the interest of simplifying our analysis and removing all potential sources of noise, we exclude cells with CD8A/B and prot_CD8 expression. 

In [ ]:
### %%% Protein expression measured by a CD8 CITEseq antibody %%%%
ax = sc.pl.scatter(
    sdata['gex'],
    x="prot_CD4_adt",
    y="prot_CD8_adt",
    size=40,
    color="cd4cd8_cs_assignment",
    show=False
)

ax.axhline(y=11, color='black', linestyle='--', lw=1)
ax.axvline(x=9, color='black', linestyle='--', lw=1)
plt.ylim(-10, 45)

plt.show()

In [ ]:
## Do I get any CD4 T cells in my CD8 gate?
obs_df = sdata["gex"].obs.copy()
obs_df[(obs_df["prot_CD8_adt"] > 10.5)][["cd4cd8_cs_assignment"]].value_counts()

#Threshold 10.5 seems acceptable as a one-threshold cutoff

In [ ]:
## The CD8 T cell outliers in black are all from CSF of patient, where I elected not to use CITEseq information because of the cell loss associated with staining CSF cells. 

obs_df = sdata["gex"].obs.copy()
obs_df[(obs_df["prot_CD8_adt"] > 10.5) & (obs_df["prot_CD4_adt"] < 9)]

### Annotate CD8_adt expression

In [ ]:
sdata["gex"].obs["CD8_adt_expression"] = pd.Categorical(np.where(sdata["gex"].obs["prot_CD8_adt"] > 10.5, "CD8_adt", "negative"))

### Annotate CD8A expression

In [ ]:
# Get the index of the CD4 and CD8 genes
cd8_idx = np.where(sdata["gex"].var_names == "CD8A")[0][0]

# Get expression values
cd8_expr = sdata["gex"].X[:, cd8_idx]

# If X is sparse, convert to array and flatten
cd8_expr = cd8_expr.toarray().flatten() if not isinstance(cd8_expr, np.ndarray) else cd8_expr.flatten()

# Create boolean mask for CD4_gex_assignment: CD4 > 1 
cd8_gex_positive = (cd8_expr > 0.1)

# Assign category: "CD4" or "other" for CD4_gex_assignment
sdata["gex"].obs["CD8A_expr"] = pd.Categorical(
             np.where(cd8_gex_positive, "CD8A", "negative"))

### Annotate CD8B expression too

In [ ]:
# Get the index of the CD4 and CD8 genes
cd8_idx = np.where(sdata["gex"].var_names == "CD8B")[0][0]

# Get expression values
cd8_expr = sdata["gex"].X[:, cd8_idx]

# If X is sparse, convert to array and flatten
cd8_expr = cd8_expr.toarray().flatten() if not isinstance(cd8_expr, np.ndarray) else cd8_expr.flatten()

# Create boolean mask for CD4_gex_assignment: CD4 > 1 
cd8_gex_positive = (cd8_expr > 0.1)

# Assign category: "CD4" or "other" for CD4_gex_assignment
sdata["gex"].obs["CD8B_expr"] = pd.Categorical(
             np.where(cd8_gex_positive, "CD8B", "negative"))

### Visualize CD8A/CD8B

In [ ]:
color_map = {
    "CD8B": "#1f77b4",       
    "negative" : "#7f7f7f"
    
}

# Ensure category order is stable
sdata["gex"].obs["CD8B_expr"] = sdata["gex"].obs["CD8B_expr"].cat.set_categories(
    ["CD8B", "negative"]
)

# Register colors in AnnData
sdata["gex"].uns["CD8B_expr_colors"] = [
    color_map[c] for c in sdata["gex"].obs["CD8B_expr"].cat.categories
]


ax = sc.pl.scatter(
    sdata['gex'],
    x="CD4",
    y="CD8B",
    size=40,
    color="CD8B_expr",
    show=False
)

ax.axhline(y=0.1, color='black', linestyle='--', lw=1)
ax.axvline(x=0.1, color='black', linestyle='--', lw=1)
plt.ylim(-0.5, 5)
plt.show()

In [ ]:
color_map = {
    "CD8A": "#1f77b4",       
    "negative" : "#7f7f7f"
    
}

# Ensure category order is stable
sdata["gex"].obs["CD8A_expr"] = sdata["gex"].obs["CD8A_expr"].cat.set_categories(
    ["CD8A", "negative"]
)

# Register colors in AnnData
sdata["gex"].uns["CD8A_expr_colors"] = [
    color_map[c] for c in sdata["gex"].obs["CD8A_expr"].cat.categories
]


ax = sc.pl.scatter(
    sdata['gex'],
    x="CD4",
    y="CD8A",
    size=40,
    color="CD8A_expr",
    show=False
)

ax.axhline(y=0.1, color='black', linestyle='--', lw=1)
ax.axvline(x=0.1, color='black', linestyle='--', lw=1)
plt.ylim(-0.5, 5)
plt.show()

## How many cells am I excluding? 
Answer: 1297

In [ ]:
obs_df = sdata["gex"].obs.copy()
obs_df = obs_df[["CD8A_expr", "CD8B_expr", "cd4cd8_cs_assignment", "CD8_adt_expression"]]

obs_df = obs_df[(obs_df["CD8A_expr"] == "CD8A") | (obs_df["CD8B_expr"] == "CD8B") | (obs_df["cd4cd8_cs_assignment"].isin(["CD8_adt", "CD4_adt/CD8_adt"])) | (obs_df["CD8_adt_expression"] == "CD8_adt")]
obs_df.value_counts().sum()


In [ ]:
# Filter
mu.pp.filter_obs(sdata["gex"], 'cd4cd8_cs_assignment', lambda x: (~x.isin(["CD8_adt", "CD4/CD8_adt"])))
mu.pp.filter_obs(sdata["gex"], 'CD8B_expr', lambda x: (x != "CD8B"))
mu.pp.filter_obs(sdata["gex"], 'CD8A_expr', lambda x: (x != "CD8A"))
mu.pp.filter_obs(sdata["gex"], 'CD8_adt_expression', lambda x: (x != "CD8_adt"))
sdata.update()

In [ ]:
sdata["gex"].obs[["CD8B_expr"]].value_counts()

# Preprocess GEX

In [ ]:
# ---
# Remove genes that are only present in 1 cell
# ---
mu.pp.filter_var(sdata["gex"], 'n_cells_by_counts', lambda x: x > 1)
sdata.update()

# ---
# Allign the modalities emphasizing cells in gex.
#   Make certain that the cells in airr and prot modalities are the same cells as are present in gex.
# ---
#Make sure only the indexes recorded in GEX are used in the o*ther modalities too
index_list = sdata["gex"].obs.index.tolist()

## define a cell index
sdata["airr"].obs["cell_index"] = sdata["airr"].obs.index
sdata["prot"].obs["cell_index"] = sdata["prot"].obs.index

## ".isin" using indexes does not seem to work on a global level, but if I annotate a column and then filter in place, maybe then it works. 
sdata["airr"].obs["filter_index"] = pd.Categorical(np.where(sdata["airr"].obs["cell_index"].isin(index_list), "GEX_present", "GEX_absent"))
sdata["prot"].obs["filter_index"] = pd.Categorical(np.where(sdata["prot"].obs["cell_index"].isin(index_list), "GEX_present", "GEX_absent"))

#Enforce that sdata["airr"].obs and sdata["prot"].obs only contain information on those indexes that exists in GEX
mu.pp.filter_obs(sdata["airr"], 'filter_index', lambda x: x == "GEX_present")
mu.pp.filter_obs(sdata["prot"], 'filter_index', lambda x: x == "GEX_present")
sdata.update()

# ---
# When the .var object has been filtered, then you need to renormalize
# ---
sdata["gex"].X = sdata["gex"].layers["counts"].copy()
sdata["gex"].uns.pop("log1p", None) #Removes log1P if it exists
sc.pp.normalize_total(sdata["gex"], target_sum=1e4, exclude_highly_expressed=False)   #1e4 is important. Do not change
sc.pp.log1p(sdata["gex"])

## Calculate clone_size

In [ ]:
## Calculate clone_size = number of rows per clone_id
sdata["gex"].obs["clone_size"] = sdata["gex"].obs.groupby("clone_handle", observed=True)["clone_handle"].transform("count")

#Mask "NaN" containing clones
sdata["gex"].obs["clone_size"] = np.where(sdata["gex"].obs["clone_handle"].str.contains("nan"), 0, sdata["gex"].obs["clone_size"])

## Share labels with airr
sdata["airr"].obs["clone_size"] = sdata["gex"].obs["clone_size"].copy()

# Select highly variable genes
* https://scanpy.readthedocs.io/en/stable/generated/scanpy.pp.highly_variable_genes.html
* https://scanpy.readthedocs.io/en/stable/api/generated/scanpy.pp.neighbors.html
* https://www.sc-best-practices.org/cellular_structure/integration.html

Data integration when experimental parameters vary. Ours experimental parameters are the same for each batch (apart from Tcsf being fresh cells, and Tpb being rested). We prefer to work with unintegrated data wherever possible to preserve differences between samples. The link describes the re-normalization of of genes, selection of hvg, ect. we need to do the same. 

### Batch aware highly variable genes using default functions

* Source: https://scanpy.readthedocs.io/en/stable/generated/scanpy.pp.highly_variable_genes.html
  
_For flavor='seurat_v3_paper', genes are first sorted by the number of batches a gene is a HVG, with ties broken by the median (across batches) rank_

* Source: https://scanpy.readthedocs.io/en/stable/generated/scanpy.pp.highly_variable_genes.htm

**batch_key**
str | None (default: None)
If specified, highly-variable genes are selected within each batch separately and merged. This simple process avoids the selection of batch-specific genes and acts as a lightweight batch correction method. For **all flavors**, except seurat_v3, **genes are first sorted by how many batches they are a HVG.** For dispersion-based flavors ties are broken by normalized dispersion. For flavor = 'seurat_v3_paper', ties are broken by the median (across batches) rank based on within-batch normalized variance.

**For flavor="seruat" (default)**: Effector genes of known function are a big part of the HVGs with high mean expression and dispersion values fx, GZMK and PRF are both above 5 in mean expression value. Many low dispersion genes are unknown of B / macrophage genes. I will prioritize a method that includes many known T cell genes

In [ ]:
#Removes columns of the var table. This is to rebuild the HVG table. 
sdata["gex"].var = sdata["gex"].var[[]]

## What is a batch?
## We want a donor aware + batch aware intergration (donor + batch)
sdata["gex"].obs["batch_handle"] = sdata["gex"].obs["batch"].astype(str) + "_" + sdata["gex"].obs["donor"].astype(str)

## Default highly variable genes
sc.pp.highly_variable_genes(sdata["gex"], 
                            n_top_genes=6500, #!! 5000 genes creates a spread out umap. 7500 HVGs creates a too dense umap.
                            flavor='seurat',  #This mimics select intergration features
                            batch_key="batch_handle",   #If a batch was "batch" - then the CITEseq multiplexed instances would immediate mark donor-specific expression of AC004448.2 and OOEP fx. this is not intended. 
                            span=1)

In [ ]:
Export_HVG = sdata["gex"].var[sdata["gex"].var["highly_variable"]==True].sort_values(by="highly_variable_nbatches", ascending = True)
print(len(Export_HVG))
Export_HVG.to_csv("..\\..\\06_csv_outputs\\HVG\\CD8_HVGs_export_before_TCR_filter.csv")

## Remove TCR genes from the list of highly variable genes

In [ ]:
#Bring gene names from index into a column
sdata["gex"].var = sdata["gex"].var.reset_index()

# Optionally rename the new column
sdata["gex"].var = sdata["gex"].var.rename(columns={"index": "genes"})

In [ ]:
# Rename the highly_varaible_intersection cells according to an ifelse statement
tcr_gene_prefixs = ['TRAV', 'TRAJ', 'TRBV', "TRBD", "TRBJ"] 

pattern = r"^(" + "|".join(tcr_gene_prefixs) + ")"
mask = sdata["gex"].var["genes"].str.contains(pattern, regex=True)
masked_genes = sdata["gex"].var[mask]["genes"].tolist() #Works - it's giving me the exact genes I want except the D-region genes (although there is only 2? I will ask Kilian about it. Lea says there is something like that with mice, that htey don't express it)

#Modify the initial list of highly variable genes to exclude TCR genes
sdata["gex"].var["highly_variable"] = np.where(sdata["gex"].var["genes"].isin(masked_genes), False, sdata["gex"].var["highly_variable"])

# Return the genes to index
sdata["gex"].var = sdata["gex"].var.set_index(["genes"], drop=True)

In [ ]:
Export_HVG = sdata["gex"].var[sdata["gex"].var["highly_variable"]==True].sort_values(by="highly_variable_nbatches", ascending = True)
print(len(Export_HVG))
Export_HVG.to_csv("..\\..\\06_csv_outputs\\HVG\\CD8_HVGs_export_after_TCR_filter.csv")

# PCR, neighours and **over**clustering
### Calculate neighbors
https://scanpy.readthedocs.io/en/stable/api/generated/scanpy.pp.neighbors.html
- n_neighbors int (default: 15)

    The size of local neighborhood (in terms of number of neighboring data points) used for manifold approximation. **Larger values result in more global views of the manifold**, while smaller values result in more local data being preserved. In general values should be in the range 2 to 100. If knn is True, number of nearest neighbors to be searched. If knn is False, a Gaussian kernel width is set to the distance of the n_neighbors neighbor.

#### General advice: Larger datasets benefit from larger n_pcs values! Source: https://satijalab.org/seurat/archive/v3.0/mca
- Here they use n_pcs = 75 for 242.000 cells, and they are generally very lax about where to set the cutoff on the elbow plot.

Advice: Try working with larger n_pcs. You can decrease the n_neighbors to 10, if you want to reproduce Seurats behavior

In [ ]:
# Run PCA
sc.pp.pca(sdata["gex"], svd_solver="arpack", use_highly_variable=True, n_comps=60)

In [ ]:
# Visualize pca
sc.pl.pca_scatter(sdata["gex"], color=["total_counts", "GZMK", "GZMB", "GZMA"])

In [ ]:
sc.pl.pca_variance_ratio(sdata["gex"], n_pcs=60, log=True) #Log tranformeD. 

In [ ]:
# Set n_pcs based on a visual impression
sc.pp.neighbors(sdata["gex"], n_pcs=40)

In [ ]:
sc.tl.leiden(sdata["gex"], key_added="leiden_4", resolution=4.0)

## Calculate umap

In [ ]:
sc.tl.umap(sdata["gex"], spread=1)

# Reporting on T-T doublets

In [ ]:
confusion_matrix = pd.crosstab(
    sdata["gex"].obs["leiden_4"],
    sdata["gex"].obs["chain_pairing"],
    rownames=['leiden_4'],
    colnames=['chain_pairing']
)


# Calculate row-wise percentages for the "two full chains" column
confusion_matrix["percent_two full chains"] = (
    confusion_matrix["two full chains"] / confusion_matrix.sum(axis=1)
) * 100

confusion_matrix


## Doublet QC plots

In [ ]:
# Compute medians per group
medians = sdata["gex"].obs.groupby("leiden_4")["n_genes_by_counts"].median()

# Sort groups by median (ascending)
sorted_groups = medians.sort_values().index.tolist()

# --- Plot violin with small boxplot inside ---
with plt.rc_context({"figure.figsize": (10, 4)}):
    ax = sc.pl.violin(
        sdata["gex"],
        keys="n_genes_by_counts",
        groupby="leiden_4",
        order=sorted_groups,  # sorted by median
        multi_panel=False,
        rotation=45,
        show=False,
        stripplot=False,      # no individual dots
        inner="box"           # small boxplot inside violin
    )

# Rotate x-axis labels
plt.xticks(rotation=45, ha='right')
plt.ylabel("n_genes_by_counts")

# Rasterize violin and box shapes for PDF
for col in ax.collections:
    col.set_rasterized(True)

# Save figure
plt.tight_layout()
# plt.savefig(
#     "..\\..\\05_plots\\08_AfterAntwerp\\88_Total_reads_L2_box.pdf",
#     bbox_inches="tight",
#     dpi=300
# )
plt.show()


#Cluster 14 and 18 has a bit mor ecounts than average

In [ ]:
# Compute medians per group
medians = sdata["gex"].obs.groupby("leiden_4")["scrublet_score"].median()

# Sort groups by median (ascending)
sorted_groups = medians.sort_values().index.tolist()

# --- Plot violin with small boxplot inside ---
with plt.rc_context({"figure.figsize": (10, 4)}):
    ax = sc.pl.violin(
        sdata["gex"],
        keys="scrublet_score",
        groupby="leiden_4",
        order=sorted_groups,  # sorted by median
        multi_panel=False,
        rotation=45,
        show=False,
        stripplot=False,      # no individual dots
        inner="box"           # small boxplot inside violin
    )

# Rotate x-axis labels
plt.xticks(rotation=45, ha='right')
plt.ylabel("scrublet_score")

# Rasterize violin and box shapes for PDF
for col in ax.collections:
    col.set_rasterized(True)

# Save figure
plt.tight_layout()
# plt.savefig(
#     "..\\..\\05_plots\\08_AfterAntwerp\\88_Total_reads_L2_box.pdf",
#     bbox_inches="tight",
#     dpi=300
# )
plt.show()


#Cluster 21 and 33 have slightly larger scrublet scores, but nothing noticable

In [ ]:
sc.pl.umap(sdata["gex"], color=["leiden_4"], groups=["14"], size = 10, legend_fontweight= 'bold', frameon= False , vmin=0, legend_fontoutline=2, show=True)

## Return cell numbers

In [ ]:
## How many cells
sdata["gex"].obs.groupby(["final_ann"]).size()

In [ ]:
## How many cells
sdata["gex"].obs.groupby(["chain_pairing"]).size()

# Remove doublet clusters (if any)

In [ ]:
### Exclude doublets 
# mu.pp.filter_obs(sdata["gex"], "leiden_4", lambda x: (x != "40")) 

#Remove unused cateorgies
sdata["gex"].obs["leiden_4"] = sdata["gex"].obs["leiden_4"].cat.remove_unused_categories()
sdata.update()

# Generate the final umap/clusters

In [ ]:
#Removes columns of the var table. This is to rebuild the HVG table. 
sdata["gex"].var = sdata["gex"].var[[]]

## What is a batch?
## We want a donor aware + batch aware intergration (donor + batch)
sdata["gex"].obs["batch_handle"] = sdata["gex"].obs["batch"].astype(str) + "_" + sdata["gex"].obs["donor"].astype(str)

## Default highly variable genes
sc.pp.highly_variable_genes(sdata["gex"], 
                            n_top_genes=6500, #!! 5000 genes creates a spread out umap. 7500 HVGs creates a more dense umap.
                            flavor='seurat',  #This mimics select intergration features
                            batch_key="batch_handle",   #If a batch was "batch" - then the CITEseq multiplexed instances would immediate mark donor-specific expression of AC004448.2 and OOEP fx. this is not intended. 
                            span=1)

# Export_HVG = sdata["gex"].var[sdata["gex"].var["highly_variable"]==True].sort_values(by="highly_variable_nbatches", ascending = True)
# print(len(Export_HVG))
# Export_HVG.to_csv("..\\..\\06_csv_outputs\\HVG\\CD4_HVGs_export_before_TCR_filter.csv")

#Bring gene names from index into a column
sdata["gex"].var = sdata["gex"].var.reset_index()

# Optionally rename the new column
sdata["gex"].var = sdata["gex"].var.rename(columns={"index": "genes"})

# Rename the highly_varaible_intersection cells according to an ifelse statement
tcr_gene_prefixs = ['TRAV', 'TRAJ', 'TRBV', "TRBD", "TRBJ"] 

pattern = r"^(" + "|".join(tcr_gene_prefixs) + ")"
mask = sdata["gex"].var["genes"].str.contains(pattern, regex=True)
masked_genes = sdata["gex"].var[mask]["genes"].tolist() #Works - it's giving me the exact genes I want except the D-region genes (although there is only 2? I will ask Kilian about it. Lea says there is something like that with mice, that htey don't express it)

#Modify the initial list of highly variable genes to exclude TCR genes
sdata["gex"].var["highly_variable"] = np.where(sdata["gex"].var["genes"].isin(masked_genes), False, sdata["gex"].var["highly_variable"])

# Return the genes to index
sdata["gex"].var = sdata["gex"].var.set_index(["genes"], drop=True)

Export_HVG = sdata["gex"].var[sdata["gex"].var["highly_variable"]==True].sort_values(by="highly_variable_nbatches", ascending = True)
print(len(Export_HVG))
Export_HVG.to_csv("..\\..\\06_csv_outputs\\HVG\\CD4_HVGs_export_after_TCR_filter.csv")

#### Redo PCA, umap and leiden

In [ ]:
# Run PCA
sc.pp.pca(sdata["gex"], svd_solver="arpack", use_highly_variable=True, n_comps=60)
sc.pl.pca_variance_ratio(sdata["gex"], n_pcs=60, log=True) #Log tranformeD. 


In [ ]:
# Set n_pcs based on a visual impression
sc.pp.neighbors(sdata["gex"], n_pcs=40)

#umap
sc.tl.umap(sdata["gex"], spread=1)

In [ ]:
# sc.tl.leiden(sdata["gex"], key_added="leiden_1", resolution=1) 
# sc.tl.leiden(sdata["gex"], key_added="leiden_1.1", resolution=1.1) 
# sc.tl.leiden(sdata["gex"], key_added="leiden_1.2", resolution=1.2) 
# sc.tl.leiden(sdata["gex"], key_added="leiden_1.3", resolution=1.3) 
# sc.tl.leiden(sdata["gex"], key_added="leiden_1.4", resolution=1.4) 
# sc.tl.leiden(sdata["gex"], key_added="leiden_1.5", resolution=1.5)
# sc.tl.leiden(sdata["gex"], key_added="leiden_1.6", resolution=1.6) 
# sc.tl.leiden(sdata["gex"], key_added="leiden_1.7", resolution=1.7) 
sc.tl.leiden(sdata["gex"], key_added="leiden_1.8", resolution=1.8)      #Works well - the outlier clusters are clearly marked in both CSF and PB without too many clusters to account for. 
# sc.tl.leiden(sdata["gex"], key_added="leiden_1.9", resolution=1.9)      
# sc.tl.leiden(sdata["gex"], key_added="leiden_2", resolution=2.0)    
# sc.tl.leiden(sdata["gex"], key_added="leiden_1.8", resolution=2.1)  
# sc.tl.leiden(sdata["gex"], key_added="leiden_2.2", resolution=2.2)
# sc.tl.leiden(sdata["gex"], key_added="leiden_2.3", resolution=2.3)
# sc.tl.leiden(sdata["gex"], key_added="leiden_2.4", resolution=2.4)
# sc.tl.leiden(sdata["gex"], key_added="leiden_2.5", resolution=2.5)

In [ ]:
sc.pl.umap(sdata["gex"], color=["prot_CD45RA_adt", "prot_CD62L_adt", "prot_CD95_adt"], size = 5, legend_fontweight= 'bold', frameon= False , legend_fontoutline=2, show=True, legend_loc = "on data", cmap="RdBu_r")


In [ ]:
sc.pl.umap(sdata["gex"], color=[#"leiden_1.3", "leiden_1.4", "leiden_1.5",  "leiden_1.6",  "leiden_1.7","leiden_1.9", "leiden_2", "leiden_1.8"
    "leiden_1.8"], size = 5, legend_fontweight= 'bold', frameon= False , legend_fontoutline=2, show=True, legend_loc = "on data")


# Return umaps

In [ ]:
color_dict = {
    # --- Pastel colors / TN ---
    "0":  "#dbe9f6",  # very light blue
    "4":  "#c7e9c0",  # pastel green
    "5":  "#e2d9f3",  # light lavender
    "6": "#fdd0a2",  # pastel peach

    # --- TCM-like: green ---
    '8':  '#a1d99b',   # light green
    '10': '#238b45',   # strong green

    # --- TEM-like: orange → red ---
    '3':  '#FFB000',   # neon-like bright orange
    '9':  '#fdae6b',  # strong orange
    '17': '#e6550d',   # dark orange
    '18': '#8B0000',   # deep red
        
    # --- Treg and CTL ---
    '15': '#8c564b',

    # # # --- TCSF ---
    # '1':  '#1f77b4',
    # '2':  '#9edae5',
    # '7':  '#17becf',
    # '11': '#ffbb78',
    # '12': '#ff7f0e',
    # '14': '#9467bd',
    # '19': "#3BFD16",  # neon'#aec7e8',
    
    # --- TCSF: blue / purple-blue gradient ---
    '1':  '#6baed6',   # light blue
    '2':  '#3182bd',   # medium blue
    '7':  '#9ecae1',   # pale blue
    '11': '#9e9ac8',   # light purple
    '12': '#756bb1',   # purple
    '14': '#54278f',   # dark purple
    '19': '#00E5CC',   # neon teal

    # --- TAct ---
    '13': '#e377c2',
    '16': '#f7b6d2', 
    '20': '#7f7f7f',
}

## Overwrite the leiden_1.7_colors
cats = sdata["gex"].obs["leiden_1.8"].cat.categories
sdata["gex"].uns["leiden_1.8_colors"] = [color_dict[c] for c in cats]

In [ ]:
sc.settings.set_figure_params(figsize=(4.3, 4))
fig_umap = sc.pl.umap(sdata["gex"], color=["leiden_1.8"], size = 5, legend_fontweight= 'bold', frameon= False ,legend_fontoutline=2, show=False, legend_loc="on data", cmap="RdBu_r", return_fig=True)
fig_umap.savefig( "../../05_plots/15_CD4annotation/1_CD4leiden_umap.pdf", bbox_inches="tight", dpi=400)
plt.show()

# Cluster annotation
## Step 1: Start by highlighting all clusters and removing an outlier cluster

In [ ]:
#Maximum
CD4_markers = {
        "CITEseq": ["prot_CD45RA_adt", "prot_CD62L_adt","prot_CD95_adt"],
        "IFN-sig": [ "IFI6", "ISG15"], 
        "Ribosomal & mito": ["RPL30", "RPL28", "RPL11", "MT-ATP8", "MT-ND5"],
        "Naive & memory markers": [ "TCF7", "LEF1", "SELL", "CCR7", "IL7R", "KLF2", "MYC", "BACH2", "LTB"],
        "Effector": ["CCL5", "GZMB", "GZMK",  "PDCD1", "TNFSF13B", "TNFSF8"],
        "Th1" : ["TBX21", "IFNG", "CXCR3"], 
        "Th2" : ["GATA3", "IL4"],
        "Th17" : ["RORC", "IL17A"],     
        "Treg": ["FOXP3", "CTLA4", "IL2RA"], 
        "Tfh": ["CXCR5", "BCL6"], 
        "Homing": ["CXCR3", "CXCR4", "CXCR5", "CXCR6", "CCR4", "CCR5", "CCR6", "CX3CR1"],
        "Integrins": ["ITGA1", "ITGA4", "ITGB1", "ITGB2", "ITGB7", "ITGAE","ITGAL", "ITGAM", "ITGAX"], 
        "Cytokines": ["TNF", "IFNG", "IL2", "CSF2"],
        "Activation": ["CD69", "TNFRSF4", "BHLHE40", "CREM", "NR4A1"], 
        "TNF-signaling" : ["TNFAIP3"],
        "NFkB signaling": ["NFKB1", "NFKB2","NFKBIA", "RELA", "RELB"], 
        "NFAT/AP-1 signaling": ["NFATC1", "NFATC2",  "FOS", "FOSB", "JUN", "JUNB", "JUND", "TGFB1", "MAF", "MAFG", "ATF2", "BATF", "BATF2", "BATF3"],
        "Survival": ["BCL2"], 
        "Other": ["TXNIP", "ARHGAP15", "S100A4", "CD52"], 
    "T cells": ["CD3E", "IL32", "LCK", "CD4", "CD8A"]}


# # Define a custom order:
Cluster_order = ["0", "4", "5", "6",  #Tn
                 "8",  "10", #TCM
                 "3", "9", "17", #TEMs
                 "15",   #Treg
                 "18",   #CTL
                  "13",   "16", "20",            #NFKsig / Ribo low
                 "1", "2",  "7", "11", "12",  "14", "19" ]     #Tcsf]   

# #Implement order
sdata["gex"].obs["leiden_1.8"] = sdata["gex"].obs["leiden_1.8"].cat.reorder_categories(
    Cluster_order, ordered=True
)

dp = sc.pl.dotplot(
    sdata["gex"],
    CD4_markers,
    groupby="leiden_1.8",
    standard_scale="var",
    dendrogram=False,
    cmap="RdBu_r",
    swap_axes=False,
    show=False,
    return_fig=True,
)

dp.savefig(
    "../../05_plots/15_CD4annotation/2_CD4_bubblechart.pdf",
    bbox_inches="tight",
    dpi=400
)

fig_umap = sc.pl.umap(sdata["gex"], color=["leiden_1.8"], size = 5, legend_fontweight= 'bold', frameon= False ,legend_fontoutline=2, show=False, legend_loc="on data", cmap="RdBu_r", return_fig=True)


## Step 4: (Re-) Annotate using CITEseq
I previously annotated all T cells using CITEseq. 

Now I focus on CD8 T cells (in PB) and readjust gates to fit this subset

In [ ]:
## How many CD4 T cells do I expect? 
len(sdata["gex"].obs[sdata["gex"].obs["Compartment"] == "PB"])

In [ ]:
#Transfer L3_ann_overview to prot
sdata["prot"].obs["leiden_1.8"] =  sdata["gex"].obs["leiden_1.8"].copy()

#Work only with pb cells
sdata_pb = sdata.copy()

mu.pp.filter_obs(sdata_pb["gex"], "Compartment", lambda x: (x == "PB"))

In [ ]:
#It's more correct to not extract it again. We need to use the prot_ values in GEX_obs
df = sdata_pb["gex"].obs[["prot_CD45RA_adt", "prot_CD62L_adt", "prot_CD95_adt", "leiden_1.8", "Compartment"]].copy()

# # -----------------------------------
# # FEATURES TO PLOT
# # -----------------------------------
# features = ["CD45RA_adt", "CD62L_adt", "CD95_adt"]

# # -----------------------------------
# # EXTRACT PER-CELL ADT VALUES
# # -----------------------------------
# # Get column indices in var
# idx = [sdata_pb["prot"].var_names.get_loc(f) for f in features]

# # Handle sparse or dense .X
# X = sdata_pb["prot"].X
# if not isinstance(X, np.ndarray):
#     X = X.toarray()

# # Build dataframe (cells × 2 ADTs)
# df = pd.DataFrame(
#     X[:, idx],
#     columns=features,
#     index=sdata_pb["prot"].obs_names
# )

# # Keep only non-NaN data points
# df = df.dropna()

# # -----------------------------------
# # ADD L3 ANNOTATION
# # -----------------------------------
# df["leiden_1.9"] = sdata_pb["prot"].obs.loc[
#     df.index, "leiden_1.9"
# ]

# # Keep only non-NaN data points
# df = df.dropna()

In [ ]:
# -----------------------------------
# PARAMETERS (RAW ADT SPACE)
# -----------------------------------
CD45RA_gate = 12
CD62L_gate = 12
xlim = (-3, 51)
ylim = (-3, 41)

# -----------------------------------
# Example plots to set thresholds
# -----------------------------------
g = sb.jointplot(
    data=df,
    x="prot_CD45RA_adt",
    y="prot_CD62L_adt",
    kind="kde",
    fill=True,
    bw_adjust=0.50,
    levels=10,
    thresh=0.05,
    clip=((-5, 50), (-5, 40)),
    height=5
)

# Move legend outside. 
# sb.move_legend(
#     g.ax_joint,
#     "upper left",
#     bbox_to_anchor=(1.4, 1),
#     frameon=False,
#     title="L3 annotation"
# )

ax = g.ax_joint

# -----------------------------------
# GATING LINES and despine
# -----------------------------------
ax.axvline(CD45RA_gate, color="black", linestyle="--")
ax.axhline(CD62L_gate, color="black", linestyle="--")

sb.despine(ax=ax)

# -----------------------------------
# Limits
# -----------------------------------
ax.set_xlim(*xlim)
ax.set_ylim(*ylim)

# -----------------------------------
# Save the figure
# -----------------------------------
plt.tight_layout()
g.savefig("../../05_plots/15_CD4annotation/3_CD45RA_CD62L_global_gate.pdf", 
    dpi=400,
    bbox_inches="tight"
)
plt.show()

In [ ]:
# -----------------------------------
# PARAMETERS (Normalized ADT SPACE)
# -----------------------------------
CD95_gate = 15
CD62L_gate = 12
xlim = (-10, 51)
ylim = (-3, 41)

# -----------------------------------
# Example plots to set thresholds
# -----------------------------------
g = sb.jointplot(
    data=df,
    x="prot_CD95_adt",
    y="prot_CD62L_adt",
    kind="kde",
    # hue="L3_ann_overview",
    fill=True,
    bw_adjust=0.50,
    levels=10,
    thresh=0.05,
    clip=((-10, 50), (-5, 40)),
    height=5
)

# Move legend outside. 
# sb.move_legend(
#     g.ax_joint,
#     "upper left",
#     bbox_to_anchor=(1.3, 1),
#     frameon=False,
#     title="L3 annotation"
# )

ax = g.ax_joint

# -----------------------------------
# GATING LINES and despine
# -----------------------------------
ax.axvline(CD95_gate, color="black", linestyle="--")
ax.axhline(CD62L_gate, color="black", linestyle="--")

sb.despine(ax=ax)

# -----------------------------------
# Limits
# -----------------------------------
ax.set_xlim(*xlim)
ax.set_ylim(*ylim)

# -----------------------------------
# Save the figure
# -----------------------------------
plt.tight_layout()
g.savefig("../../05_plots/15_CD4annotation/4_CD95_CD62L_global_gate.pdf", 
    dpi=400,
    bbox_inches="tight"
)
plt.show()

In [ ]:
# -----------------------------------
# Collective gates
# -----------------------------------
CD95_gate = 15
CD62L_gate = 12
CD45RA_gate = 12

# -----------------------------------
# Add qudrant and CD95 annotation
# -----------------------------------
df["Quadrant"] = pd.Categorical(np.where(
    (df["prot_CD45RA_adt"] >= CD45RA_gate) & (df["prot_CD62L_adt"]  >= CD62L_gate), "Tn-like", np.where(
        (df["prot_CD45RA_adt"] < CD45RA_gate) & (df["prot_CD62L_adt"]  >= CD62L_gate), "Tcm: CD45RA-CD62L+", np.where(
            (df["prot_CD45RA_adt"] < CD45RA_gate) & (df["prot_CD62L_adt"]  < CD62L_gate), "Tem: CD45RA-CD62L-", np.where(
                (df["prot_CD45RA_adt"] >= CD45RA_gate) & (df["prot_CD62L_adt"] < CD62L_gate), "Temra: CD45RA+CD62L-", "error")))))

df["CD95_ann"] = pd.Categorical(np.where(
    df["prot_CD95_adt"] >= CD95_gate, "CD95+", "CD95-"))

df["Quadrant_final"] = pd.Categorical(np.where(
    (df["Quadrant"] == "Tn-like") & (df["CD95_ann"] == "CD95+"), "Tscm: CD45RA+CD62L+CD95+", np.where(
        (df["Quadrant"] == "Tn-like") & (df["CD95_ann"] == "CD95-"), "Tn: CD45RA+CD62L+CD95-", df["Quadrant"])))

df["Quadrant_final"].value_counts()

### Flow-like plots for CITEseq data

In [ ]:
# -----------------------------------
# PARAMETERS (RAW ADT SPACE)
# -----------------------------------
x_gate = 12
y_gate = 12

xlim = (-5, 61)
ylim = (-5, 41)

# -----------------------------------
# LOOP OVER L3 CATEGORIES
# -----------------------------------
for l3 in df["leiden_1.8"].dropna().unique():

    df_sub = df[df["leiden_1.8"] == l3]

    if len(df_sub) < 50:
        print(f"Skipping {l3} (n={len(df_sub)})")
        continue

    x = df_sub["prot_CD45RA_adt"]
    y = df_sub["prot_CD62L_adt"]

    # -----------------------------------
    # JOINT KDE PLOT (new figure per L3)
    # -----------------------------------
    g = sb.jointplot(
        data=df_sub,
        x="prot_CD45RA_adt",
        y="prot_CD62L_adt",
        kind="kde",
        fill=True,
        bw_adjust=0.50,
        levels=10,
        thresh=0.05,
        clip=((-5, 60), (-5, 40)),
        height=5
    )

    ax = g.ax_joint

    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)

    # -----------------------------------
    # GATING LINES
    # -----------------------------------
    ax.axvline(x_gate, color="black", linestyle="--")
    ax.axhline(y_gate, color="black", linestyle="--")

    sb.despine(ax=ax)

    # -----------------------------------
    # QUADRANT PERCENTAGES (PER L3)
    # -----------------------------------
    total = len(df_sub)

    q1 = (x >= x_gate) & (y >= y_gate)
    q2 = (x < x_gate) & (y >= y_gate)
    q3 = (x < x_gate) & (y < y_gate)
    q4 = (x >= x_gate) & (y < y_gate)

    pcts = [
        100 * q1.sum() / total,
        100 * q2.sum() / total,
        100 * q3.sum() / total,
        100 * q4.sum() / total,
    ]

    # -----------------------------------
    # QUADRANT ANNOTATION
    # -----------------------------------
    dx = 0.05 * (xlim[1] - xlim[0])
    dy = 0.10 * (ylim[1] - ylim[0])

    ax.text(x_gate + dx, y_gate + dy, f"{pcts[0]:.1f}%", fontsize=14)
    ax.text(x_gate - 3*dx, y_gate + dy, f"{pcts[1]:.1f}%", fontsize=14)
    ax.text(x_gate - 3*dx, y_gate - 2*dy, f"{pcts[2]:.1f}%", fontsize=14)
    ax.text(x_gate + dx, y_gate - 2*dy, f"{pcts[3]:.1f}%", fontsize=14)

    # -----------------------------------
    # TITLE + EXPORT
    # -----------------------------------
    ax.set_title(f"{l3} (n={total:,})")

    plt.tight_layout()
    plt.savefig(
        f"../../05_plots/15_CD4annotation/5_CD4_CD45RA_CD62L_{l3}.pdf",
        dpi=400,
        bbox_inches="tight"
    )
    plt.show()


# Correspondance between Gex and Prot phenotypes
How well my GEX annotations recapitulate CITEseq annotations?

In [ ]:
df_sub[df_sub["Quadrant_final"] == "error"]

In [ ]:
###### Skip CSF clusters
df_sub = df[~df["leiden_1.8"].isin(["1", "2", "7", "11", "12", "14","19"])]

# Crosstab (rows = Quadrant, columns = L3 annotation)
ct = pd.crosstab(df_sub["leiden_1.8"], df_sub["Quadrant_final"])

# Sort rows by total number of cells
ct = ct.loc[ct.sum(axis=1).sort_values(ascending=True).index]

# Plot horizontal stacked bar chart
ax = ct.plot(
    kind="barh",       # horizontal bars
    stacked=True,
    figsize=(8, 5),
    colormap="tab20"   # nice color palette
)

plt.ylabel("Leiden clusters")  # now vertical axis
plt.xlabel("Number of cells")          # horizontal axis
plt.title("")
plt.legend(title="CITEseq annotations", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig(
    "../../05_plots/15_CD4annotation/6_CD4_CITEseq_reannotation_leiden_1.7.pdf",
    dpi=400,
    bbox_inches="tight"
)
plt.show()


In [ ]:
# Crosstab (rows = L3 annotation, columns = Quadrant)
ct = pd.crosstab(df_sub["Quadrant_final"], df_sub["leiden_1.8"])

quadrant_order = [
    "Tn: CD45RA+CD62L+CD95-",
    "Tscm: CD45RA+CD62L+CD95+",
    "Tcm: CD45RA-CD62L+",
    "Tem: CD45RA-CD62L-",
    "Temra: CD45RA+CD62L-"
]

# Desired column order (example — adjust to your actual cluster labels)
leiden_order = [
    "0","5","6",  "4",  "8","15", "10", "3", "9", "17",  "18",  "13", "16", "20", 
]

# Apply both row and column ordering
ct = ct.reindex(index=quadrant_order, columns=leiden_order)

# Optionally normalize per row for proportions
ct_norm = ct.div(ct.sum(axis=0), axis=1) * 100  # percentages

# Plot heatmap
plt.figure(figsize=(14, 3))
sb.heatmap(
    ct_norm,
    annot=True,        # show numbers
    fmt=".1f",         # one decimal
    cmap="Reds",    # color map
    linewidths=0.5,
    linecolor="white",
    cbar_kws={"label": "% of cells per L3 annotation"}
)

plt.ylabel("CITEseq annotations")
plt.xlabel("Leiden 1.8")
plt.title("Phenotypic distribution across L3 clusters")
plt.tight_layout()
plt.savefig(
     "../../05_plots/15_CD4annotation/7_CD4_CITEseq_reannotation_correspondance.pdf",
    dpi=400,
    bbox_inches="tight"
)
plt.show()

# Report UMAPs

### Compartment

In [ ]:
#color dictionary
Compartment_map = {
    "PB": "#C06A6A",   # red
    "CSF":  "#4B8AC1",   # blue
}

In [ ]:
sc.settings.set_figure_params(figsize=(4.3, 4))
# Create the plot but do NOT show it
sc.pl.umap(
    sdata["gex"],
    color="Compartment",
    palette=Compartment_map,
    size=10,
    legend_fontweight="bold",
    legend_fontsize=6,
    frameon=False,
    show=False,
    legend_loc="on data",
    title=""
)

# Save exactly like your example
plt.tight_layout()
plt.savefig(
    "../../05_plots/15_CD4annotation/15_Compartment_CSF_PB_umap.pdf",
    bbox_inches="tight",
    dpi=400
)
plt.show()

### Main umaps

### Inteferon type 1 signature

In [ ]:
IFN_ab = [
    "TANK", "PSMD14", "ADAR", "IFITM3", "IFITM2", "ADRM1", "CDC37", "USP18",
    "TREX1", "TRIM6", "CR2", "DCST1", "TRIM65", "TTLL12", "YTHDF3", "SAMHD1",
    "SIN3A", "GIGYF2", "LSM14A", "SETD2", "TBK1", "CNOT7", "UBE2K", "IFNE",
    "STING1", "IFI27", "IFIT1", "IFNA1", "IFNA2", "IFNA4", "IFNA5", "IFNA6",
    "IFNA7", "IFNA8", "IFNA10", "IFNA14", "IFNA16", "IFNA17", "IFNA21",
    "IFNAR1", "IFNAR2", "IFNB1", "IFNW1", "IRAK1", "IRF3", "IRF7", "JAK1",
    "USP27X", "SMIM30", "MIR21", "MMP12", "MX1", "MYD88", "OAS1", "OAS2",
    "OAS3", "YTHDF2", "RBM47", "SHFL", "METTL3", "PSMA1", "PSMA2", "IFNK",
    "PSMA3", "PSMA4", "PSMA5", "PSMA6", "PSMA7", "PSMB1", "PSMB2", "PSMB3",
    "PSMB4", "GPR108", "PSMB5", "PSMB6", "PSMB7", "PSMC1", "PSMC2", "PSMC3",
    "PSMC4", "PSMC5", "PSMC6", "PSMD1", "PSMD2", "PSMD3", "PSMD4", "PSMD7",
    "PSMD8", "PSMD11", "PSMD12", "PSMD13", "PSME1", "PSME2", "MAVS", "USP29",
    "PTPN1", "PTPN2", "PTPN6", "PTPN11", "CACTIN", "IFIH1", "AZI2", "SHMT2",
    "SMPD1", "SP100", "STAT1", "STAT2", "TRAF3", "TYK2", "WNT5A", "MUL1",
    "SEM1", "ZBP1", "TRIM56", "NLRC5", "IFITM1", "OASL", "FADD", "CH25H",
    "TRIM41", "RNF185", "EIF4E2", "ISG15", "IKBKE", "TBKBP1", "HDAC4", "PSMD6"
]

sc.tl.score_genes(sdata["gex"], 
                      IFN_ab,  
                      score_name='Response to type I IFN (GOBP)')



### Tr1 signatures from Geginat
What is the Tr1 signature from the Pulvirenti et al preprint (PI: Geginat)?
- CHIL2, CCL4, CCR5 and PDCD1
- ""Cluster 7 displayed the gene signature of EOMES+Tr1-like cells (Bonnal et al., 2021; De Simone et al., 2021; Gruarin et al., 2019), as evidenced by a highly significant Gene Set Enrichment Analysis (GSEA, Figure 2B). Indeed, several key EOMES+Tr1-like signature genes, namely EOMES (Gruarin et al., 2019) (Figure 2C), CHI3L2 (Bonnal et al., 2021), CCL4, CCR5 and PDCD1 (PD1) (Alfen et al., 2018) were selectively expressed in this cluster (sFigure 4C).""
- Gene set enrichment analysis (GSEA) was performed using the fgsea() function from the fgsea package (v1.20.0). The reference Tr1 associated gene set for the analysis was obtained from our previous studies(Bonnal et al., 2021; De Simone et al., 2021).

In [ ]:
### Bonnal Eomes+ Treg signature
genes = [
    "SH2D1A", "HLA-B", "LYST", "DTHD1", "PACSIN1", "PRR5L", "CD81", "CNN2",
    "IFNG-AS1", "CCL4", "GIMAP4", "GZMA", "SUN2", "LIME1", "AC116366.3",
    "ATP5F1E", "CLEC2B", "HLA-E", "LINC02084", "BTN3A2", "CPNE7", "APMAP",
    "DRAP1", "CD3D", "CCL5", "MLLT3", "RABAC1", "PRF1", "DKK3", "LITAF",
    "GZMK", "PFN1", "NAA50", "RARRES3", "TC2N", "HLA-A", "ABI3", "YARS",
    "FABP5", "LYAR", "HMGB1", "STK17A", "ITM2C", "NUCB2", "CD27", "TBCD",
    "CCR5", "APOBEC3C", "OAZ1", "UBE2L6", "AC243829.4", "CST7", "PTPRCAP",
    "SYNM", "GZMH", "RAC2", "CHI3L2", "NKG7", "CD99", "CXCR3", "IDH2",
    "DUSP4", "SLF1", "GYG1", "CD3E", "ITGB2", "C12orf75", "IL9R", "SEPT9",
    "HLA-DRB1", "HLA-DMA", "TNIP3", "SAMD3", "APOBEC3G", "PTK2B", "F2R",
    "C1orf21", "CYBA", "KLRG1", "PECAM1", "PDCD1", "PTPRC", "RNF213",
    "PDE4DIP", "SH3BGRL3", "HLA-C", "PSMB9", "COTL1", "HCST", "LCP1",
    "EIF4EBP1", "RCAN2", "ATP5MPL", "OASL", "CTSW", "CTSC", "GGA2",
    "EOMES", "CD74", "UBL5", "LY6E", "PLEK", "APOBEC3H", "FAM102A",
    "IFNG", "HLA-DPA1", "HSPB11", "CHST12", "HLA-DPB1", "GZMM",
    "SLAMF7", "REEP5", "CMC1", "LAG3", "ITGA1", "B2M", "GIMAP7",
    "RPS27", "PSMA7", "ATP5IF1", "AOAH"
]

# Cacluate a score
sc.tl.score_genes(sdata["gex"], 
                      genes,  
                      score_name='EOMES+ Treg (Bonnel et al 2021)')

In [ ]:
### De Simone et al 2021 (from Geginat lab) Eomes+ Tr1 signature calculated versus other EOMES+ cells
## 
genes_DeSimone = [
    "GZMK", "CD27", "NELL2", "PASK", "LEF1", "CCR7",
    "GZMA", "HLA-DRA", "CCL5", "HLA-DPA1", "CST7", "NKG7",
    "CD74", "EOMES", "HLA-DMA", "S100A4", "ANXA5", "PLEK",
    "ALOX5AP", "LYAR", "OSTF1", "IL10RA", "TXN", "APOBEC3G",
    "ZBP1", "S100A6", "SH3BGRL3",
    "HSPA1B", 
    #"LOC221136" # is not found
]

# Cacluate a score
sc.tl.score_genes(sdata["gex"], 
                      genes_DeSimone,  
                      score_name='Tr1+ cells (De Simone et al 2021)')

In [ ]:
sc.settings.set_figure_params(figsize=(4.3, 4))
fig = sc.pl.umap(sdata["gex"], color=["EOMES+ Treg (Bonnel et al 2021)", "Tr1+ cells (De Simone et al 2021)"], size = 15, legend_fontweight= 'bold', frameon= False ,legend_fontoutline=2, show=True, legend_loc="on data", cmap="RdBu_r", return_fig=True)
fig.savefig("../../05_plots/15_CD4annotation/10_CD4_umap_pathways_Tr1_cells.pdf", bbox_inches="tight",  dpi=400)

## Ostkampf signatures
These were derived from Monaco et al Cell Reports

In [ ]:
ann_dat = pd.read_excel("../../03_annotation/02_gene_sets/Ostkampf_all_gene_sets.xlsx")

#Filter
ann_dat.head(2)

In [ ]:
gene_list = ann_dat["Tfh"].drop_duplicates()
# Cacluate a score
sc.tl.score_genes(sdata["gex"], 
                      gene_list,  
                      score_name='Tfh (Ostkampf et al. 2022)')

In [ ]:
gene_list = ann_dat["Th17"].drop_duplicates()
# Cacluate a score
sc.tl.score_genes(sdata["gex"], 
                      gene_list,  
                      score_name='Th17 (Ostkampf et al. 2022)')

In [ ]:
gene_list = ann_dat["Th1"].drop_duplicates()
# Cacluate a score
sc.tl.score_genes(sdata["gex"], 
                      gene_list,  
                      score_name='Th1 (Ostkampf et al. 2022)')

In [ ]:
gene_list = ann_dat["Tregs"].drop_duplicates()
# Cacluate a score
sc.tl.score_genes(sdata["gex"], 
                      gene_list,  
                      score_name='Tregs (Ostkampf et al. 2022)')

In [ ]:
gene_list = ann_dat["Th2"].drop_duplicates()
# Cacluate a score
sc.tl.score_genes(sdata["gex"], 
                      gene_list,  
                      score_name='Th2 (Ostkampf et al. 2022)')

In [ ]:
gene_list = ann_dat["CD4 TEMRA"].drop_duplicates()
# Cacluate a score
sc.tl.score_genes(sdata["gex"], 
                      gene_list,  
                      score_name='CD4 TEMRA (Ostkampf et al. 2022)')

In [ ]:
gene_list = ann_dat["Th22"].drop_duplicates()
# Cacluate a score
sc.tl.score_genes(sdata["gex"], 
                      gene_list,  
                      score_name='Th22 (Ostkampf et al. 2022)')

In [ ]:
gene_list = ann_dat["TRM"].drop_duplicates()
# Cacluate a score
sc.tl.score_genes(sdata["gex"], 
                      gene_list,  
                      score_name='TRM (Ostkampf et al. 2022)')

In [ ]:
gene_list = ann_dat["TRM CD103+"].drop_duplicates()
# Cacluate a score
sc.tl.score_genes(sdata["gex"], 
                      gene_list,  
                      score_name='TRM CD103+ (Ostkampf et al. 2022)')

# Report CITEseq

In [ ]:
wishlistv2 = [
    "prot_CD45RA_adt", "prot_CD62L_adt", "prot_CD95_adt",
]

sc.settings.set_figure_params(figsize=(4.3, 4))
sc.pl.umap(sdata["gex"], color=wishlistv2, size = 15, legend_fontweight= 'bold', frameon= False ,legend_fontoutline=2, show=True, legend_loc="on data", cmap="RdBu_r", return_fig=True)
plt.savefig("../../05_plots/15_CD4annotation/10_CD4_umap_CITEseq.pdf", bbox_inches="tight",  dpi=400)

# Report marker genes

In [ ]:
wishlistv3 = [
    ## Effector molecules
    "IL2", "IFNG", "TNF", "IL4", "TNFRSF4", "CD40LG", "TNFSF13B", "CSF2", "PRF1", "GZMB", "FASLG", "GZMK", "IL17A", "CCL5", 
]

sc.settings.set_figure_params(figsize=(4.3, 4))
sc.pl.umap(sdata["gex"], color=wishlistv3, size = 15, legend_fontweight= 'bold', frameon= False ,legend_fontoutline=2, show=True, legend_loc="on data", cmap="RdBu_r", return_fig=True)
plt.savefig("../../05_plots/15_CD4annotation/10_CD4_umap_effector_genes.pdf", bbox_inches="tight",  dpi=400)


In [ ]:
wishlistv3 = [
    ## TFs
    "TBX21", "GATA3", "RORC", "FOXP3", "BCL6", "EOMES", "BHLHE40", "ZEB2", "IKZF2", "TOX", 
]
    
sc.settings.set_figure_params(figsize=(4.3, 4))
sc.pl.umap(sdata["gex"], color=wishlistv3, size = 15, legend_fontweight= 'bold', frameon= False ,legend_fontoutline=2, show=True, legend_loc="on data", cmap="RdBu_r", return_fig=True)
plt.savefig("../../05_plots/15_CD4annotation/10_CD4_umap_TFs.pdf", bbox_inches="tight",  dpi=400)


In [ ]:
wishlistv3 = [
    ## chemokine receptors and intergrins,
    "CCR2", "CCR7", "CXCR3", "CXCR4", "CXCR5", "CXCR6", "CX3CR1", "GPR183", "GPR171", "S1PR1", "ICAM3", "ITGAL", "ITGB1", "ITGA4", "ITGB2", "ITGAE", 
]
    
sc.settings.set_figure_params(figsize=(4.3, 4))
sc.pl.umap(sdata["gex"], color=wishlistv3, size = 15, legend_fontweight= 'bold', frameon= False ,legend_fontoutline=2, show=True, legend_loc="on data", cmap="RdBu_r", return_fig=True)
plt.savefig("../../05_plots/15_CD4annotation/10_CD4_umap_intergrins.pdf", bbox_inches="tight",  dpi=400)


In [ ]:
wishlistv3 = [
    ## chemokine receptors and intergrins,
    "CD69", "NFKB2", "RELA", "JUNB", "FOS",  
]
    
sc.settings.set_figure_params(figsize=(4.3, 4))
sc.pl.umap(sdata["gex"], color=wishlistv3, size = 15, legend_fontweight= 'bold', frameon= False ,legend_fontoutline=2, show=True, legend_loc="on data", cmap="RdBu_r", return_fig=True)
plt.savefig("../../05_plots/15_CD4annotation/10_CD4_umap_activation.pdf", bbox_inches="tight",  dpi=400)


# Report signatures

In [ ]:
wishlistv3 = [
    ## Signatures
    "Response to type I IFN (GOBP)", "Tr1+ cells (De Simone et al 2021)",
    "Th1 (Ostkampf et al. 2022)","Th2 (Ostkampf et al. 2022)",  "Th17 (Ostkampf et al. 2022)", 
    "Tregs (Ostkampf et al. 2022)",  "Tfh (Ostkampf et al. 2022)",
]

sc.settings.set_figure_params(figsize=(4.3, 4))
sc.pl.umap(sdata["gex"], color=wishlistv3, size = 15, legend_fontweight= 'bold', frameon= False ,legend_fontoutline=2, show=True, legend_loc="on data", cmap="RdBu_r", return_fig=True)
plt.savefig("../../05_plots/15_CD4annotation/10_CD4_umap_signatures.pdf", bbox_inches="tight",  dpi=400)

    # "CSF2", "PDCD1", "CXCR5", "prot_CD62L_adt",
    # "prot_CD45RA_adt", "prot_CD95_adt", 

# Marker genes per cluster - DEGs

## Global differential gene expression analysis

In [ ]:
sc.tl.rank_genes_groups(sdata["gex"], 'leiden_1.8', method='wilcoxon', key_added="leiden_1.8_DEG")
Export_DEG = sc.get.rank_genes_groups_df(sdata["gex"], group = None, key = "leiden_1.8_DEG")
# Export_DEG = Export_DEG.groupby(['group']).head(50)
Export_DEG.to_csv("../../06_csv_outputs/DEG/1_CD4_DEGs_globally.csv")


In [ ]:
sc.pl.rank_genes_groups(sdata["gex"], key="leiden_1.8_DEG")

# Differentiation gene expression without ribosomal genes

In [ ]:
# Remove TCR-related and ribosomal genes
# TCR genes often start with 'TRAV', 'TRBV', 'TRGV', etc.
# Ribosomal genes often start with 'RPS' or 'RPL'

sdata_filtered = sdata.copy()

# IMPORTANT TO SUBSET ONLY ON CELLS OF INTEREST- because the filter_rank_genes behaves odd if there are cells outside the groups (target and reference) that express a gene of interest. CCR7 kept getting filtered out because naive cells/other cells were part of the analysis
# mu.pp.filter_obs(sdata_pb_filtered["gex"], "top_vs_bot_clones", lambda x: (x != "unselected"))   #Going back to using memory cells as background for filtering

# Create a mask for genes to exclude
genes = sdata["gex"].var_names
exclude_mask = genes.str.startswith(("TRAV", "TRBV", "TRGV", "TRDV", "TRAJ", "TRBJ", "TRGJ", "TRDJ", "TRAC", "TRBC", "TRGC", "TRDC", "RPS", "RPL", "MT-"))

# Keep only genes not matching the exclusion
genes_to_keep = genes[~exclude_mask]
sdata_filtered = sdata_filtered["gex"][:, genes_to_keep]

#
sc.tl.rank_genes_groups(sdata_filtered, 'leiden_1.8', method='wilcoxon', key_added="leiden_1.8_DEG")
Export_DEG = sc.get.rank_genes_groups_df(sdata_filtered, group = None, key = "leiden_1.8_DEG")
# Export_DEG = Export_DEG.groupby(['group']).head(50)
# Export_DEG.to_csv("../../06_csv_outputs/DEG/1_CD4_DEGs_globally.csv")

Export_DEG.head(10)


### Format the table for differentially expressed genes. 

In [ ]:
Export_DEG = sc.get.rank_genes_groups_df(sdata_filtered, group = None, key = "leiden_1.8_DEG")

DEGs_export = (
    Export_DEG[
        (Export_DEG["logfoldchanges"] > 1) &
        (Export_DEG["pvals_adj"] < 0.01)
    ]
    .sort_values("scores", ascending=False)
)

gene_lists = (
    DEGs_export
        # .groupby("group")
        # .head(20)
        .groupby("group")["names"]
        .apply(list)
)

gene_lists_df = pd.DataFrame({
    group: pd.Series(genes)
    for group, genes in gene_lists.items()
})

gene_lists_df.to_excel("../../06_csv_outputs/DEG/2_CD4_Leiden2_LFC_above_1_pval_below_0.01_filtered.xlsx")

In [ ]:
fig_deg = sc.pl.rank_genes_groups_dotplot(
    sdata_filtered, groupby="leiden_1.8", standard_scale="var", n_genes=10, key="leiden_1.8_DEG", cmap="RdBu_r", show=False, return_fig=True, dendrogram=True)

fig_deg.savefig("../../05_plots/15_CD4annotation/11_CD4_leiden_DEG_bubblechart.pdf", bbox_inches="tight",  dpi=400)
plt.show()
# fig_umap = sc.pl.umap(sdata["gex"], color=["leiden_1.7"], size = 5, legend_fontweight= 'bold', frameon= False ,legend_fontoutline=2, show=False, legend_loc="on data", cmap="RdBu_r", return_fig=True)


## Display hallmark genes

In [ ]:
Export_DEG = sc.get.rank_genes_groups_df(sdata_filtered, group = None, key = "leiden_1.8_DEG")



In [ ]:
### Cluster 0
Cluster_genes = Export_DEG[
    (Export_DEG["group"]== "0") & 
    (Export_DEG["logfoldchanges"] > 1) &
    (Export_DEG["pvals_adj"] < 0.01)].head(50)["names"].tolist()

sc.pl.dotplot(
    sdata["gex"],
    Cluster_genes,
    groupby="leiden_1.8",
    standard_scale="var",
    dendrogram=False,
    cmap="RdBu_r",
    swap_axes=False,
)

#Naive genes

In [ ]:
### Cluster 1
Cluster_genes = Export_DEG[
    (Export_DEG["group"]== "1") & 
    (Export_DEG["logfoldchanges"] > 1) &
    (Export_DEG["pvals_adj"] < 0.01)].head(50)["names"].tolist()

sc.pl.dotplot(
    sdata["gex"],
    Cluster_genes,
    groupby="leiden_1.8",
    standard_scale="var",
    dendrogram=False,
    cmap="RdBu_r",
    swap_axes=False,
)

#EGR1

In [ ]:
### Cluster 2
Cluster_genes = Export_DEG[
    (Export_DEG["group"]== "2") & 
    (Export_DEG["logfoldchanges"] > 1) &
    (Export_DEG["pvals_adj"] < 0.01)].head(50)["names"].tolist()

sc.pl.dotplot(
    sdata["gex"],
    Cluster_genes,
    groupby="leiden_1.8",
    standard_scale="var",
    dendrogram=False,
    cmap="RdBu_r",
    swap_axes=False,
)

#CSF cluster + NOSIIP, CXCR3, LGALS1, 

In [ ]:
### Cluster 3
Cluster_genes = Export_DEG[
    (Export_DEG["group"]== "3") & 
    (Export_DEG["logfoldchanges"] > 1) &
    (Export_DEG["pvals_adj"] < 0.01)].head(50)["names"].tolist()

sc.pl.dotplot(
    sdata["gex"],
    Cluster_genes,
    groupby="leiden_1.8",
    standard_scale="var",
    dendrogram=False,
    cmap="RdBu_r",
    swap_axes=False,
)


In [ ]:
### Cluster 4
Cluster_genes = Export_DEG[
    (Export_DEG["group"]== "4") & 
    (Export_DEG["logfoldchanges"] > 1) &
    (Export_DEG["pvals_adj"] < 0.01)].head(50)["names"].tolist()

sc.pl.dotplot(
    sdata["gex"],
    Cluster_genes,
    groupby="leiden_1.8",
    standard_scale="var",
    dendrogram=False,
    cmap="RdBu_r",
    swap_axes=False,
)

# IFNhi + naive genes

In [ ]:
### Cluster 5
Cluster_genes = Export_DEG[
    (Export_DEG["group"]== "5") & 
    (Export_DEG["logfoldchanges"] > 1) &
    (Export_DEG["pvals_adj"] < 0.01)].head(50)["names"].tolist()

sc.pl.dotplot(
    sdata["gex"],
    Cluster_genes,
    groupby="leiden_1.8",
    standard_scale="var",
    dendrogram=False,
    cmap="RdBu_r",
    swap_axes=False,
)

#Naive genes + NOG

In [ ]:
### Cluster 6
Cluster_genes = Export_DEG[
    (Export_DEG["group"]== "6") & 
    (Export_DEG["logfoldchanges"] > 1) &
    (Export_DEG["pvals_adj"] < 0.01)].head(50)["names"].tolist()

sc.pl.dotplot(
    sdata["gex"],
    Cluster_genes,
    groupby="leiden_1.8",
    standard_scale="var",
    dendrogram=False,
    cmap="RdBu_r",
    swap_axes=False,
)

# Naive genes + BACh2, ITGA6, MYC

In [ ]:
### Cluster 7
Cluster_genes = Export_DEG[
    (Export_DEG["group"]== "7") & 
    (Export_DEG["logfoldchanges"] > 1) &
    (Export_DEG["pvals_adj"] < 0.01)].head(50)["names"].tolist()

sc.pl.dotplot(
    sdata["gex"],
    Cluster_genes,
    groupby="leiden_1.8",
    standard_scale="var",
    dendrogram=False,
    cmap="RdBu_r",
    swap_axes=False,
)

#CSF genes, ITGA4, ITGB1, AQP3, LIMS1, ID3. 

In [ ]:
### Cluster 8
Cluster_genes = Export_DEG[
    (Export_DEG["group"]== "8") & 
    (Export_DEG["logfoldchanges"] > 1) &
    (Export_DEG["pvals_adj"] < 0.01)].head(50)["names"].tolist()

sc.pl.dotplot(
    sdata["gex"],
    Cluster_genes,
    groupby="leiden_1.8",
    standard_scale="var",
    dendrogram=False,
    cmap="RdBu_r",
    swap_axes=False,
)

# CISH, PIM memory, SOCS2, IFNGR2, CYLD

In [ ]:
### Cluster 9
Cluster_genes = Export_DEG[
    (Export_DEG["group"]== "9") & 
    (Export_DEG["logfoldchanges"] > 1) &
    (Export_DEG["pvals_adj"] < 0.01)].head(50)["names"].tolist()

sc.pl.dotplot(
    sdata["gex"],
    Cluster_genes,
    groupby="leiden_1.8",
    standard_scale="var",
    dendrogram=False,
    cmap="RdBu_r",
    swap_axes=False,
)

## ACT, LTB, Calcium-signaling, CISH, CD63, BHLHE40, LGALS1, CAPG

In [ ]:
### Cluster 10
Cluster_genes = Export_DEG[
    (Export_DEG["group"]== "10") & 
    (Export_DEG["logfoldchanges"] > 1) &
    (Export_DEG["pvals_adj"] < 0.01)].head(50)["names"].tolist()

sc.pl.dotplot(
    sdata["gex"],
    Cluster_genes,
    groupby="leiden_1.8",
    standard_scale="var",
    dendrogram=False,
    cmap="RdBu_r",
    swap_axes=False,
)

#Memory, IFNhi, 

In [ ]:
### Cluster 11
Cluster_genes = Export_DEG[
    (Export_DEG["group"]== "11") & 
    (Export_DEG["logfoldchanges"] > 1) &
    (Export_DEG["pvals_adj"] < 0.01)].head(50)["names"].tolist()

sc.pl.dotplot(
    sdata["gex"],
    Cluster_genes,
    groupby="leiden_1.8",
    standard_scale="var",
    dendrogram=False,
    cmap="RdBu_r",
    swap_axes=False,
)


In [ ]:
### Cluster 12
Cluster_genes = Export_DEG[
    (Export_DEG["group"]== "12") & 
    (Export_DEG["logfoldchanges"] > 1) &
    (Export_DEG["pvals_adj"] < 0.01)].head(50)["names"].tolist()

sc.pl.dotplot(
    sdata["gex"],
    Cluster_genes,
    groupby="leiden_1.8",
    standard_scale="var",
    dendrogram=False,
    cmap="RdBu_r",
    swap_axes=False,
)


In [ ]:
### Cluster 13
Cluster_genes = Export_DEG[
    (Export_DEG["group"]== "13") & 
    (Export_DEG["logfoldchanges"] > 1) &
    (Export_DEG["pvals_adj"] < 0.01)].head(50)["names"].tolist()

sc.pl.dotplot(
    sdata["gex"],
    Cluster_genes,
    groupby="leiden_1.8",
    standard_scale="var",
    dendrogram=False,
    cmap="RdBu_r",
    swap_axes=False,
)



In [ ]:
### Cluster 14
Cluster_genes = Export_DEG[
    (Export_DEG["group"]== "14") & 
    (Export_DEG["logfoldchanges"] > 1) &
    (Export_DEG["pvals_adj"] < 0.01)].head(50)["names"].tolist()

sc.pl.dotplot(
    sdata["gex"],
    Cluster_genes,
    groupby="leiden_1.8",
    standard_scale="var",
    dendrogram=False,
    cmap="RdBu_r",
    swap_axes=False,
)



In [ ]:
### Cluster 15
Cluster_genes = Export_DEG[
    (Export_DEG["group"]== "15") & 
    (Export_DEG["logfoldchanges"] > 1) &
    (Export_DEG["pvals_adj"] < 0.01)].head(50)["names"].tolist()

sc.pl.dotplot(
    sdata["gex"],
    Cluster_genes,
    groupby="leiden_1.8",
    standard_scale="var",
    dendrogram=False,
    cmap="RdBu_r",
    swap_axes=False,
)



In [ ]:
sc.settings.set_figure_params(figsize=(6, 4))
sc.pl.violin(sdata["gex"], groupby="donor", keys=["AC004448.2"])

In [ ]:
### Cluster 16
Cluster_genes = Export_DEG[
    (Export_DEG["group"]== "16") & 
    (Export_DEG["logfoldchanges"] > 1) &
    (Export_DEG["pvals_adj"] < 0.01)].head(50)["names"].tolist()

sc.pl.dotplot(
    sdata["gex"],
    Cluster_genes,
    groupby="leiden_1.8",
    standard_scale="var",
    dendrogram=False,
    cmap="RdBu_r",
    swap_axes=False,
)


In [ ]:
### Cluster 17
Cluster_genes = Export_DEG[
    (Export_DEG["group"]== "17") & 
    (Export_DEG["logfoldchanges"] > 1) &
    (Export_DEG["pvals_adj"] < 0.01)].head(50)["names"].tolist()

sc.pl.dotplot(
    sdata["gex"],
    Cluster_genes,
    groupby="leiden_1.8",
    standard_scale="var",
    dendrogram=False,
    cmap="RdBu_r",
    swap_axes=False,
)


In [ ]:
### Cluster 18
Cluster_genes = Export_DEG[
    (Export_DEG["group"]== "18") & 
    (Export_DEG["logfoldchanges"] > 1) &
    (Export_DEG["pvals_adj"] < 0.01)].head(50)["names"].tolist()

sc.pl.dotplot(
    sdata["gex"],
    Cluster_genes,
    groupby="leiden_1.8",
    standard_scale="var",
    dendrogram=False,
    cmap="RdBu_r",
    swap_axes=False,
)



In [ ]:
### Cluster 19
Cluster_genes = Export_DEG[
    (Export_DEG["group"]== "19") & 
    (Export_DEG["logfoldchanges"] > 1) &
    (Export_DEG["pvals_adj"] < 0.01)].head(50)["names"].tolist()

sc.pl.dotplot(
    sdata["gex"],
    Cluster_genes,
    groupby="leiden_1.8",
    standard_scale="var",
    dendrogram=False,
    cmap="RdBu_r",
    swap_axes=False,
)


In [ ]:
### Cluster 20
Cluster_genes = Export_DEG[
    (Export_DEG["group"]== "20") & 
    (Export_DEG["logfoldchanges"] > 1) &
    (Export_DEG["pvals_adj"] < 0.01)].head(50)["names"].tolist()

sc.pl.dotplot(
    sdata["gex"],
    Cluster_genes,
    groupby="leiden_1.8",
    standard_scale="var",
    dendrogram=False,
    cmap="RdBu_r",
    swap_axes=False,
)


# Specific differential gene expression

In [ ]:
# Cluster X versus y
sc.tl.rank_genes_groups(sdata["gex"], groupby="leiden_1.8", groups=["10"], reference="4", key_added="upX")
sc.tl.rank_genes_groups(sdata["gex"], groupby="leiden_1.8",  groups=["4"], reference="10", key_added="upY")
sc.settings.set_figure_params(figsize=(3, 2.5))
sc.pl.rank_genes_groups(sdata["gex"], key="upX", n_genes=20)
sc.pl.rank_genes_groups(sdata["gex"], key="upY", n_genes=20)

In [ ]:
# Cluster X versus y
sc.tl.rank_genes_groups(sdata["gex"], groupby="leiden_1.8", groups=["10"], reference="17", key_added="upX")
sc.tl.rank_genes_groups(sdata["gex"], groupby="leiden_1.8",  groups=["17"], reference="10", key_added="upY")
sc.settings.set_figure_params(figsize=(3, 2.5))
sc.pl.rank_genes_groups(sdata["gex"], key="upX", n_genes=20)
sc.pl.rank_genes_groups(sdata["gex"], key="upY", n_genes=20) 

In [ ]:
# Cluster X versus y
sc.tl.rank_genes_groups(sdata["gex"], groupby="leiden_1.8", groups=["3"], reference="17", key_added="upX")
sc.tl.rank_genes_groups(sdata["gex"], groupby="leiden_1.8",  groups=["17"], reference="3", key_added="upY")
sc.settings.set_figure_params(figsize=(3, 2.5))
sc.pl.rank_genes_groups(sdata["gex"], key="upX", n_genes=20)
sc.pl.rank_genes_groups(sdata["gex"], key="upY", n_genes=20) 

In [ ]:
# Cluster X versus y
sc.tl.rank_genes_groups(sdata["gex"], groupby="leiden_1.8", groups=["1"], reference="7", key_added="upX")
sc.tl.rank_genes_groups(sdata["gex"], groupby="leiden_1.8",  groups=["7"], reference="1", key_added="upY")
sc.settings.set_figure_params(figsize=(3, 2.5))
sc.pl.rank_genes_groups(sdata["gex"], key="upX", n_genes=20)
sc.pl.rank_genes_groups(sdata["gex"], key="upY", n_genes=20) 

In [ ]:
# Cluster X versus y
sc.tl.rank_genes_groups(sdata["gex"], groupby="leiden_1.8", groups=["2"], reference="11", key_added="upX")
sc.tl.rank_genes_groups(sdata["gex"], groupby="leiden_1.8",  groups=["11"], reference="2", key_added="upY")
sc.settings.set_figure_params(figsize=(3, 2.5))
sc.pl.rank_genes_groups(sdata["gex"], key="upX", n_genes=20)
sc.pl.rank_genes_groups(sdata["gex"], key="upY", n_genes=20) 

# Final cluster annotation

In [ ]:
#Kilian's suggestions
Cluster_order = [
    "6",  "0", "5", "4",       #naive
     "8", "10", "3",          #Mem
    "9",  "17",                #EM
    "18",  "15",               #Treg
      "13",  "17", "20",       #Act + ribo low
    "7", "2", "11",  "12", "14",  "19", "1"]      #Tcsf]   


leiden_annot = {
    "6":  "6 TN (BACH2hi IFNint)",
    "0":  "0 TN (BACH2hi IFNlo)",
    "5":  "5 TN (BACH2lo IFNlo)",
    "4":  "4 TN (BACH2lo IFNhi)",
    "8":  "8 TCM (CYLDhi IFNlo)",
    "10": "10 TCM (PASKhi IFNhi)",
    "3":  "3 TEM/CM (PASKint FTH1hi)",
    "9":  "9 TEM (TNFhi TNFRSF4hi)",
    "17": "17 TEM (ITGA4hi TNFSF13Bhi)",
    "18": "18 TC (GZMBhi PRF1hi)",
    "15": "15 TREG (FOXP3hi CTLA4hi)",
    "13": "13 TAct (NFKB2hi CXCR4hi)",
    "16" : "16 TAct (JUNBhi FOShi)", 
    "20": "20 TAct (Ribolo)",
    "7":  "7 TCSF (ITGA4hi TCF7hi)",
    "11": "11 TCSF (CXCR3hi KLRB1hi)",
    "2":  "2 TCSF (CXCR3hi KLRBint)",
    "12": "12 TCSF (CXCR3hi GZMKhi)",
    "14": "14 TCSF (H1FXlo JUNBlo)",
    "19": "19 TCSF (TOXhi FOXP3int)",
    "1":  "1 TCSF (JUNBhi CD69hi)",
}
# Add cell type column based on annotation
sdata["gex"].obs['L3_ann'] = [leiden_annot[sample] for sample in sdata["gex"].obs['leiden_1.8']]

## Bubble plot

In [ ]:
## Maximal
marker_dict = {"CITEseq" : ["prot_CD45RA_adt", "prot_CD62L_adt", "prot_CD95_adt"], 
               "TN" : ["CCR7", "SELL", "LEF1", "TCF7", "BACH2", "NOG"],
               "IFNsig" : ["STAT1", "IFI6", "ISG15"], 
               #Memory / Effector         
               "C8" : ["CYLD", "PIM1"], 
                "C10" : ["PASK"], 
                "C3" : [ "FTH1"],
               "C9" : ["TNF", "TNFRSF4", "CD40LG", "BHLHE40"], 
                "C17" : ["ITGA4", "TNFSF13B", "CXCR3"], 
                "C18" : ["CCL5", "GZMB", "PRF1"], 
                "C15" : ["FOXP3", "IL2RA", "CTLA4"], 

               #Tact
               "C13" : ["NFKB2", "CXCR4"], 
               "C16" : ["JUNB", "FOS"], 
                "C20" : ["RPS8","RPL30"],
    
               #CSF
               "C7" :  ["ITGA4", "ITGB1", "TCF7"],
               "C11/C2" : ["CXCR3", "KLRB1"],
               "C12" : ["GZMK", "GZMA", "GZMH", "EOMES"],
               "C14" : ["H1FX","JUNB"], 
               "C19" :["TOX", "FOXP3"], 
                "C1" :  ["JUNB","CD69", "DUSP2", "NR4A1"], 

                #Shared
                "Shared CSF"  : ["TXNIP", "S100A4", "CD52"]
              }

# # Define a custom order by following the order of clusters in your annotation box above
ordered_categories = list(leiden_annot.values())

# #Implement order
# Convert to categorical
sdata["gex"].obs["L3_ann"] = pd.Categorical(
    sdata["gex"].obs["L3_ann"],
    categories=ordered_categories,
    ordered=True
)

sdata["gex"].obs["L3_ann"] = sdata["gex"].obs["L3_ann"].cat.reorder_categories(
    ordered_categories, ordered=True
)

dp = sc.pl.dotplot(
    sdata["gex"],
    marker_dict,
    groupby="L3_ann",
    standard_scale="var",
    dendrogram=False,
    cmap="RdBu_r",
    swap_axes=False,
    show=False,
    return_fig=True,
)

dp.savefig(
    "../../05_plots/15_CD4annotation/2_CD4_bubblechart_minimal_annotated.pdf",
    bbox_inches="tight",
    dpi=400
)


# Repertoire information

### Step 1: Transfer important information and make sure the clone_handle does not contain artefacts

In [ ]:
#Make into categorical columns
sdata["gex"].obs["donor"] = sdata["gex"].obs["donor"].astype("category")

#Transfer important information
sdata["gex"].obs["IR_VDJ_1_junction_aa"] = sdata["airr"].obs["IR_VDJ_1_junction_aa"].copy()

# Transfer remaining important information
sdata["airr"].obs["donor"] = sdata["gex"].obs["donor"].copy()
sdata["airr"].obs["pool"] = sdata["gex"].obs["pool"].copy()
sdata["airr"].obs["Compartment"] = sdata["gex"].obs["Compartment"].copy()
sdata["airr"].obs["clone_handle"] = sdata["gex"].obs["clone_handle"].copy()

# If clone_id is nan then clone_handle is also na
sdata["gex"].obs["clone_handle"] = np.where(sdata["gex"].obs["clone_id"].isna(), None, sdata["gex"].obs["clone_handle"])

sdata.update()

### Step 2: Make sure the GEX, AIRR and PROT modalities represent the same cells found in GEX

In [ ]:
#Make sure only the indexes recorded in GEX are used in the o*ther modalities too
index_list = sdata["gex"].obs.index.tolist()

## define a cell index
sdata["airr"].obs["cell_index"] = sdata["airr"].obs.index
sdata["prot"].obs["cell_index"] = sdata["prot"].obs.index

## ".isin" using indexes does not seem to work on a global level, but if I annotate a column and then filter in place, maybe then it works. 
sdata["airr"].obs["filter_index"] = pd.Categorical(np.where(sdata["airr"].obs["cell_index"].isin(index_list), "GEX_present", "GEX_absent"))
sdata["prot"].obs["filter_index"] = pd.Categorical(np.where(sdata["prot"].obs["cell_index"].isin(index_list), "GEX_present", "GEX_absent"))

#Enforce that sdata["airr"].obs and sdata["prot"].obs only contain information on those indexes that exists in GEX
mu.pp.filter_obs(sdata["airr"], 'filter_index', lambda x: x == "GEX_present")
mu.pp.filter_obs(sdata["prot"], 'filter_index', lambda x: x == "GEX_present")

### Step 3: Define clonotypes and type_x_clone_count (compartment-wise per clonotype)

In [ ]:
#Transfer labels from the airr to gex modality
# sdata["gex"].obs["antigen.species"] = sdata["airr"].obs["antigen.species"].copy()
# sdata["gex"].obs["antigen.epitope"] = sdata["airr"].obs["antigen.epitope"].copy()

#CDR3aa
sdata["gex"].obs["IR_VDJ_1_junction_aa"] = sdata["airr"].obs["IR_VDJ_1_junction_aa"].copy()
sdata["gex"].obs["IR_VJ_1_junction_aa"] = sdata["airr"].obs["IR_VJ_1_junction_aa"].copy()

#TRAV and TRBV
sdata["gex"].obs["IR_VJ_1_v_call"] = sdata["airr"].obs["IR_VJ_1_v_call"].copy()
sdata["gex"].obs["IR_VDJ_1_v_call"] = sdata["airr"].obs["IR_VDJ_1_v_call"].copy()

#D regions
sdata["gex"].obs["IR_VDJ_1_d_call"] = sdata["airr"].obs["IR_VDJ_1_d_call"].copy()

#J regions
sdata["gex"].obs["IR_VJ_1_j_call"] = sdata["airr"].obs["IR_VJ_1_j_call"].copy()
sdata["gex"].obs["IR_VDJ_1_j_call"] = sdata["airr"].obs["IR_VDJ_1_j_call"].copy()


In [ ]:
sdata_backup = sdata.copy()
airr_export = sdata["airr"].obs[["pool","donor","Compartment", "clone_id", "clone_handle",
                   'IR_VJ_1_junction_aa', 'IR_VDJ_1_junction_aa',
                   'IR_VJ_1_v_call', 'IR_VDJ_1_v_call',
                   'IR_VJ_1_j_call', 'IR_VDJ_1_j_call',
                   'IR_VDJ_1_d_call',
                   'IR_VJ_2_junction_aa', 'IR_VDJ_2_junction_aa',
                   'IR_VJ_2_v_call',  'IR_VDJ_2_v_call', 
                   'IR_VJ_2_j_call', 'IR_VDJ_2_j_call',
                   'IR_VDJ_2_d_call']]

# Drop rows where 'clone_handle' is NaN
airr_export = airr_export.dropna(subset=['Compartment']) #Subset tells pandas to dropna only for Comparment
airr_export = airr_export.dropna(subset=['clone_id']) #Subset tells pandas to dropna only for clone_id

## First generate a new clone ID label that is a concatenation of donor and clone_id
airr_export["clone_handle"] #Already generated

## Second generate a summary table with counts per clone_handle per compartment - clone_handles are used here to avoid counting public shared by donors together
counts_per_id = airr_export.groupby(["Compartment", 'clone_handle']).size().reset_index(name='type_x_clone_count')

# Merge the count information back into the original DataFrame
airr_export = pd.merge(airr_export, counts_per_id, on=["Compartment", 'clone_handle'], how='left')

#### Clean the data frame

In [ ]:
airr_export_backup = airr_export.copy()
airr_export = airr_export[["pool", "donor", "Compartment", "clone_id", "clone_handle", "type_x_clone_count", "IR_VJ_1_junction_aa", "IR_VDJ_1_junction_aa"]]

### Step 4: A venn diagram -- non-naive clonotypes

In [ ]:
## Non naive clone handles
NonNaive_set = set(sdata["gex"].obs[~sdata["gex"].obs["leiden_1.8"].isin(["0", "4", "5", "6"])]["clone_handle"])

In [ ]:
from matplotlib_venn import venn2

# Create subplots
fig, axes = plt.subplots(1, 1, figsize=(5, 5))

# %%% Patient 1 %%%%
# Extract unique values from each column
set1 = set(airr_export[airr_export['pool'].isin(["Pt1_PBMC", "Pt4_PBMC", "Pt5_PBMC", "Pt8_PBMC", "Pt11_PBMC", "Pt12_PBMC", "Pt14_PBMC", "Pt17_PBMC", "Pt18_PBMC", "Pt19_PBMC", "Pt20_PBMC", "Pt21_PBMC"])]["clone_handle"])
set2 = set(airr_export[airr_export['pool'].isin(["Pt1", "Pt4", "Pt5", "Pt8", "Pt11", "Pt12", "Pt14", "Pt17", "Pt18", "Pt19", "Pt20", "Pt20+21"])]["clone_handle"])

#Non_naive set 1
set1_nonnaive = set1 & NonNaive_set

# Compute sizes for each section
only_set1 = len(set1_nonnaive - set2)
only_set2 = len(set2 - set1_nonnaive)
intersection = len(set1_nonnaive & set2)

total = only_set1 + only_set2 + intersection

# Compute percentages
pct1 = 100 * only_set1 / total
pct2 = 100 * only_set2 / total
pct_inter = 100 * intersection / total

# Create a Venn diagram
vennAll = venn2(subsets=(only_set1, only_set2, intersection), set_labels=('PB', 'CSF'))

# Modify labels to show percentages
if vennAll.get_label_by_id('10'):
    vennAll.get_label_by_id('10').set_text(f"{only_set1}\n({pct1:.1f}%)")
if vennAll.get_label_by_id('01'):
    vennAll.get_label_by_id('01').set_text(f"{only_set2}\n({pct2:.1f}%)")
if vennAll.get_label_by_id('11'):
    vennAll.get_label_by_id('11').set_text(f"{intersection}\n({pct_inter:.1f}%)")

# Set colors to a colorblind-friendly palette
vennAll.get_patch_by_id('10').set_color('#C06A6A')  # left = Muted red
vennAll.get_patch_by_id('01').set_color('#4B8AC1')  # right = Blue
vennAll.get_patch_by_id('11').set_color('#D89A52')  # overlap = orange

# Adjust transparency for all patches
for venn in [vennAll]:
    for patch in venn.patches:
        if patch:
            patch.set_alpha(1)

# Display the plot
plt.title("Clonotype overlap, CD4 T cells") 
plt.tight_layout()
plt.show()

# Save the figure 
fig.savefig("../../05_plots/15_CD4annotation/16_VennCompartmentOverlap.pdf", bbox_inches='tight', format="pdf", dpi=400)

### Step 5: Binomial test
https://www.statology.org/binomial-test-python/

Example 1: We roll a 6-sided die 24 times and it lands on the number “3” exactly 6 times. Perform a binomial test to determine if the die is biased towards the number “3.”

**binom_test(x=6, n=24, p=1/6, alternative='greater')**

0.1995295129479586

Because this p-value (0.1995) is not less than 0.05, we fail to reject the null hypothesis. We do not have sufficient evidence to say the die is biased towards the number “3.”

#### Can we do the same for compartment resampling
From Kilian'sa advice: 
The clone size corresponds to the probability of sampling a given clone, fx. if the clone size is 30 out of 3000 cells, then the probability is 30/3000 or 1%
For any given clone this corresponds to a binomial distribution with the probability being equal to the clone fraction in PB, and
n being the number of T cells surveyed in the CSF repertoire. A binomal test can be made. 

So: binom_test(
    x=    "how many X clones we saw", 
    n=    "how many cells we sampled in CSF", 
    p=    "a baseline distribution based on resampling PB", 
    alternative='two-sided')

Under the assumption of random clone sharing between PB and CSF, we can assume that CSF can be "created" from the binomal distribution observed in the PB samples. 
The clone size in the PB sample corresponds to the probability of sampling a given clone. For any given clone this corresponds to a binomial distribution with the probability being equal
to the clone fraction in PB and n being the number of T cells surveyed in CSF. Using a binomial test, we can test if the observed size of a given clone in the CSF 
corresponds to the expected under the assumption of random clone sharing (observed CSF fraction should under the null hypothesis be similar to fraction in CSF * N cells in CSF) 
This method only simulates the probabilities of clone sharing. 

(Nikolaj) Why not just use Fischer's exact test?


In [ ]:
# Definition of cell total per sample
airr_export['cells_in_sample'] = airr_export.groupby(['pool'])['type_x_clone_count'].transform('size')

# %%%%%%%%%%%%
# Definition of clone_x_fra_PB (used as reference frequency later)
# %%%%%%%%%%%%

## For each clonotype calculate the fraction of cells out of all cells in PB. I.e. the fraction of the dataset made up by the given clone. 
PB_clones = airr_export[airr_export["Compartment"] == "PB"]
PB_clones = PB_clones[["donor", "clone_handle", "type_x_clone_count", "cells_in_sample"]]
PB_clones = PB_clones.drop_duplicates().reset_index()
PB_clones["clone_x_fra_PB"] = PB_clones["type_x_clone_count"] / PB_clones["cells_in_sample"]
PB_clones = PB_clones[["donor", "clone_handle", "clone_x_fra_PB"]]

## merge the summary information back
airr_export = pd.merge(airr_export, PB_clones, on=["donor", 'clone_handle'], how='left')

# Define minimum values as a lower obund too
dfc = PB_clones.groupby('donor')['clone_x_fra_PB']
Min_df = PB_clones.assign(min=dfc.transform(min))

Min_df = Min_df[["donor", "min"]].drop_duplicates()

## merge the min data information back
airr_export = pd.merge(airr_export, Min_df, on=["donor"], how='left')

## Now replace the NaN values in clone_x_fra_PB with the lower bound min value
airr_export['clone_x_fra_PB'] = airr_export['clone_x_fra_PB'].fillna(airr_export.pop('min'))

# %%%%%%%%%%%%
# Definition of clone_x_fra_CSF (used as reference frequency later)
# %%%%%%%%%%%%

## Get the fraction of cells for a given clone from the PB compartment per clone
CSF_clones = airr_export[airr_export["Compartment"] == "CSF"]
CSF_clones = CSF_clones[["donor", "clone_handle", "type_x_clone_count", "cells_in_sample"]]
CSF_clones = CSF_clones.drop_duplicates().reset_index()
CSF_clones["clone_x_fra_CSF"] = CSF_clones["type_x_clone_count"] / CSF_clones["cells_in_sample"]
CSF_clones = CSF_clones[["donor", "clone_handle", "clone_x_fra_CSF"]]

## merge the summary information back
airr_export = pd.merge(airr_export, CSF_clones, on=["donor", 'clone_handle'], how='left')

# Define minimum values as a lower obund too
dfc = CSF_clones.groupby('donor')['clone_x_fra_CSF']
Min_df = CSF_clones.assign(min=dfc.transform(min))

Min_df = Min_df[["donor", "min"]].drop_duplicates()

## merge the min data information back
airr_export = pd.merge(airr_export, Min_df, on=["donor"], how='left')

## Now replace the NaN values in clone_x_fra_PB with the lower bound min value
airr_export['clone_x_fra_CSF'] = airr_export['clone_x_fra_CSF'].fillna(airr_export.pop('min'))

In [ ]:
#How does the binom test work: Example: 
binomtest(6, 24, p=1/6)

In [ ]:
## For CSF cells compared to PB background distribution
airr_export["p_csf"] = airr_export.apply(lambda x: binomtest(x['type_x_clone_count'],   #k - number of successes (number of observed cells in the given compartment)
                                        x['cells_in_sample'],                       #n - number of trials (based on cells in the given compartment)
                                        x['clone_x_fra_PB'],                        #p - null expected likelihood of success (based on freq. in PB)
                                        alternative= "two-sided").pvalue, axis=1)

## For PB cells compared to CSF background distribution
airr_export["p_PB"] = airr_export.apply(lambda x: binomtest(x['type_x_clone_count'],   #k - number of successes (number of observed cells in the given compartment)
                                        x['cells_in_sample'],                       #n - number of trials (based on cells in the given compartment)
                                        x['clone_x_fra_CSF'],                        #p - null expected likelihood of success (based on freq. in PB)
                                        alternative= "two-sided").pvalue, axis=1)

## Merge the two p-values - take the smallest of the two first. 
airr_export["p"] = np.where(airr_export["p_csf"] < airr_export["p_PB"], airr_export["p_csf"], airr_export["p_PB"])

#### Bonferroni adj
The Bonferroni correction can also be applied as a p-value adjustment: Using that approach, instead of adjusting the alpha level, each p-value is multiplied by the number of tests (with adjusted p-values that exceed 1 then being reduced to 1), and the alpha level is left unchanged. The significance decisions using this approach will be the same as when using the alpha-level adjustment approach.

In [ ]:
## Correct the p-value by dividing with the number of test made for every donor
N_tests_df = airr_export[["donor", "Compartment", "clone_handle"]].drop_duplicates().reset_index()
N_tests_df["N_tests"] = N_tests_df.groupby(['donor', "Compartment"])['clone_handle'].transform('size')

# Merge the count information back into the original DataFrame
airr_export = pd.merge(airr_export, N_tests_df, on=["donor","Compartment",'clone_handle'], how='left')

In [ ]:
#Bonferroni p_adj calculation
airr_export["p_adj"] = airr_export["p"]*airr_export["N_tests"]  

#Mask p_adj-value to 1
airr_export['p_adj'] = airr_export['p_adj'].apply(lambda x: 1 if x > 1 else x)

#Generate a label if significant. 
airr_export["Sig_label"] = airr_export['p_adj'].apply(lambda x: "ns" if x >= 0.05 else "q < 0.05")

### Step 6: Biaxial plot w. cell counts

In [ ]:
### First you need the % values to later give color annotations correctly
# Add a column with the total number of observations for each combination
airr_export['cells_in_sample'] = airr_export.groupby(['pool'])['type_x_clone_count'].transform('size')

# Return unique values of two columns is in pandas defined as a "drop_duplicates()" problem
airr_export[["pool", "cells_in_sample"]].drop_duplicates() #Number of cells in sample seems realisitic - although 22 and 87 cells in CSF is going to be a problem...

#Define the percentage-wise observation from columns you have
airr_export['type_x_clone_pct'] = (airr_export['type_x_clone_count'] / airr_export['cells_in_sample']) * 100

# General columns to keep
cols = ["Compartment", "clone_handle", "type_x_clone_pct"] 

# %%% Patient samples %%%
PT_all = airr_export # All of it [airr_export["pool"].isin(["Pt1_PBMC", "Pt1", "Pt4_PBMC", "Pt4", "Pt5", "Pt5_PBMC", "Pt8", "Pt8_PBMC", "Pt11", "Pt11_PBMC", "Pt12", "Pt12_PBMC"])]
PT_all = PT_all[cols]
PT_all = PT_all.drop_duplicates(subset=cols)

#df.pivot(index='foo', columns='bar', values='baz')
PT_all = PT_all.pivot(index=["clone_handle"], columns=["Compartment"], values="type_x_clone_pct").fillna(0).reset_index()

#Rename to make merging easier later
Pct_df = PT_all.rename(columns={
    "CSF":"Pct_CSF", 
    "PB": "Pct_PB"
})

In [ ]:
# General columns to keep
cols = ["Compartment", "clone_handle", "type_x_clone_count", "Sig_label"]

# Assign colors based on categories
colors = {'ns': 'grey', 'q < 0.05': "red"}

# %%% Patient samples %%%
PT_all = airr_export
PT_all = PT_all[cols]
PT_all = PT_all.drop_duplicates(subset=cols)

# Pivot table
PT_all = PT_all.pivot(index=["clone_handle", "Sig_label"], columns=["Compartment"], values="type_x_clone_count").fillna(0).reset_index()

PT_all = PT_all.merge(Pct_df, how="left", on="clone_handle")

#Define color
PT_all["color"] = np.where((PT_all["Sig_label"] != "ns") & (PT_all["Pct_CSF"] > PT_all["Pct_PB"]), "CSF-enriched",
                          np.where((PT_all["Sig_label"] != "ns") & (PT_all["Pct_CSF"] < PT_all["Pct_PB"]), "PB-enriched", "ns"))

In [ ]:
# Create subplots
fig, axes = plt.subplots(1, 1, figsize=(6.75, 4))

#color dictionary
color_map = {
    "PB-enriched": "#C06A6A",   # red
    "CSF-enriched":  "#4B8AC1",   # blue
    "ns":           "lightgrey"
}


# Plotting
x = PT_all["CSF"]
y = PT_all["PB"]

# Rasterize scatter plot & place grid behind data
axes.set_axisbelow(True)
scatter = axes.scatter(
    x=PT_all["CSF"],
    y=PT_all["PB"],
    alpha=1,
    s=50,
    c=PT_all["color"].map(color_map),
    edgecolors="white",
    linewidths=0.2,   # slim white border
    rasterized=True
)

# Labels and grid
# axes.set_ylim(-55,1300)
# axes.set_xlim(-5, 90)
axes.set_xlabel(r'CSF - No. of cells', fontsize=15)
axes.set_ylabel(r'PB - No. of cells', fontsize=15)
axes.set_title('Clonotype overlap, CD4 T cells, CSF vs PB T cells')
axes.grid(False)
sb.despine()

# Legend (explicitly tied to color_map)
legend_handles = [
    mpatches.Patch(color=color_map[label], label=label)
    for label in ["CSF-enriched", "PB-enriched", "ns"]
    if label in color_map
]

axes.legend(
    handles=legend_handles,
    title="Significance",
    bbox_to_anchor=(1.05, 1),
    loc="upper left",
    borderaxespad=0.
)

# Layout and save
plt.tight_layout(rect=[0, 0, 0.85, 1])
fig.savefig("../../05_plots/16_CD4analysis/1_CD4_XY_expansion.pdf",
            bbox_inches='tight', format="pdf", dpi=400)
plt.show()

#### Save the statistical output

In [ ]:
#Is clone counts in airr_export the same as airr_backup - yes
len(airr_export[airr_export_backup["type_x_clone_count"] == airr_export["type_x_clone_count"]]) == len(airr_export)

### Step 7: Add back the clonal expansion information to the mudata object (sdata)

In [ ]:
#Generating a total count - adding back in sdata

# Create deduplicated DataFrame
airr_total_count = (
    airr_export[["clone_handle", "Compartment", "type_x_clone_count", "p_csf", "p_PB", "p_adj", "Sig_label"]]
    .drop_duplicates()
    .sort_values(by="clone_handle")
)

# Add total_clone_count column using transform to preserve row count
airr_total_count["total_clone_count"] = (
    airr_total_count
    .groupby("clone_handle")["type_x_clone_count"]
    .transform("sum")
)

# (Optional) Sort by total_clone_count
airr_total_count = airr_total_count.sort_values(by="total_clone_count", ascending=False)
airr_total_count

In [ ]:
## %%
# Add "type_x_clone_count" as well as "total_clone_count" back into the single-cell object
# %%

sdata["gex"].obs = sdata["gex"].obs.reset_index().merge(airr_total_count, on=["clone_handle", "Compartment"], how="left").set_index('index')  
  #Note: The reset_index to .set_index("index") works to save the current index of the left data frame, and then use that later in set_index() again. 
  #It's a common problem described here: https://stackoverflow.com/questions/11976503/how-to-keep-index-when-using-pandas-merge

sdata.update()

# Report clonal expansion

### Final clone umap of clonal expansion

In [ ]:
## Note that total_clone_count is the more correct data frame
sdata["gex"].obs[sdata["gex"].obs["clone_size"]!= sdata["gex"].obs["total_clone_count"]][["clone_handle", "clone_size", "total_clone_count"]].drop_duplicates().head(10)


In [ ]:
## Create clonal bins
# Define binned categories
sdata["gex"].obs["clone_count_binned"] = pd.Categorical(np.where(sdata["gex"].obs["clone_handle"].isna(), "NaN", 
                                                                 np.where(sdata["gex"].obs["total_clone_count"] == 1, "1", 
                                                                         np.where(sdata["gex"].obs["total_clone_count"] < 6, "2-5",  
                                                                                  np.where(sdata["gex"].obs["total_clone_count"] < 11, "6-10",
                                                                                           np.where(sdata["gex"].obs["total_clone_count"] < 21, "11-20", "> 20"))))))

#color dictionary
binned_map = {
    "NaN": "#FFFFFF",   # white
    "1":  "#F0F0F0",   # grey1
    "2-5":  "#D9D9D9",   # grey2
    "6-10":  "#BDBDBD",   # grey3
    "11-20":  "#737373",   # grey4
    "> 20":  "#252525",   # grey5
}

sc.settings.set_figure_params(figsize=(4.3, 4))
# Create the plot but do NOT show it
sc.pl.umap(
    sdata["gex"],
    color="clone_count_binned",
    palette=binned_map,
    size=60,
    add_outline=True,
    legend_fontweight="bold",
    legend_fontsize=6,
    frameon=False,
    show=False,
    legend_loc="on data",
    title=""
)

# Save exactly like your example
plt.tight_layout()
plt.savefig(
    "../../05_plots/15_CD4annotation/4_Clonal_bin_umap.pdf",
    bbox_inches="tight",
    dpi=400
)
plt.show()

### Gini index per cluster
The different index metrics are described here: https://scikit.bio/docs/dev/generated/skbio.diversity.alpha.html

The function to calculate these metrics is described here: https://scirpy.scverse.org/en/latest/generated/scirpy.tl.alpha_diversity.html

**Gini index** portrays dominance defined as: Dominance [5] (or concentration) measures the degree that one or a few most abundant species (taxa) represent the great majority of a community. It can be considered as a measure of community unevenness.

In [ ]:
sdata.update()

In [ ]:
###Clean-up.
#Make sure only the indexes recorded in GEX are used in the o*ther modalities too
index_list = sdata["gex"].obs.index.tolist()

## define a cell index
sdata["airr"].obs["cell_index"] = sdata["airr"].obs.index
sdata["prot"].obs["cell_index"] = sdata["prot"].obs.index

## ".isin" using indexes does not seem to work on a global level, but if I annotate a column and then filter in place, maybe then it works. 
sdata["airr"].obs["filter_index"] = pd.Categorical(np.where(sdata["airr"].obs["cell_index"].isin(index_list), "GEX_present", "GEX_absent"))
sdata["prot"].obs["filter_index"] = pd.Categorical(np.where(sdata["prot"].obs["cell_index"].isin(index_list), "GEX_present", "GEX_absent"))

#Enforce that sdata["airr"].obs and sdata["prot"].obs only contain information on those indexes that exists in GEX
mu.pp.filter_obs(sdata["airr"], 'filter_index', lambda x: x == "GEX_present")
mu.pp.filter_obs(sdata["prot"], 'filter_index', lambda x: x == "GEX_present")

sdata.update()

In [ ]:
sdata_gini = sdata.copy()
sdata_gini["airr"].obs["clone_handle"] = sdata_gini["gex"].obs["clone_handle"].copy()
sdata_gini["airr"].obs["leiden_1.8"] = sdata_gini["gex"].obs["leiden_1.8"].copy()
sdata_gini["airr"].obs["donor"] = sdata_gini["gex"].obs["donor"].copy()
sdata_gini["airr"].obs["catpat"] = sdata_gini["gex"].obs["catpat"].copy()

#Make grouping variables
sdata_gini["airr"].obs["donor_leiden_1.8"] = sdata_gini["airr"].obs["donor"].astype(str) + "-" + sdata_gini["airr"].obs["leiden_1.8"].astype(str)

#filter
mu.pp.filter_obs(
    sdata_gini["airr"],
    "clone_id",
    lambda x: ~pd.isna(x)
)

In [ ]:
## %% Calculate the diversity metrics andsave them under a column key. 
ir.tl.alpha_diversity(sdata_gini["airr"], groupby="leiden_1.8", target_col='clone_handle', metric='gini_index', inplace=True, key_added="gini_per_cluster", airr_mod='airr')

In [ ]:
# Drop rows with NaN gini index
df_gini = sdata_gini["airr"].obs[["leiden_1.8", "gini_per_cluster"]].drop_duplicates().reset_index()
df_gini = df_gini.dropna(subset=["gini_per_cluster"])
df_gini["gini_per_cluster"] = pd.to_numeric(df_gini["gini_per_cluster"])

## Use order if you want the order to be decided by highest value
order = df_gini.sort_values("gini_per_cluster",ascending=False)["leiden_1.8"].to_list()

## leiden order for the manual order by Kilian
leiden_order = ["6", "0", "5", "4", "8", "10", "3", "9", "17", "18", "15", "13", "16", "20", "7", "11", "2", "12", "14", "19", "1"]


In [ ]:
df_gini["leiden_1.8"] = df_gini["leiden_1.8"].astype(str)

plt.figure(figsize=(8,2))

sb.barplot(
    data=df_gini,
    x='leiden_1.8',
    y='gini_per_cluster',
    order=leiden_order,
    color='lightgrey'
)

plt.xlabel("leiden_1.8")
plt.ylabel("Gini index")
plt.title("Gini per Leiden cluster")
# edit layout
sb.despine()
ax.grid(False)

plt.tight_layout()
plt.savefig("../../05_plots/15_CD4annotation/17_Gini_per_cluster.pdf", bbox_inches="tight",  dpi=400)
plt.show()

### Report clonal expansion per cluster
Proportion of each cluster made up of clonally expanded cells at 5 different bins

In [ ]:
#First, generate 5 different bins of biological relevance
bins = [0, 1, 5, 10, 20, float("inf")]
labels = ["1", "2-5", "6-10", "11-20", ">20"]

# Bin the total_clone_count column
sdata["gex"].obs["clone_count_bin"] = pd.cut(
    sdata["gex"].obs["total_clone_count"],
    bins=bins,
    labels=labels,
    right=True,
    include_lowest=True
)

In [ ]:
sdata["gex"].obs[["clone_handle", "clone_count_bin"]].drop_duplicates()["clone_count_bin"].value_counts()

In [ ]:
# Extract relevant data
df = sdata["gex"].obs[["clone_handle", "clone_count_bin", "leiden_1.8"]].copy()

# Drop missing annotations if any
df = df.dropna(subset=["clone_count_bin", "leiden_1.8"])

# Compute counts
counts = (
    df.groupby(["leiden_1.8", "clone_count_bin"])
    .size()
    .unstack(fill_value=0)
)

# Convert to proportions
proportions = counts.div(counts.sum(axis=1), axis=0)

# custom cmap
colors = ['lightgrey', '#ff7f0e', '#279e68', '#d62728', '#aa40fc']  # matches bin_order
custom_cmap = ListedColormap(colors)


#re-index in a specific order
proportions = proportions.reindex(
    [cl for cl in leiden_order if cl in proportions.index]
)

# Plot
proportions.plot(
    kind="bar",
    stacked=True,
    figsize=(8, 2),
    width=0.8,
    colormap=custom_cmap
)

plt.ylabel("Proportion")
plt.xticks(rotation=0)
plt.title("Proportional Contribution of Clone Count Bins per L3 Annotation")
plt.legend(title="Clone Count Bin", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(False)
sb.despine()
plt.tight_layout()
plt.savefig("../../05_plots/15_CD4annotation/Expansion_over_clusters.pdf")
plt.show()

# Export L3_annotations

In [ ]:
sdata.write(filename="../../04_data_objects/06_preprocessed_v2/2026_8_CD4_annotated.h5mu")

# Old code

# Removing doublets based on conventional QC

In [ ]:
# Plot the QC metrics, and decide a cutoff %%%%%%%%%%%%
# Number of Axes & plot size
ncols = 4
nrows = 1
figsize = 4
wspace = 1

fig, axs = plt.subplots(
    nrows=nrows,
    ncols=ncols,
    figsize=(ncols * figsize + figsize * wspace * (ncols - 1), nrows * figsize),
)

# Plot individual values  
sc.pl.violin(
    sdata["gex"], 
    keys="n_genes_by_counts",
    groupby="Compartment",
    jitter=0.4,
    multi_panel=False,
    show=False,
    ax=axs[0]
)
sc.pl.violin(
    sdata["gex"], 
    keys="total_counts",
    groupby="Compartment",
    jitter=0.4,
    multi_panel=False,
    show=False,
    ax=axs[1]
)
sc.pl.violin(
    sdata["gex"], 
    keys="pct_counts_mt",
    groupby="Compartment",
    jitter=0.4,
    multi_panel=False,
    show=False,
    ax=axs[2]
)

sc.pl.violin(
    sdata["gex"], 
    keys="scrublet_score",
    groupby="Compartment",
    jitter=0.4,
    multi_panel=False,
    show=False,
    ax=axs[3]
)


#n_genes_total (i.e. number of genes per cell)
axs[0].axhline(y=200, linewidth = 1.5, color = "black", linestyle = "--")
axs[0].axhline(y=3000, linewidth = 1.5, color = "black", linestyle = "--")

#number of total reads 
axs[1].axhline(y=400, linewidth = 1.5, color = "black", linestyle = "--")
axs[1].axhline(y=4000, linewidth = 1.5, color = "black", linestyle = "--")

#mitochondrial gene expression
axs[2].axhline(y=10, linewidth = 1.5, color = "black", linestyle = "--")

#mitochondrial gene expression
axs[3].axhline(y=0.5, linewidth = 1.5, color = "black", linestyle = "--")

# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%

In [ ]:
# ## filter to remove doublets
mu.pp.filter_obs(sdata["gex"], 'n_genes_by_counts', lambda x: (x < 3000)) 
mu.pp.filter_obs(sdata["gex"], 'total_counts', lambda x: (x < 4000))
mu.pp.filter_obs(sdata["gex"], 'scrublet_score', lambda x: (x < 0.5))
sdata.update()

In [ ]:
# Compute medians per group
medians = sdata["gex"].obs.groupby("chain_pairing")["scrublet_score"].median()

# Sort groups by median (ascending)
sorted_groups = medians.sort_values().index.tolist()

# --- Plot violin with small boxplot inside ---
with plt.rc_context({"figure.figsize": (6, 4)}):
    ax = sc.pl.violin(
        sdata["gex"],
        keys="scrublet_score",
        groupby="chain_pairing",
        order=sorted_groups,  # sorted by median
        multi_panel=False,
        rotation=45,
        show=False,
        stripplot=True,      # no individual dots
        inner="box"           # small boxplot inside violin
    )

# Rotate x-axis labels
plt.xticks(rotation=45, ha='right')
plt.ylabel("scrublet_score")
plt.legend(loc='upper right', bbox_to_anchor=(1.35, 1))

# Rasterize violin and box shapes for PDF
for col in ax.collections:
    col.set_rasterized(True)

# Save figure
plt.tight_layout()
plt.savefig(
    "../../05_plots/11_CD8annotation/1_Violin_chain_pairing.pdf",
    bbox_inches="tight",
    dpi=300
)
plt.show()


#Cluster 28 do have more counts - but this cluster has a very low fraction of two-chain TCRs. So I think it's okay. 

# CITEseq annotations

In [ ]:
# Proteins to transfer
proteins = [
    "CD45RA_adt",
    "CD4_adt",
    "CD8_adt",
    "CD62L_adt",
    "CD95_adt",
]

# Extract protein expression matrix
prot_df = pd.DataFrame(
    sdata["prot"][:, proteins].X.toarray()
    if hasattr(sdata["prot"].X, "toarray") else sdata["prot"][:, proteins].X,
    index=sdata["prot"].obs_names,
    columns=[f"prot_{p}" for p in proteins]
)

# Transfer into gex.obs
sdata["gex"].obs = sdata["gex"].obs.join(prot_df)

In [ ]:
#Also transfer the donor-dependent assignments of surface differentiation status. 
sdata["gex"].obs["Differentiation"] = sdata["prot"].obs["Differentiation"].copy()

# Removing CD8 T cells (Mostly CD4/CD8 T cells) 
While voting, some CD4 T cells expressing prot_CD8 or CD8A/B were included as CD4 T cells, because their clonotype exists primarily among clusters with strong CD4 expression. However, in the interest of simplifying our analysis and removing all potential sources of noise, we exclude cells with CD8A/B and prot_CD8 expression. 

In [ ]:
### %%% Protein expression measured by a CD8 CITEseq antibody %%%%
ax = sc.pl.scatter(
    sdata['gex'],
    x="prot_CD4_adt",
    y="prot_CD8_adt",
    size=40,
    color="cd4cd8_cs_assignment",
    show=False
)

ax.axhline(y=11, color='black', linestyle='--', lw=1)
ax.axvline(x=9, color='black', linestyle='--', lw=1)
plt.ylim(-10, 45)

plt.show()

In [ ]:
## Do I get any CD4 T cells in my CD8 gate?
obs_df = sdata["gex"].obs.copy()
obs_df[(obs_df["prot_CD8_adt"] > 10.5)][["cd4cd8_cs_assignment"]].value_counts()

#Threshold 10.5 seems acceptable as a one-threshold cutoff

In [ ]:
## The CD8 T cell outliers in black are all from CSF of patient, where I elected not to use CITEseq information because of the cell loss associated with staining CSF cells. 

obs_df = sdata["gex"].obs.copy()
obs_df[(obs_df["prot_CD8_adt"] > 10.5) & (obs_df["prot_CD4_adt"] < 9)]

### Annotate CD8_adt expression

In [ ]:
sdata["gex"].obs["CD8_adt_expression"] = pd.Categorical(np.where(sdata["gex"].obs["prot_CD8_adt"] > 10.5, "CD8_adt", "negative"))

### Annotate CD8A expression

In [ ]:
# Get the index of the CD4 and CD8 genes
cd8_idx = np.where(sdata["gex"].var_names == "CD8A")[0][0]

# Get expression values
cd8_expr = sdata["gex"].X[:, cd8_idx]

# If X is sparse, convert to array and flatten
cd8_expr = cd8_expr.toarray().flatten() if not isinstance(cd8_expr, np.ndarray) else cd8_expr.flatten()

# Create boolean mask for CD4_gex_assignment: CD4 > 1 
cd8_gex_positive = (cd8_expr > 0.1)

# Assign category: "CD4" or "other" for CD4_gex_assignment
sdata["gex"].obs["CD8A_expr"] = pd.Categorical(
             np.where(cd8_gex_positive, "CD8A", "negative"))

### Annotate CD8B expression too

In [ ]:
# Get the index of the CD4 and CD8 genes
cd8_idx = np.where(sdata["gex"].var_names == "CD8B")[0][0]

# Get expression values
cd8_expr = sdata["gex"].X[:, cd8_idx]

# If X is sparse, convert to array and flatten
cd8_expr = cd8_expr.toarray().flatten() if not isinstance(cd8_expr, np.ndarray) else cd8_expr.flatten()

# Create boolean mask for CD4_gex_assignment: CD4 > 1 
cd8_gex_positive = (cd8_expr > 0.1)

# Assign category: "CD4" or "other" for CD4_gex_assignment
sdata["gex"].obs["CD8B_expr"] = pd.Categorical(
             np.where(cd8_gex_positive, "CD8B", "negative"))

### Visualize CD8A/CD8B

In [ ]:
color_map = {
    "CD8B": "#1f77b4",       
    "negative" : "#7f7f7f"
    
}

# Ensure category order is stable
sdata["gex"].obs["CD8B_expr"] = sdata["gex"].obs["CD8B_expr"].cat.set_categories(
    ["CD8B", "negative"]
)

# Register colors in AnnData
sdata["gex"].uns["CD8B_expr_colors"] = [
    color_map[c] for c in sdata["gex"].obs["CD8B_expr"].cat.categories
]


ax = sc.pl.scatter(
    sdata['gex'],
    x="CD4",
    y="CD8B",
    size=40,
    color="CD8B_expr",
    show=False
)

ax.axhline(y=0.1, color='black', linestyle='--', lw=1)
ax.axvline(x=0.1, color='black', linestyle='--', lw=1)
plt.ylim(-0.5, 5)
plt.show()

In [ ]:
color_map = {
    "CD8A": "#1f77b4",       
    "negative" : "#7f7f7f"
    
}

# Ensure category order is stable
sdata["gex"].obs["CD8A_expr"] = sdata["gex"].obs["CD8A_expr"].cat.set_categories(
    ["CD8A", "negative"]
)

# Register colors in AnnData
sdata["gex"].uns["CD8A_expr_colors"] = [
    color_map[c] for c in sdata["gex"].obs["CD8A_expr"].cat.categories
]


ax = sc.pl.scatter(
    sdata['gex'],
    x="CD4",
    y="CD8A",
    size=40,
    color="CD8A_expr",
    show=False
)

ax.axhline(y=0.1, color='black', linestyle='--', lw=1)
ax.axvline(x=0.1, color='black', linestyle='--', lw=1)
plt.ylim(-0.5, 5)
plt.show()

## How many cells am I excluding? 
Answer: 1835

In [ ]:
obs_df = sdata["gex"].obs.copy()
obs_df = obs_df[["CD8A_expr", "CD8B_expr", "cd4cd8_cs_assignment", "CD8_adt_expression"]]

obs_df = obs_df[(obs_df["CD8A_expr"] == "CD8A") | (obs_df["CD8B_expr"] == "CD8B") | (obs_df["cd4cd8_cs_assignment"].isin(["CD8_adt", "CD4_adt/CD8_adt"])) | (obs_df["CD8_adt_expression"] == "CD8_adt")]
obs_df.value_counts().sum()


In [ ]:
# Filter
mu.pp.filter_obs(sdata["gex"], 'cd4cd8_cs_assignment', lambda x: (~x.isin(["CD8_adt", "CD4/CD8_adt"])))
mu.pp.filter_obs(sdata["gex"], 'CD8B_expr', lambda x: (x != "CD8B"))
mu.pp.filter_obs(sdata["gex"], 'CD8A_expr', lambda x: (x != "CD8A"))
mu.pp.filter_obs(sdata["gex"], 'CD8_adt_expression', lambda x: (x != "CD8_adt"))
sdata.update()

In [ ]:
sdata["gex"].obs[["CD8B_expr"]].value_counts()

# CD4 Annotations

In [ ]:
df_total = sdata["gex"].obs.groupby(["Compartment", "catpat", "donor"], observed=True).size()
df_total

# Clean up

In [ ]:
#All genes must be present with at least 1 cell before normalization
mu.pp.filter_var(sdata["gex"], 'n_cells_by_counts', lambda x: x > 1)
sdata.update()

In [ ]:
###Clean-up.
#Make sure only the indexes recorded in GEX are used in the o*ther modalities too
index_list = sdata["gex"].obs.index.tolist()

## define a cell index
sdata["airr"].obs["cell_index"] = sdata["airr"].obs.index
sdata["prot"].obs["cell_index"] = sdata["prot"].obs.index

## ".isin" using indexes does not seem to work on a global level, but if I annotate a column and then filter in place, maybe then it works. 
sdata["airr"].obs["filter_index"] = pd.Categorical(np.where(sdata["airr"].obs["cell_index"].isin(index_list), "GEX_present", "GEX_absent"))
sdata["prot"].obs["filter_index"] = pd.Categorical(np.where(sdata["prot"].obs["cell_index"].isin(index_list), "GEX_present", "GEX_absent"))

#Enforce that sdata["airr"].obs and sdata["prot"].obs only contain information on those indexes that exists in GEX
mu.pp.filter_obs(sdata["airr"], 'filter_index', lambda x: x == "GEX_present")
mu.pp.filter_obs(sdata["prot"], 'filter_index', lambda x: x == "GEX_present")

sdata.update()

In [ ]:
# I renormalize as it is recommended by sc-best-practices when merging data.
sdata["gex"].X = sdata["gex"].layers["counts"].copy()
sc.pp.normalize_total(sdata["gex"], target_sum=1e4, exclude_highly_expressed=False)   #1e4 is important. Do not change
sc.pp.log1p(sdata["gex"])

In [ ]:
## Calculate clone_size = number of rows per clone_id
sdata["gex"].obs["clone_size"] = sdata["gex"].obs.groupby("clone_handle", observed=True)["clone_handle"].transform("count")

#Mask "NaN" containing clones
sdata["gex"].obs["clone_size"] = np.where(sdata["gex"].obs["clone_handle"].str.contains("nan"), 0, sdata["gex"].obs["clone_size"])

## Share labels with airr
sdata["airr"].obs["clone_size"] = sdata["gex"].obs["clone_size"].copy()

# Select highly variable genes
* https://scanpy.readthedocs.io/en/stable/generated/scanpy.pp.highly_variable_genes.html
* https://scanpy.readthedocs.io/en/stable/api/generated/scanpy.pp.neighbors.html
* https://www.sc-best-practices.org/cellular_structure/integration.html

Data integration when experimental parameters vary. Ours experimental parameters are the same for each batch (apart from Tcsf being fresh cells, and Tpb being rested). We prefer to work with unintegrated data wherever possible to preserve differences between samples. The link describes the re-normalization of of genes, selection of hvg, ect. we need to do the same. 

### Batch aware highly variable genes using default functions

* Source: https://scanpy.readthedocs.io/en/stable/generated/scanpy.pp.highly_variable_genes.html
  
_For flavor='seurat_v3_paper', genes are first sorted by the number of batches a gene is a HVG, with ties broken by the median (across batches) rank_

* Source: https://scanpy.readthedocs.io/en/stable/generated/scanpy.pp.highly_variable_genes.htm

**batch_key**
str | None (default: None)
If specified, highly-variable genes are selected within each batch separately and merged. This simple process avoids the selection of batch-specific genes and acts as a lightweight batch correction method. For **all flavors**, except seurat_v3, genes are first sorted by how many batches they are a HVG. For dispersion-based flavors ties are broken by normalized dispersion. For flavor = 'seurat_v3_paper', ties are broken by the median (across batches) rank based on within-batch normalized variance.

**For flavor="seruat" (default)**: Effector genes of known function are a big part of the HVGs with high mean expression and dispersion values fx, GZMK and PRF are both above 5 in mean expression value. Many low dispersion genes are unknown of B / macrophage genes. I will prioritize a method that includes many known T cell genes

In [ ]:
#Removes columns of the var table. This is to rebuild the HVG table. 
sdata["gex"].var = sdata["gex"].var[[]]

## What is a batch?
## We want a donor aware + batch aware intergration (donor + batch)
sdata["gex"].obs["batch_handle"] = sdata["gex"].obs["batch"].astype(str) + "_" + sdata["gex"].obs["donor"].astype(str)

## Default highly variable genes
sc.pp.highly_variable_genes(sdata["gex"], 
                            n_top_genes=6000, #!! 5000 genes creates a spread out umap. 7500 HVGs creates a too dense umap.
                            flavor='seurat',  #This mimics select intergration features
                            batch_key="batch_handle",   #If a batch was "batch" - then the CITEseq multiplexed instances would immediate mark donor-specific expression of AC004448.2 and OOEP fx. this is not intended. 
                            span=1)

In [ ]:
Export_HVG = sdata["gex"].var[sdata["gex"].var["highly_variable"]==True].sort_values(by="highly_variable_nbatches", ascending = True)
print(len(Export_HVG))
Export_HVG.to_csv("..\\..\\06_csv_outputs\\HVG\\CD4_HVGs_export_before_TCR_filter.csv")

## Remove TCR genes from the list of highly variable genes

In [ ]:
#Bring gene names from index into a column
sdata["gex"].var = sdata["gex"].var.reset_index()

# Optionally rename the new column
sdata["gex"].var = sdata["gex"].var.rename(columns={"index": "genes"})

In [ ]:
# Rename the highly_varaible_intersection cells according to an ifelse statement
tcr_gene_prefixs = ['TRAV', 'TRAJ', 'TRBV', "TRBD", "TRBJ"] 

pattern = r"^(" + "|".join(tcr_gene_prefixs) + ")"
mask = sdata["gex"].var["genes"].str.contains(pattern, regex=True)
masked_genes = sdata["gex"].var[mask]["genes"].tolist() #Works - it's giving me the exact genes I want except the D-region genes (although there is only 2? I will ask Kilian about it. Lea says there is something like that with mice, that htey don't express it)

#Modify the initial list of highly variable genes to exclude TCR genes
sdata["gex"].var["highly_variable"] = np.where(sdata["gex"].var["genes"].isin(masked_genes), False, sdata["gex"].var["highly_variable"])

# Return the genes to index
sdata["gex"].var = sdata["gex"].var.set_index(["genes"], drop=True)

In [ ]:
Export_HVG = sdata["gex"].var[sdata["gex"].var["highly_variable"]==True].sort_values(by="highly_variable_nbatches", ascending = True)
print(len(Export_HVG))
Export_HVG.to_csv("..\\..\\06_csv_outputs\\HVG\\CD4_HVGs_export_after_TCR_filter.csv")

## PCR, neighours and **over**clustering
### Calculate neighbors
https://scanpy.readthedocs.io/en/stable/api/generated/scanpy.pp.neighbors.html
- n_neighbors int (default: 15)

    The size of local neighborhood (in terms of number of neighboring data points) used for manifold approximation. **Larger values result in more global views of the manifold**, while smaller values result in more local data being preserved. In general values should be in the range 2 to 100. If knn is True, number of nearest neighbors to be searched. If knn is False, a Gaussian kernel width is set to the distance of the n_neighbors neighbor.

#### General advice: Larger datasets benefit from larger n_pcs values! Source: https://satijalab.org/seurat/archive/v3.0/mca
- Here they use n_pcs = 75 for 242.000 cells, and they are generally very lax about where to set the cutoff on the elbow plot.

Advice: Try working with larger n_pcs. You can decrease the n_neighbors to 10, if you want to reproduce Seurats behavior

In [ ]:
# Run PCA
sc.pp.pca(sdata["gex"], svd_solver="arpack", use_highly_variable=True, n_comps=60)

In [ ]:
# Visualize pca
sc.pl.pca_scatter(sdata["gex"], color=["total_counts", "GZMK", "GZMB", "GZMA"])

In [ ]:
sc.pl.pca_variance_ratio(sdata["gex"], n_pcs=60, log=True) #Log tranformeD. 

In [ ]:
# Set n_pcs based on a visual impression
sc.pp.neighbors(sdata["gex"], n_pcs=45)

In [ ]:
sc.tl.leiden(sdata["gex"], key_added="leiden_4", resolution=4.0)

## Calculate umap

In [ ]:
sc.tl.umap(sdata["gex"], spread=1)

# Reporting on T-T doublets

In [ ]:
confusion_matrix = pd.crosstab(
    sdata["gex"].obs["leiden_4"],
    sdata["gex"].obs["chain_pairing"],
    rownames=['leiden_4'],
    colnames=['chain_pairing']
)


# Calculate row-wise percentages for the "two full chains" column
confusion_matrix["percent_two full chains"] = (
    confusion_matrix["two full chains"] / confusion_matrix.sum(axis=1)
) * 100

confusion_matrix


## Doublet QC plots

In [ ]:
# Compute medians per group
medians = sdata["gex"].obs.groupby("leiden_4")["n_genes_by_counts"].median()

# Sort groups by median (ascending)
sorted_groups = medians.sort_values().index.tolist()

# --- Plot violin with small boxplot inside ---
with plt.rc_context({"figure.figsize": (10, 4)}):
    ax = sc.pl.violin(
        sdata["gex"],
        keys="n_genes_by_counts",
        groupby="leiden_4",
        order=sorted_groups,  # sorted by median
        multi_panel=False,
        rotation=45,
        show=False,
        stripplot=False,      # no individual dots
        inner="box"           # small boxplot inside violin
    )

# Rotate x-axis labels
plt.xticks(rotation=45, ha='right')
plt.ylabel("n_genes_by_counts")

# Rasterize violin and box shapes for PDF
for col in ax.collections:
    col.set_rasterized(True)

# Save figure
plt.tight_layout()
# plt.savefig(
#     "..\\..\\05_plots\\08_AfterAntwerp\\88_Total_reads_L2_box.pdf",
#     bbox_inches="tight",
#     dpi=300
# )
plt.show()


#Cluster 28 do have more counts - but this cluster has a very low fraction of two-chain TCRs. So I think it's okay. 

In [ ]:
# Compute medians per group
medians = sdata["gex"].obs.groupby("leiden_4")["scrublet_score"].median()

# Sort groups by median (ascending)
sorted_groups = medians.sort_values().index.tolist()

# --- Plot violin with small boxplot inside ---
with plt.rc_context({"figure.figsize": (10, 4)}):
    ax = sc.pl.violin(
        sdata["gex"],
        keys="scrublet_score",
        groupby="leiden_4",
        order=sorted_groups,  # sorted by median
        multi_panel=False,
        rotation=45,
        show=False,
        stripplot=False,      # no individual dots
        inner="box"           # small boxplot inside violin
    )

# Rotate x-axis labels
plt.xticks(rotation=45, ha='right')
plt.ylabel("scrublet_score")

# Rasterize violin and box shapes for PDF
for col in ax.collections:
    col.set_rasterized(True)

# Save figure
plt.tight_layout()
# plt.savefig(
#     "..\\..\\05_plots\\08_AfterAntwerp\\88_Total_reads_L2_box.pdf",
#     bbox_inches="tight",
#     dpi=300
# )
plt.show()


#Cluster 28 do have more counts - but this cluster has a very low fraction of two-chain TCRs. So I think it's okay. 

In [ ]:
sc.pl.umap(sdata["gex"], color=["leiden_4"], groups=["40"], size = 10, legend_fontweight= 'bold', frameon= False , vmin=0, legend_fontoutline=2, show=True)

## Return cell numbers

In [ ]:
## How many cells
sdata["gex"].obs.groupby(["final_ann"]).size()

In [ ]:
## How many cells
sdata["gex"].obs.groupby(["chain_pairing"]).size()

# Remove the doublet cluster
We typically want to use the scrublet score as an indicator whether or not a cluster is enriched for doublets. A pure doublet cluster will have the full population of cells above the median of all other clusters (in my experience). Here we only see a trend for 2 clusters, and so I do not exclude any additional cells here other than the cells already excluded in the initial QC. 

In [ ]:
### Exclude doublets 

mu.pp.filter_obs(sdata["gex"], "leiden_4", lambda x: (x != "40")) 

#Remove unused cateorgies
sdata["gex"].obs["leiden_4"] = sdata["gex"].obs["leiden_4"].cat.remove_unused_categories()
sdata.update()

# Redo preprocessing and make final umap/clusters

In [ ]:
#Removes columns of the var table. This is to rebuild the HVG table. 
sdata["gex"].var = sdata["gex"].var[[]]

## What is a batch?
## We want a donor aware + batch aware intergration (donor + batch)
sdata["gex"].obs["batch_handle"] = sdata["gex"].obs["batch"].astype(str) + "_" + sdata["gex"].obs["donor"].astype(str)

## Default highly variable genes
sc.pp.highly_variable_genes(sdata["gex"], 
                            n_top_genes=6500, #!! 5000 genes creates a spread out umap. 7500 HVGs creates a more dense umap.
                            flavor='seurat',  #This mimics select intergration features
                            batch_key="batch_handle",   #If a batch was "batch" - then the CITEseq multiplexed instances would immediate mark donor-specific expression of AC004448.2 and OOEP fx. this is not intended. 
                            span=1)

Export_HVG = sdata["gex"].var[sdata["gex"].var["highly_variable"]==True].sort_values(by="highly_variable_nbatches", ascending = True)
print(len(Export_HVG))
Export_HVG.to_csv("..\\..\\06_csv_outputs\\HVG\\CD4_HVGs_export_before_TCR_filter.csv")

#### Remove TCR genes from the list of highly variable genes

In [ ]:
#Bring gene names from index into a column
sdata["gex"].var = sdata["gex"].var.reset_index()

# Optionally rename the new column
sdata["gex"].var = sdata["gex"].var.rename(columns={"index": "genes"})

# Rename the highly_varaible_intersection cells according to an ifelse statement
tcr_gene_prefixs = ['TRAV', 'TRAJ', 'TRBV', "TRBD", "TRBJ"] 

pattern = r"^(" + "|".join(tcr_gene_prefixs) + ")"
mask = sdata["gex"].var["genes"].str.contains(pattern, regex=True)
masked_genes = sdata["gex"].var[mask]["genes"].tolist() #Works - it's giving me the exact genes I want except the D-region genes (although there is only 2? I will ask Kilian about it. Lea says there is something like that with mice, that htey don't express it)

#Modify the initial list of highly variable genes to exclude TCR genes
sdata["gex"].var["highly_variable"] = np.where(sdata["gex"].var["genes"].isin(masked_genes), False, sdata["gex"].var["highly_variable"])

# Return the genes to index
sdata["gex"].var = sdata["gex"].var.set_index(["genes"], drop=True)

In [ ]:
Export_HVG = sdata["gex"].var[sdata["gex"].var["highly_variable"]==True].sort_values(by="highly_variable_nbatches", ascending = True)
print(len(Export_HVG))
Export_HVG.to_csv("..\\..\\06_csv_outputs\\HVG\\CD4_HVGs_export_after_TCR_filter.csv")

#### Redo PCA, umap and leiden

In [ ]:
# Run PCA
sc.pp.pca(sdata["gex"], svd_solver="arpack", use_highly_variable=True, n_comps=60)
sc.pl.pca_variance_ratio(sdata["gex"], n_pcs=60, log=True) #Log tranformeD. 


In [ ]:
# Set n_pcs based on a visual impression
sc.pp.neighbors(sdata["gex"], n_pcs=45)

#umap
sc.tl.umap(sdata["gex"], spread=1)

# Leiden clustering
Things I tried:
- 6500 HVGs, 45 PCS, leiden cluster 1.7 was capable of finding trace presence of Tregs and ITGA4/ITGB1 high cells. This is highly important. But it had clusters that were highly intermixed in PB.
- 5000 HVGs, 45 PCS, leiden clusters were not capable of finding trace presence of Tregs in CSF nor were they capable of finding the ITGA4/ITB1 high subpopulation. The PB clusters were more distinct though.
- 5000 HVGs, 30 PCS, leiden clusters again looked more distinct. Fails at identifying CSF Tregs. A large CSF cluster is labelled ITGA4/ITGB1high, which may contain a subset of cells with very high ITGA4/ITGB1 expression, that we cannot isolate. A second rightmost cluster is identified.
- 3000 HVGs, 30 PCS, leiden clusters again looked more distinct. Fails at identifying CSF Tregs. Overall conclusions look similar to 5000 HVGs. 

In [ ]:
# sc.tl.leiden(sdata["gex"], key_added="leiden_1", resolution=1) 
# sc.tl.leiden(sdata["gex"], key_added="leiden_1.1", resolution=1.1) 
# sc.tl.leiden(sdata["gex"], key_added="leiden_1.2", resolution=1.2) 
# # sc.tl.leiden(sdata["gex"], key_added="leiden_1.3", resolution=1.3) 
# sc.tl.leiden(sdata["gex"], key_added="leiden_1.4", resolution=1.4) 
# sc.tl.leiden(sdata["gex"], key_added="leiden_1.5", resolution=1.5)
# sc.tl.leiden(sdata["gex"], key_added="leiden_1.6", resolution=1.6)    
sc.tl.leiden(sdata["gex"], key_added="leiden_1.7", resolution=1.7)    
# sc.tl.leiden(sdata["gex"], key_added="leiden_1.8", resolution=1.8)     
# sc.tl.leiden(sdata["gex"], key_added="leiden_1.9", resolution=1.9)   
# sc.tl.leiden(sdata["gex"], key_added="leiden_2", resolution=2.0)   
# sc.tl.leiden(sdata["gex"], key_added="leiden_1.7.1", resolution=2.1) 
# sc.tl.leiden(sdata["gex"], key_added="leiden_1.7.2", resolution=2.2)
# sc.tl.leiden(sdata["gex"], key_added="leiden_1.7.3", resolution=2.3)
# sc.tl.leiden(sdata["gex"], key_added="leiden_1.7.4", resolution=2.4)
# sc.tl.leiden(sdata["gex"], key_added="leiden_1.7.5", resolution=2.5)

In [ ]:
sc.settings.set_figure_params(figsize=(4.3, 4))
# sc.pl.umap(sdata["gex"], color=["leiden_1.2", "leiden_1.4", "leiden_1.5", "leiden_1.6", "leiden_1.7", "leiden_1.8", "leiden_1.9", "leiden_2"], size = 5, legend_fontweight= 'bold', frameon= False , legend_fontoutline=2, show=True, legend_loc = "on data")
sc.pl.umap(sdata["gex"], color=["leiden_1.7"], size = 5, legend_fontweight= 'bold', frameon= False , legend_fontoutline=2, show=True, legend_loc = "on data")

### Did the CD8 get excluded? 

In [ ]:
sc.settings.set_figure_params(figsize=(4.3, 4))
sc.pl.umap(sdata["gex"], color=["CD8A", "CD8B", "prot_CD8_adt", "prot_CD4_adt"], size = 10, legend_fontweight= 'bold', frameon= False , legend_fontoutline=2, show=True, legend_loc = "on data", cmap="RdBu_r")

### Can I isolated the GZMK+ cluster in CSF?

In [ ]:
sc.settings.set_figure_params(figsize=(4.3, 4))
sc.pl.umap(sdata["gex"], color=["leiden_1.7"], groups=["14", "19"], size = 10, legend_fontweight= 'bold', frameon= False , legend_fontoutline=2, show=True, legend_loc = "on data", cmap="RdBu_r")

### Can I isolated the FOXP3+ cluster in CSF?

In [ ]:
sc.settings.set_figure_params(figsize=(4.3, 4))
sc.pl.umap(sdata["gex"], color=["FOXP3", "ISG15", "GZMK"], size = 10, legend_fontweight= 'bold', frameon= False , legend_fontoutline=2, show=True, legend_loc = "on data", cmap="RdBu_r")


# Return umaps

In [ ]:
# clusters = sdata["gex"].obs["leiden_1.7"].cat.categories

# print(clusters)  # check order!

# colors = sdata["gex"].uns["leiden_1.7_colors"]

# print(colors)  # check colors

# clusters = ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 
#             '10', '11', '12', '13', '14', '15', '16', '17', 
#             '18', '19', '20']

# colors = ['#023fa5', '#7d87b9', '#bec1d4', '#d6bcc0', '#bb7784', '#8e063b',
#           '#4a6fe3', '#8595e1', '#b5bbe3', '#e6afb9', '#e07b91', '#d33f6a',
#           '#11c638', '#8dd593', '#c6dec7', '#ead3c6', '#f0b98d', '#ef9708',
#           '#0fcfc0', '#9cded6', '#d5eae7']

# color_dict = dict(zip(clusters, colors))

color_dict= {'0': '#023fa5',
 '1': '#7d87b9',
 '2': '#bec1d4',
 '3': '#d6bcc0',
 '4': '#bb7784',
 '5': '#8e063b',
 '6': '#4a6fe3',
 '7': '#8595e1',
 '8': '#b5bbe3',
 '9': '#e6afb9',
 '10': '#e07b91',
 '11': '#d33f6a',
 '12': '#11c638',
 '13': '#8dd593',
 '14': '#ef9708',
 '15': '#ead3c6',
 '16': '#f0b98d',
 '17': '#c6dec7',
 '18': '#0fcfc0',
 '19': '#9cded6',
 '20': "#C7C7C7"}  #'#d5eae7'

color_dict = {

    # --- TN (soft / pastel) ---
    '0':  '#dbe9f6',  # light blue
    '1':  '#fde0c5',  # light orange
    '3':  '#e2d9f3',  # light purple
    '5':  '#D7BCA6',  # muted beige

    # --- TM (memory spectrum: green → blue) ---
    '2':  '#3182bd',  # #74c476 TCM-like green
    '4':  '#9ecae1',  # TSCM-like light green
    '11': '#a1d99b',  # TEM blue
    '12': '#003FFF',  # TEM/TCM light blue

    # --- Treg and CTL (neutral / distinct) ---
    '14': '#8C6D31',  # Treg 8c6d31 brownish
    '18': '#6D469F',  # (cytotoxic alt lineage)

    # --- TCSF (warm spectrum: orange → red) ---
    '6':  '#fdae6b',  # orange
    '7':  '#bcbd22',  # darker orange
    '8':  '#d6616b',  # muted red
    '9':  '#8D0505',  #  C00000 strong red
    '10': '#843c39',  # deep red-brown
    '15': '#ff9896',  # light red
    '16': '#FF740F',  # olive (kept for diversity)
    '19': '#0fcfc0',  # dark red  8b0000

    # --- TAct (pink / activation + grey reserved) ---
    '13': '#f768a1',  # pink
    '17': '#c51b8a',  # strong magenta
    '20': '#C7C7C7'   # grey (reserved / low-quality / ribo)
}

## Overwrite the leiden_1.7_colors
cats = sdata["gex"].obs["leiden_1.7"].cat.categories
sdata["gex"].uns["leiden_1.7_colors"] = [color_dict[c] for c in cats]

In [ ]:
sc.settings.set_figure_params(figsize=(4.3, 4))
fig_umap = sc.pl.umap(sdata["gex"], color=["leiden_1.7"], size = 5, legend_fontweight= 'bold', frameon= False ,legend_fontoutline=2, show=False, legend_loc="on data", cmap="RdBu_r", return_fig=True)
fig_umap.savefig( "../../05_plots/15_CD4annotation/1_CD4leiden_umap.pdf", bbox_inches="tight", dpi=400)
plt.show()


# Cluster annotation
## Step 1: Start by highlighting all clusters and removing an outlier cluster

In [ ]:
#Maximum
CD4_markers = {
        "CITEseq": ["prot_CD45RA_adt", "prot_CD62L_adt","prot_CD95_adt"],
        "IFN-sig": [ "IFI6", "ISG15"], 
        "Ribosomal & mito": ["RPL30", "RPL28", "RPL11", "MT-ATP8", "MT-ND5"],
        "Naive & memory markers": [ "TCF7", "LEF1", "SELL", "CCR7", "IL7R", "KLF2", "MYC", "BACH2", "LTB"],
        "Effector": ["CCL5", "GZMB", "GZMK",  "PDCD1", "TNFSF13B", "TNFSF8"],
        "Th1" : ["TBX21", "IFNG", "CXCR3"], 
        "Th2" : ["GATA3", "IL4"],
        "Th17" : ["RORC", "IL17A"],     
        "Treg": ["FOXP3", "CTLA4", "IL2RA"], 
        "Tfh": ["CXCR5", "BCL6"], 
        "Homing": ["CXCR3", "CXCR4", "CXCR5", "CXCR6", "CCR4", "CCR5", "CCR6", "CX3CR1"],
        "Integrins": ["ITGA1", "ITGA4", "ITGB1", "ITGB2", "ITGB7", "ITGAE","ITGAL", "ITGAM", "ITGAX"], 
        "Cytokines": ["TNF", "IFNG", "IL2", "CSF2"],
        "Activation": ["CD69", "TNFRSF4", "BHLHE40", "CREM", "NR4A1"], 
        "TNF-signaling" : ["TNFAIP3"],
        "NFkB signaling": ["NFKB1", "NFKB2","NFKBIA", "RELA", "RELB"], 
        "NFAT/AP-1 signaling": ["NFATC1", "NFATC2",  "FOS", "FOSB", "JUN", "JUNB", "JUND", "TGFB1", "MAF", "MAFG", "ATF2", "BATF", "BATF2", "BATF3"],
        "Survival": ["BCL2"], 
        "Other": ["TXNIP", "ARHGAP15", "S100A4", "CD52"], 
    "T cells": ["CD3E", "IL32", "LCK", "CD4", "CD8A"]}


# # # Define a custom order:
# Cluster_order = ["0", "1", "6", "7",  #Tn
#                  "8",  "5", "10",   #Tmem
#                  "12",   #Treg
#                  "16",   #CTL
#                  "9",      #TNFRSF4hi
#                   "11",   "15", "18",            #NFKsig / Ribo low
#                  "2", "4",  "13", "3", "17",  "14", ]     #Tcsf]   

# # #Implement order
# sdata["gex"].obs["leiden_1.7"] = sdata["gex"].obs["leiden_1.7"].cat.reorder_categories(
#     Cluster_order, ordered=True
# )

dp = sc.pl.dotplot(
    sdata["gex"],
    CD4_markers,
    groupby="leiden_1.7",
    standard_scale="var",
    dendrogram=False,
    cmap="RdBu_r",
    swap_axes=False,
    show=False,
    return_fig=True,
)

dp.savefig(
    "../../05_plots/15_CD4annotation/2_CD4_bubblechart.pdf",
    bbox_inches="tight",
    dpi=400
)

fig_umap = sc.pl.umap(sdata["gex"], color=["leiden_1.7"], size = 5, legend_fontweight= 'bold', frameon= False ,legend_fontoutline=2, show=False, legend_loc="on data", cmap="RdBu_r", return_fig=True)


## Step 4: (Re-) Annotate using CITEseq
I previously annotated all T cells using CITEseq. 

Now I focus on CD8 T cells (in PB) and readjust gates to fit this subset

In [ ]:
## How many CD4 T cells do I expect? 
len(sdata["gex"].obs[sdata["gex"].obs["Compartment"] == "PB"])

In [ ]:
#Transfer L3_ann_overview to prot
sdata["prot"].obs["leiden_1.7"] =  sdata["gex"].obs["leiden_1.7"].copy()

#Work only with pb cells
sdata_pb = sdata.copy()

mu.pp.filter_obs(sdata_pb["gex"], "Compartment", lambda x: (x == "PB"))

In [ ]:
#It's more correct to not extract it again. We need to use the prot_ values in GEX_obs
df = sdata_pb["gex"].obs[["prot_CD45RA_adt", "prot_CD62L_adt", "prot_CD95_adt", "leiden_1.7", "Compartment"]].copy()

# # -----------------------------------
# # FEATURES TO PLOT
# # -----------------------------------
# features = ["CD45RA_adt", "CD62L_adt", "CD95_adt"]

# # -----------------------------------
# # EXTRACT PER-CELL ADT VALUES
# # -----------------------------------
# # Get column indices in var
# idx = [sdata_pb["prot"].var_names.get_loc(f) for f in features]

# # Handle sparse or dense .X
# X = sdata_pb["prot"].X
# if not isinstance(X, np.ndarray):
#     X = X.toarray()

# # Build dataframe (cells × 2 ADTs)
# df = pd.DataFrame(
#     X[:, idx],
#     columns=features,
#     index=sdata_pb["prot"].obs_names
# )

# # Keep only non-NaN data points
# df = df.dropna()

# # -----------------------------------
# # ADD L3 ANNOTATION
# # -----------------------------------
# df["leiden_1.9"] = sdata_pb["prot"].obs.loc[
#     df.index, "leiden_1.9"
# ]

# # Keep only non-NaN data points
# df = df.dropna()

In [ ]:
# -----------------------------------
# PARAMETERS (RAW ADT SPACE)
# -----------------------------------
CD45RA_gate = 12
CD62L_gate = 12
xlim = (-3, 51)
ylim = (-3, 41)

# -----------------------------------
# Example plots to set thresholds
# -----------------------------------
g = sb.jointplot(
    data=df,
    x="prot_CD45RA_adt",
    y="prot_CD62L_adt",
    kind="kde",
    fill=True,
    bw_adjust=0.50,
    levels=10,
    thresh=0.05,
    clip=((-5, 50), (-5, 40)),
    height=5
)

# Move legend outside. 
# sb.move_legend(
#     g.ax_joint,
#     "upper left",
#     bbox_to_anchor=(1.4, 1),
#     frameon=False,
#     title="L3 annotation"
# )

ax = g.ax_joint

# -----------------------------------
# GATING LINES and despine
# -----------------------------------
ax.axvline(CD45RA_gate, color="black", linestyle="--")
ax.axhline(CD62L_gate, color="black", linestyle="--")

sb.despine(ax=ax)

# -----------------------------------
# Limits
# -----------------------------------
ax.set_xlim(*xlim)
ax.set_ylim(*ylim)

# -----------------------------------
# Save the figure
# -----------------------------------
plt.tight_layout()
g.savefig("../../05_plots/15_CD4annotation/3_CD45RA_CD62L_global_gate.pdf", 
    dpi=400,
    bbox_inches="tight"
)
plt.show()

In [ ]:
# -----------------------------------
# PARAMETERS (Normalized ADT SPACE)
# -----------------------------------
CD95_gate = 15
CD62L_gate = 12
xlim = (-10, 51)
ylim = (-3, 41)

# -----------------------------------
# Example plots to set thresholds
# -----------------------------------
g = sb.jointplot(
    data=df,
    x="prot_CD95_adt",
    y="prot_CD62L_adt",
    kind="kde",
    # hue="L3_ann_overview",
    fill=True,
    bw_adjust=0.50,
    levels=10,
    thresh=0.05,
    clip=((-10, 50), (-5, 40)),
    height=5
)

# Move legend outside. 
# sb.move_legend(
#     g.ax_joint,
#     "upper left",
#     bbox_to_anchor=(1.3, 1),
#     frameon=False,
#     title="L3 annotation"
# )

ax = g.ax_joint

# -----------------------------------
# GATING LINES and despine
# -----------------------------------
ax.axvline(CD95_gate, color="black", linestyle="--")
ax.axhline(CD62L_gate, color="black", linestyle="--")

sb.despine(ax=ax)

# -----------------------------------
# Limits
# -----------------------------------
ax.set_xlim(*xlim)
ax.set_ylim(*ylim)

# -----------------------------------
# Save the figure
# -----------------------------------
plt.tight_layout()
g.savefig("../../05_plots/15_CD4annotation/4_CD95_CD62L_global_gate.pdf", 
    dpi=400,
    bbox_inches="tight"
)
plt.show()

In [ ]:
# -----------------------------------
# Collective gates
# -----------------------------------
CD95_gate = 15
CD62L_gate = 12
CD45RA_gate = 12

# -----------------------------------
# Add qudrant and CD95 annotation
# -----------------------------------
df["Quadrant"] = pd.Categorical(np.where(
    (df["prot_CD45RA_adt"] >= CD45RA_gate) & (df["prot_CD62L_adt"]  >= CD62L_gate), "Tn-like", np.where(
        (df["prot_CD45RA_adt"] < CD45RA_gate) & (df["prot_CD62L_adt"]  >= CD62L_gate), "Tcm: CD45RA-CD62L+", np.where(
            (df["prot_CD45RA_adt"] < CD45RA_gate) & (df["prot_CD62L_adt"]  < CD62L_gate), "Tem: CD45RA-CD62L-", np.where(
                (df["prot_CD45RA_adt"] >= CD45RA_gate) & (df["prot_CD62L_adt"] < CD62L_gate), "Temra: CD45RA+CD62L-", "error")))))

df["CD95_ann"] = pd.Categorical(np.where(
    df["prot_CD95_adt"] >= CD95_gate, "CD95+", "CD95-"))

df["Quadrant_final"] = pd.Categorical(np.where(
    (df["Quadrant"] == "Tn-like") & (df["CD95_ann"] == "CD95+"), "Tscm: CD45RA+CD62L+CD95+", np.where(
        (df["Quadrant"] == "Tn-like") & (df["CD95_ann"] == "CD95-"), "Tn: CD45RA+CD62L+CD95-", df["Quadrant"])))

df["Quadrant_final"].value_counts()

### Flow-like plots for CITEseq data

In [ ]:
# -----------------------------------
# PARAMETERS (RAW ADT SPACE)
# -----------------------------------
x_gate = 12
y_gate = 12

xlim = (-5, 61)
ylim = (-5, 41)

# -----------------------------------
# LOOP OVER L3 CATEGORIES
# -----------------------------------
for l3 in df["leiden_1.7"].dropna().unique():

    df_sub = df[df["leiden_1.7"] == l3]

    if len(df_sub) < 50:
        print(f"Skipping {l3} (n={len(df_sub)})")
        continue

    x = df_sub["prot_CD45RA_adt"]
    y = df_sub["prot_CD62L_adt"]

    # -----------------------------------
    # JOINT KDE PLOT (new figure per L3)
    # -----------------------------------
    g = sb.jointplot(
        data=df_sub,
        x="prot_CD45RA_adt",
        y="prot_CD62L_adt",
        kind="kde",
        fill=True,
        bw_adjust=0.50,
        levels=10,
        thresh=0.05,
        clip=((-5, 60), (-5, 40)),
        height=5
    )

    ax = g.ax_joint

    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)

    # -----------------------------------
    # GATING LINES
    # -----------------------------------
    ax.axvline(x_gate, color="black", linestyle="--")
    ax.axhline(y_gate, color="black", linestyle="--")

    sb.despine(ax=ax)

    # -----------------------------------
    # QUADRANT PERCENTAGES (PER L3)
    # -----------------------------------
    total = len(df_sub)

    q1 = (x >= x_gate) & (y >= y_gate)
    q2 = (x < x_gate) & (y >= y_gate)
    q3 = (x < x_gate) & (y < y_gate)
    q4 = (x >= x_gate) & (y < y_gate)

    pcts = [
        100 * q1.sum() / total,
        100 * q2.sum() / total,
        100 * q3.sum() / total,
        100 * q4.sum() / total,
    ]

    # -----------------------------------
    # QUADRANT ANNOTATION
    # -----------------------------------
    dx = 0.05 * (xlim[1] - xlim[0])
    dy = 0.10 * (ylim[1] - ylim[0])

    ax.text(x_gate + dx, y_gate + dy, f"{pcts[0]:.1f}%", fontsize=14)
    ax.text(x_gate - 3*dx, y_gate + dy, f"{pcts[1]:.1f}%", fontsize=14)
    ax.text(x_gate - 3*dx, y_gate - 2*dy, f"{pcts[2]:.1f}%", fontsize=14)
    ax.text(x_gate + dx, y_gate - 2*dy, f"{pcts[3]:.1f}%", fontsize=14)

    # -----------------------------------
    # TITLE + EXPORT
    # -----------------------------------
    ax.set_title(f"{l3} (n={total:,})")

    plt.tight_layout()
    plt.savefig(
        f"../../05_plots/15_CD4annotation/5_CD4_CD45RA_CD62L_{l3}.pdf",
        dpi=400,
        bbox_inches="tight"
    )
    plt.show()


# Correspondance between Gex and Prot phenotypes
How well my GEX annotations recapitulate CITEseq annotations?

In [ ]:
df_sub[df_sub["Quadrant_final"] == "error"]

In [ ]:
###### Skip CSF clusters
df_sub = df[~df["leiden_1.7"].isin(["6", "7", "8", "9", "10", "15","16", "19"])]

# Crosstab (rows = Quadrant, columns = L3 annotation)
ct = pd.crosstab(df_sub["leiden_1.7"], df_sub["Quadrant_final"])

# Sort rows by total number of cells
ct = ct.loc[ct.sum(axis=1).sort_values(ascending=True).index]

# Plot horizontal stacked bar chart
ax = ct.plot(
    kind="barh",       # horizontal bars
    stacked=True,
    figsize=(8, 5),
    colormap="tab20"   # nice color palette
)

plt.ylabel("Leiden clusters")  # now vertical axis
plt.xlabel("Number of cells")          # horizontal axis
plt.title("")
plt.legend(title="CITEseq annotations", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig(
    "../../05_plots/15_CD4annotation/6_CD4_CITEseq_reannotation_leiden_1.7.pdf",
    dpi=400,
    bbox_inches="tight"
)
plt.show()


In [ ]:
# Crosstab (rows = L3 annotation, columns = Quadrant)
ct = pd.crosstab(df_sub["Quadrant_final"], df_sub["leiden_1.7"])

quadrant_order = [
    "Tn: CD45RA+CD62L+CD95-",
    "Tscm: CD45RA+CD62L+CD95+",
    "Tcm: CD45RA-CD62L+",
    "Tem: CD45RA-CD62L-",
    "Temra: CD45RA+CD62L-"
]

# Desired column order (example — adjust to your actual cluster labels)
leiden_order = [
    "0", "5", "1", "3", "11", "2", "4", "12", "18", "14", "13", "17", "20"
]

# Apply both row and column ordering
ct = ct.reindex(index=quadrant_order, columns=leiden_order)

# Optionally normalize per row for proportions
ct_norm = ct.div(ct.sum(axis=0), axis=1) * 100  # percentages

# Plot heatmap
plt.figure(figsize=(14, 3))
sb.heatmap(
    ct_norm,
    annot=True,        # show numbers
    fmt=".1f",         # one decimal
    cmap="Reds",    # color map
    linewidths=0.5,
    linecolor="white",
    cbar_kws={"label": "% of cells per L3 annotation"}
)

plt.ylabel("CITEseq annotations")
plt.xlabel("Leiden 1.7")
plt.title("Phenotypic distribution across L3 clusters")
plt.tight_layout()
plt.savefig(
     "../../05_plots/15_CD4annotation/7_CD4_CITEseq_reannotation_correspondance.pdf",
    dpi=400,
    bbox_inches="tight"
)
plt.show()

In [ ]:
### Add the quadrant_final labels back to sdata
# -----------------------------------
# Collective gates
# -----------------------------------
CD95_gate = 15
CD62L_gate = 12
CD45RA_gate = 12


# -----------------------------------
# Add qudrant and CD95 annotation
# -----------------------------------
sdata["gex"].obs["Quadrant"] = pd.Categorical(np.where(
    (sdata["gex"].obs["prot_CD45RA_adt"] >= CD45RA_gate) & (sdata["gex"].obs["prot_CD62L_adt"]  >= CD62L_gate), "Tn-like", np.where(
        (sdata["gex"].obs["prot_CD45RA_adt"] < CD45RA_gate) & (sdata["gex"].obs["prot_CD62L_adt"]  >= CD62L_gate), "Tcm: CD45RA-CD62L+", np.where(
            (sdata["gex"].obs["prot_CD45RA_adt"] < CD45RA_gate) & (sdata["gex"].obs["prot_CD62L_adt"]  < CD62L_gate), "Tem: CD45RA-CD62L-", np.where(
                (sdata["gex"].obs["prot_CD45RA_adt"] >= CD45RA_gate) & (sdata["gex"].obs["prot_CD62L_adt"] < CD62L_gate), "Temra: CD45RA+CD62L-", "error")))))

sdata["gex"].obs["CD95_ann"] = pd.Categorical(np.where(
    sdata["gex"].obs["prot_CD95_adt"] >= CD95_gate, "CD95+", "CD95-"))

sdata["gex"].obs["Quadrant_final"] = pd.Categorical(np.where(
    (sdata["gex"].obs["Quadrant"] == "Tn-like") & (sdata["gex"].obs["CD95_ann"] == "CD95+"), "Tscm: CD45RA+CD62L+CD95+", np.where(
        (sdata["gex"].obs["Quadrant"] == "Tn-like") & (sdata["gex"].obs["CD95_ann"] == "CD95-"), "Tn: CD45RA+CD62L+CD95-", sdata["gex"].obs["Quadrant"])))

sdata["gex"].obs["Quadrant_final"].value_counts()

In [ ]:
obs_df = sdata["gex"].obs.copy()

# Exclude clusters from CSF
obs_df = obs_df[~obs_df["leiden_1.7"].isin(["6", "7", "8", "9", "10", "15", "16", "19"])]
obs_df = obs_df[~obs_df["Compartment"].isin(["CSF"])]

# Get grouped counts per differentiation assignment
sum_df = obs_df.groupby(["Quadrant_final", "leiden_1.7"], observed=True).size().reset_index(name="count")

# Add sums per leiden cluster
sum_df["total"] = sum_df.groupby(["leiden_1.7"],observed=True)["count"].transform("sum")

# Calculate percentages
sum_df["freq"] = 100 * sum_df["count"] / sum_df["total"]

# Pivot to wide format: rows = leiden_1.9, columns = Differentiation
plot_df = (
    sum_df
    .pivot(index="leiden_1.7", columns="Quadrant_final", values="freq")
    .fillna(0)
)

# Sort leiden clusters numerically if needed
plot_df = plot_df.sort_index()

ax = plot_df.plot(
    kind="bar",
    stacked=True,
    figsize=(10, 4),
    width=0.75
)

ax.set_xlabel("leiden_1.7")
ax.set_ylabel("Frequency (%)")
ax.set_title("Differentiation based on CD45RA, CD62L, and CD95 surface expression")

ax.legend(
    title="Subset",
    bbox_to_anchor=(1.05, 1),
    loc="upper left"
)

# edit layout
sb.despine()
ax.grid(False)

plt.tight_layout()
plt.savefig("../../05_plots/15_CD4annotation/8_CD4_CITEseq_gated_phenotypes_UPDATED.pdf", bbox_inches="tight",  dpi=400)
plt.show()

## Step 5: TCR diversity

### Gini index per cluster
The different index metrics are described here: https://scikit.bio/docs/dev/generated/skbio.diversity.alpha.html

The function to calculate these metrics is described here: https://scirpy.scverse.org/en/latest/generated/scirpy.tl.alpha_diversity.html

**Gini index** portrays dominance defined as: Dominance [5] (or concentration) measures the degree that one or a few most abundant species (taxa) represent the great majority of a community. It can be considered as a measure of community unevenness.

In [ ]:
sdata.update()

In [ ]:
###Clean-up.
#Make sure only the indexes recorded in GEX are used in the o*ther modalities too
index_list = sdata["gex"].obs.index.tolist()

## define a cell index
sdata["airr"].obs["cell_index"] = sdata["airr"].obs.index
sdata["prot"].obs["cell_index"] = sdata["prot"].obs.index

## ".isin" using indexes does not seem to work on a global level, but if I annotate a column and then filter in place, maybe then it works. 
sdata["airr"].obs["filter_index"] = pd.Categorical(np.where(sdata["airr"].obs["cell_index"].isin(index_list), "GEX_present", "GEX_absent"))
sdata["prot"].obs["filter_index"] = pd.Categorical(np.where(sdata["prot"].obs["cell_index"].isin(index_list), "GEX_present", "GEX_absent"))

#Enforce that sdata["airr"].obs and sdata["prot"].obs only contain information on those indexes that exists in GEX
mu.pp.filter_obs(sdata["airr"], 'filter_index', lambda x: x == "GEX_present")
mu.pp.filter_obs(sdata["prot"], 'filter_index', lambda x: x == "GEX_present")

sdata.update()

In [ ]:
sdata_gini = sdata.copy()
sdata_gini["airr"].obs["clone_handle"] = sdata_gini["gex"].obs["clone_handle"].copy()
sdata_gini["airr"].obs["leiden_1.7"] = sdata_gini["gex"].obs["leiden_1.7"].copy()
sdata_gini["airr"].obs["donor"] = sdata_gini["gex"].obs["donor"].copy()
sdata_gini["airr"].obs["catpat"] = sdata_gini["gex"].obs["catpat"].copy()

#Make grouping variables
sdata_gini["airr"].obs["donor_leiden_1.7"] = sdata_gini["airr"].obs["donor"].astype(str) + "-" + sdata_gini["airr"].obs["leiden_1.7"].astype(str)

#filter
mu.pp.filter_obs(
    sdata_gini["airr"],
    "clone_id",
    lambda x: ~pd.isna(x)
)

In [ ]:
## %% Calculate the diversity metrics andsave them under a column key. 
ir.tl.alpha_diversity(sdata_gini["airr"], groupby="leiden_1.7", target_col='clone_handle', metric='gini_index', inplace=True, key_added="gini_per_cluster", airr_mod='airr')

In [ ]:
# ## Get diversity as a copy-pastable list per leiden
# df_gini = sdata["airr"].obs[["leiden_1.7", "gini_per_cluster"]].drop_duplicates().reset_index()
# df_gini["gini_mean"] = df_gini.groupby(["leiden_1.7"])["gini_per_cluster"].transform("median")

# df_gini[["leiden_1.7", "gini_mean"]].drop_duplicates().dropna()

In [ ]:
# Drop rows with NaN gini index
df_gini = sdata_gini["airr"].obs[["leiden_1.7", "gini_per_cluster"]].drop_duplicates().reset_index()
df_gini = df_gini.dropna(subset=["gini_per_cluster"])
df_gini["gini_per_cluster"] = pd.to_numeric(df_gini["gini_per_cluster"])

## Use order if you want the order to be decided by highest value
order = df_gini.sort_values("gini_per_cluster",ascending=False)["leiden_1.7"].to_list()

## leiden order for the manual order by Kilian
leiden_order = [
    "0", "5", "1", "3", "11", "2", "4", "12", "18", "14", "13", "17", "20", "7", "6", "8", "15", "16", "19", "9", "10"
]


In [ ]:
df_gini["leiden_1.7"] = df_gini["leiden_1.7"].astype(str)

plt.figure(figsize=(8,5))

sb.barplot(
    data=df_gini,
    x='leiden_1.7',
    y='gini_per_cluster',
    order=leiden_order,
    color='lightgrey'
)

plt.xlabel("leiden_1.7")
plt.ylabel("Gini index")
plt.title("Gini per Leiden cluster")
# edit layout
sb.despine()
ax.grid(False)

plt.tight_layout()
plt.savefig("../../05_plots/11_CD8annotation/17_Gini_per_cluster.pdf", bbox_inches="tight",  dpi=400)
plt.show()

### Normalized Shannon Entropy
The different index metrics are described here: https://scikit.bio/docs/dev/generated/skbio.diversity.alpha.html

For immunologists Shannon entropy is typically explanationed as measuring diversity as a combination of Richness (number of different clones) and Eveness (distribution of the observations). 

In [ ]:
## %% Calculate the diversity metrics andsave them under a column key. 
ir.tl.alpha_diversity(sdata_gini["airr"], groupby="leiden_1.7", target_col='clone_handle', metric='normalized_shannon_entropy', inplace=True, key_added="shannon_per_cluster", airr_mod='airr')

In [ ]:
# Drop rows with NaN gini index
df_gini = sdata_gini["airr"].obs[["leiden_1.7", "shannon_per_cluster"]].drop_duplicates().reset_index()
df_gini = df_gini.dropna(subset=["shannon_per_cluster"])
df_gini["shannon_per_cluster"] = pd.to_numeric(df_gini["shannon_per_cluster"])

## Use order if you want the highest number first
order = df_gini.sort_values("shannon_per_cluster",ascending=False)["leiden_1.7"].to_list()

In [ ]:
df_gini["leiden_1.7"] = df_gini["leiden_1.7"].astype(str)

plt.figure(figsize=(8,5))

sb.barplot(
    data=df_gini,
    x='leiden_1.7',
    y='shannon_per_cluster',
    order=order,
    color='lightgrey'
)

plt.xlabel("leiden_1.7")
plt.ylabel("Normalized Shannon entropy")

# edit layout
sb.despine()
ax.grid(False)

plt.tight_layout()
plt.savefig("../../05_plots/11_CD8annotation/17_Shannon_per_cluster.pdf", bbox_inches="tight",  dpi=400)
plt.show()

### Shannon Entropy PER DONOR
The different index metrics are described here: https://scikit.bio/docs/dev/generated/skbio.diversity.alpha.html

For immunologists Shannon entropy is typically explanationed as measuring diversity as a combination of Richness (number of different clones) and Eveness (distribution of the observations). 

In [ ]:
sdata_gini["airr"].obs.columns

In [ ]:
## %% Calculate the diversity metrics andsave them under a column key. 
ir.tl.alpha_diversity(sdata_gini["airr"], groupby="donor_leiden_1.7", target_col='clone_handle', metric='shannon', inplace=True, key_added="shannon_per_cluster", airr_mod='airr')

In [ ]:
# Drop rows with NaN gini index
df_gini = sdata_gini["airr"].obs[["leiden_1.7", "shannon_per_cluster"]].drop_duplicates().reset_index()
df_gini = df_gini.dropna(subset=["shannon_per_cluster"])
df_gini["shannon_per_cluster"] = pd.to_numeric(df_gini["shannon_per_cluster"])

## Use order if you want the highest number first
## Use leiden_order if you want the manually curated order
order = df_gini.sort_values("shannon_per_cluster",ascending=False)["leiden_1.7"].to_list()

In [ ]:
df_gini["leiden_1.7"] = df_gini["leiden_1.7"].astype(str)

plt.figure(figsize=(8,5))

sb.barplot(
    data=df_gini,
    x='leiden_1.7',
    y='shannon_per_cluster',
    order=leiden_order,
    color='lightgrey'
)

plt.xlabel("leiden_1.7")
plt.ylabel("Shannon entropy")

# edit layout
sb.despine()
ax.grid(False)

plt.tight_layout()
plt.savefig("../../05_plots/11_CD8annotation/17_Shannon_per_cluster.pdf", bbox_inches="tight",  dpi=400)
plt.show()

## Step 5: Work signatures
List of libraries that can be imported: https://maayanlab.cloud/Enrichr/#libraries

This script was made using Claude!

### Tr1 signatures from Geginat
What is the Tr1 signature from the Pulvirenti et al preprint (PI: Geginat)?
- CHIL2, CCL4, CCR5 and PDCD1
- ""Cluster 7 displayed the gene signature of EOMES+Tr1-like cells (Bonnal et al., 2021; De Simone et al., 2021; Gruarin et al., 2019), as evidenced by a highly significant Gene Set Enrichment Analysis (GSEA, Figure 2B). Indeed, several key EOMES+Tr1-like signature genes, namely EOMES (Gruarin et al., 2019) (Figure 2C), CHI3L2 (Bonnal et al., 2021), CCL4, CCR5 and PDCD1 (PD1) (Alfen et al., 2018) were selectively expressed in this cluster (sFigure 4C).""
- Gene set enrichment analysis (GSEA) was performed using the fgsea() function from the fgsea package (v1.20.0). The reference Tr1 associated gene set for the analysis was obtained from our previous studies(Bonnal et al., 2021; De Simone et al., 2021).

In [ ]:
### Bonnal Eomes+ Treg signature
genes = [
    "SH2D1A", "HLA-B", "LYST", "DTHD1", "PACSIN1", "PRR5L", "CD81", "CNN2",
    "IFNG-AS1", "CCL4", "GIMAP4", "GZMA", "SUN2", "LIME1", "AC116366.3",
    "ATP5F1E", "CLEC2B", "HLA-E", "LINC02084", "BTN3A2", "CPNE7", "APMAP",
    "DRAP1", "CD3D", "CCL5", "MLLT3", "RABAC1", "PRF1", "DKK3", "LITAF",
    "GZMK", "PFN1", "NAA50", "RARRES3", "TC2N", "HLA-A", "ABI3", "YARS",
    "FABP5", "LYAR", "HMGB1", "STK17A", "ITM2C", "NUCB2", "CD27", "TBCD",
    "CCR5", "APOBEC3C", "OAZ1", "UBE2L6", "AC243829.4", "CST7", "PTPRCAP",
    "SYNM", "GZMH", "RAC2", "CHI3L2", "NKG7", "CD99", "CXCR3", "IDH2",
    "DUSP4", "SLF1", "GYG1", "CD3E", "ITGB2", "C12orf75", "IL9R", "SEPT9",
    "HLA-DRB1", "HLA-DMA", "TNIP3", "SAMD3", "APOBEC3G", "PTK2B", "F2R",
    "C1orf21", "CYBA", "KLRG1", "PECAM1", "PDCD1", "PTPRC", "RNF213",
    "PDE4DIP", "SH3BGRL3", "HLA-C", "PSMB9", "COTL1", "HCST", "LCP1",
    "EIF4EBP1", "RCAN2", "ATP5MPL", "OASL", "CTSW", "CTSC", "GGA2",
    "EOMES", "CD74", "UBL5", "LY6E", "PLEK", "APOBEC3H", "FAM102A",
    "IFNG", "HLA-DPA1", "HSPB11", "CHST12", "HLA-DPB1", "GZMM",
    "SLAMF7", "REEP5", "CMC1", "LAG3", "ITGA1", "B2M", "GIMAP7",
    "RPS27", "PSMA7", "ATP5IF1", "AOAH"
]

# Cacluate a score
sc.tl.score_genes(sdata["gex"], 
                      genes,  
                      score_name='EOMES+ Treg (Bonnel et al 2021)')

In [ ]:
### De Simone et al 2021 (from Geginat lab) Eomes+ Tr1 signature calculated versus other EOMES+ cells
genes_DeSimone = [
    "GZMK", "CD27", "NELL2", "PASK", "LEF1", "CCR7",
    "GZMA", "HLA-DRA", "CCL5", "HLA-DPA1", "CST7", "NKG7",
    "CD74", "EOMES", "HLA-DMA", "S100A4", "ANXA5", "PLEK",
    "ALOX5AP", "LYAR", "OSTF1", "IL10RA", "TXN", "APOBEC3G",
    "ZBP1", "S100A6", "SH3BGRL3",
    "HSPA1B", 
    #"LOC221136" # is not found
]

# Cacluate a score
sc.tl.score_genes(sdata["gex"], 
                      genes_DeSimone,  
                      score_name='Tr1+ cells (De Simone et al 2021)')

In [ ]:
sc.settings.set_figure_params(figsize=(4.3, 4))
fig = sc.pl.umap(sdata["gex"], color=["EOMES+ Treg (Bonnel et al 2021)", "Tr1+ cells (De Simone et al 2021)"], size = 15, legend_fontweight= 'bold', frameon= False ,legend_fontoutline=2, show=True, legend_loc="on data", cmap="RdBu_r", return_fig=True)
fig.savefig("../../05_plots/15_CD4annotation/10_CD4_umap_pathways_Tr1_cells.pdf", bbox_inches="tight",  dpi=400)

### Apoptosis (KEGG)

In [ ]:
# Retriving a select pathway
k = KEGG()

# Get pathway information
pathway = k.get('hsa04210')  # Apoptosis for humans
parsed = k.parse(pathway)

# Extract genes
genes = []
if 'GENE' in parsed:
    for gene_id, gene_info in parsed['GENE'].items():
        genes.append({
            'id': gene_id,
            'name': gene_info.split(';')[0]
        })

print(f"Found {len(genes)} genes")
gene_list = [d["name"] for d in genes]

# Cacluate a score
sc.tl.score_genes(sdata["gex"], 
                      gene_list,  
                      score_name='Apoptosis (KEGG)')

### Ribosomes (KEGG)

In [ ]:
# Retriving a select pathway
k = KEGG()

# Get pathway information
pathway = k.get('hsa03008')  # Ribosomes
parsed = k.parse(pathway)

# Extract genes
genes = []
if 'GENE' in parsed:
    for gene_id, gene_info in parsed['GENE'].items():
        genes.append({
            'id': gene_id,
            'name': gene_info.split(';')[0]
        })

print(f"Found {len(genes)} genes")
gene_list = [d["name"] for d in genes]

# Cacluate a score
sc.tl.score_genes(sdata["gex"], 
                      gene_list,  
                      score_name='Ribosomes (KEGG)')

### Ribosome biogenesis (GOBP)

In [ ]:
## I use GProfiler for this. 
gp = GProfiler(return_dataframe=True)

go_term = "GO:0042254"

res = gp.convert(
    organism="hsapiens",
    target_namespace="ENSG",
    query=go_term
)

genes = res["name"].unique().tolist()
print("I have this many genes:", len(genes))

# Cacluate a score
sc.tl.score_genes(sdata["gex"], 
                      genes,  
                      score_name='Ribosome biogenesis (GOBP)')

### Translation elongation (GOBP)

In [ ]:
## I use GProfiler for this. 
gp = GProfiler(return_dataframe=True)

go_term = "GO:0006414"

res = gp.convert(
    organism="hsapiens",
    target_namespace="ENSG",
    query=go_term
)

genes = res["name"].unique().tolist()
print("I have this many genes:", len(genes))

# Cacluate a score
sc.tl.score_genes(sdata["gex"], 
                      genes,  
                      score_name='Translational elongation (GOBP)')

### Translation initiation (GOBP)

In [ ]:
## I use GProfiler for this. 
gp = GProfiler(return_dataframe=True)

go_term = "GO:0006413"

res = gp.convert(
    organism="hsapiens",
    target_namespace="ENSG",
    query=go_term
)

genes = res["name"].unique().tolist()
print("I have this many genes:", len(genes))

# Cacluate a score
sc.tl.score_genes(sdata["gex"], 
                      genes,  
                      score_name='Translational initiation (GOBP)')

### Execution phase of apoptosis (GOBP)

In [ ]:
## I use GProfiler for this. 
gp = GProfiler(return_dataframe=True)

go_term = "GO:0097194"

res = gp.convert(
    organism="hsapiens",
    target_namespace="ENSG",
    query=go_term
)

genes = res["name"].unique().tolist()
print("I have this many genes:", len(genes))

# Cacluate a score
sc.tl.score_genes(sdata["gex"], 
                      genes,  
                      score_name='Execution phase of apoptosis (GOBP)')

### Base exicision repair (KEGG)

In [ ]:
# Retriving a select pathway
k = KEGG()

# Get pathway information
pathway = k.get('hsa03410')  # Base excision repair.
parsed = k.parse(pathway)

# Extract genes
genes = []
if 'GENE' in parsed:
    for gene_id, gene_info in parsed['GENE'].items():
        genes.append({
            'id': gene_id,
            'name': gene_info.split(';')[0]
        })

print(f"Found {len(genes)} genes")
gene_list = [d["name"] for d in genes]

# Cacluate a score
sc.tl.score_genes(sdata["gex"], 
                      gene_list,  
                      score_name='Base excision repair (KEGG)')

### Oxidative phosphorylation (KEGG)

In [ ]:
# Retriving a select pathway
k = KEGG()

# Get pathway information
pathway = k.get('hsa00190')  # Oxphos
parsed = k.parse(pathway)

# Extract genes
genes = []
if 'GENE' in parsed:
    for gene_id, gene_info in parsed['GENE'].items():
        genes.append({
            'id': gene_id,
            'name': gene_info.split(';')[0]
        })

print(f"Found {len(genes)} genes")
gene_list = [d["name"] for d in genes]

# Cacluate a score
sc.tl.score_genes(sdata["gex"], 
                      gene_list,  
                      score_name='Oxidative phosphorylation (KEGG)')

### Glycolysis/ gluconeogenesis (KEGG)

In [ ]:
# Retriving a select pathway
k = KEGG()

# Get pathway information
pathway = k.get('hsa00010')  # Base excision repair.
parsed = k.parse(pathway)

# Extract genes
genes = []
if 'GENE' in parsed:
    for gene_id, gene_info in parsed['GENE'].items():
        genes.append({
            'id': gene_id,
            'name': gene_info.split(';')[0]
        })

print(f"Found {len(genes)} genes")
gene_list = [d["name"] for d in genes]

# Cacluate a score
sc.tl.score_genes(sdata["gex"], 
                      gene_list,  
                      score_name='Glycolysis and gluconeogenesis (KEGG)')

### Pentose phosphate pathway (KEGG)

In [ ]:
# Retriving a select pathway
k = KEGG()

# Get pathway information
pathway = k.get('hsa00030')  # Pentose phosphate pathway
parsed = k.parse(pathway)

# Extract genes
genes = []
if 'GENE' in parsed:
    for gene_id, gene_info in parsed['GENE'].items():
        genes.append({
            'id': gene_id,
            'name': gene_info.split(';')[0]
        })

print(f"Found {len(genes)} genes")
gene_list = [d["name"] for d in genes]

# Cacluate a score
sc.tl.score_genes(sdata["gex"], 
                      gene_list,  
                      score_name='Pentose phosphate pathawy (KEGG)')

### Proteasome (KEGG)

In [ ]:
# Retriving a select pathway
k = KEGG()

# Get pathway information
pathway = k.get('hsa03050')  # PRoteasome
parsed = k.parse(pathway)

# Extract genes
genes = []
if 'GENE' in parsed:
    for gene_id, gene_info in parsed['GENE'].items():
        genes.append({
            'id': gene_id,
            'name': gene_info.split(';')[0]
        })

print(f"Found {len(genes)} genes")
gene_list = [d["name"] for d in genes]

# Cacluate a score
sc.tl.score_genes(sdata["gex"], 
                      gene_list,  
                      score_name='Proteasome (KEGG)')

### T cell differentiation (GOBP)

In [ ]:
## I use GProfiler for this. 
gp = GProfiler(return_dataframe=True)

go_term = "GO:0030217"

res = gp.convert(
    organism="hsapiens",
    target_namespace="ENSG",
    query=go_term
)

genes = res["name"].unique().tolist()
print("I have this many genes:", len(genes))

# Cacluate a score
sc.tl.score_genes(sdata["gex"], 
                      genes,  
                      score_name='T cell differentiation (GOBP)')

### T cell activation (GOBP)

In [ ]:
## I use GProfiler for this. 
gp = GProfiler(return_dataframe=True)

go_term = "GO:0042110"

res = gp.convert(
    organism="hsapiens",
    target_namespace="ENSG",
    query=go_term
)

genes = res["name"].unique().tolist()
print("I have this many genes:", len(genes))

# Cacluate a score
sc.tl.score_genes(sdata["gex"], 
                      genes,  
                      score_name='T cell activation (GOBP)')

### Regulation of actin cytoskeleton (KEGG)

In [ ]:
# Retriving a select pathway
k = KEGG()

# Get pathway information
pathway = k.get('hsa04810')  # Regulation of actin cytoskeleton
parsed = k.parse(pathway)

# Extract genes
genes = []
if 'GENE' in parsed:
    for gene_id, gene_info in parsed['GENE'].items():
        genes.append({
            'id': gene_id,
            'name': gene_info.split(';')[0]
        })

print(f"Found {len(genes)} genes")
gene_list = [d["name"] for d in genes]

# Cacluate a score
sc.tl.score_genes(sdata["gex"], 
                      gene_list,  
                      score_name='Regulation of actin cytoskeleton (KEGG)')

### T-cell prolifferation (GOBP)

In [ ]:
## I use GProfiler for this. 
gp = GProfiler(return_dataframe=True)

go_term = "GO:0042098"

res = gp.convert(
    organism="hsapiens",
    target_namespace="ENSG",
    query=go_term
)

genes = res["name"].unique().tolist()
print("I have this many genes:", len(genes))

# Cacluate a score
sc.tl.score_genes(sdata["gex"], 
                      genes,  
                      score_name='T cell prolifferation (GOBP)')

### Cell Cycle-Associated Genes (KEGG)

In [ ]:
# Retriving a select pathway
k = KEGG()

# Get pathway information
pathway = k.get('hsa04110')  # Cytokine-cytokine receptor interaction 
parsed = k.parse(pathway)

# Extract genes
genes = []
if 'GENE' in parsed:
    for gene_id, gene_info in parsed['GENE'].items():
        genes.append({
            'id': gene_id,
            'name': gene_info.split(';')[0]
        })

print(f"Found {len(genes)} genes")
gene_list = [d["name"] for d in genes]

# Cacluate a score
sc.tl.score_genes(sdata["gex"], 
                      gene_list,  
                      score_name='Cell cycle - Homo sapiens (KEGG)')

### Cytokine-cytokine receptor interaction (KEGG)

In [ ]:
# Retriving a select pathway
k = KEGG()

# Get pathway information
pathway = k.get('hsa04060')  # Cytokine-cytokine receptor interaction 
parsed = k.parse(pathway)

# Extract genes
genes = []
if 'GENE' in parsed:
    for gene_id, gene_info in parsed['GENE'].items():
        genes.append({
            'id': gene_id,
            'name': gene_info.split(';')[0]
        })

print(f"Found {len(genes)} genes")
gene_list = [d["name"] for d in genes]

# Cacluate a score
sc.tl.score_genes(sdata["gex"], 
                      gene_list,  
                      score_name='Cytokine-cytokine receptor interaction (KEGG)')

### T CELL_RECEPTOR_SIGNALING_PATHWAY 

In [ ]:
## I use GProfiler for this. 
gp = GProfiler(return_dataframe=True)

go_term = "GO:0050852"

res = gp.convert(
    organism="hsapiens",
    target_namespace="ENSG",
    query=go_term
)

genes = res["name"].unique().tolist()
print("I have this many genes:", len(genes))

# Cacluate a score
sc.tl.score_genes(sdata["gex"], 
                      genes,  
                      score_name='TCR signaling (GOBP)')

### Cytotoxicity and interferon scores

In [ ]:
## Cytotoxicity score
Toxic_score = ["GZMB", "GZMH","GZMA", "PRF1", "GNLY", "NKG7"]

sc.tl.score_genes(sdata["gex"], 
                      Toxic_score,  
                      score_name='Cytotoxic\nscore')


In [ ]:
## CANONICAL_NF_KAPPAB_SIGNAL_TRANSDUCTION (GOBP)
NFKB_sig = [
    "TANK","MIR365A","HUWE1","ADAM8","OPTN","NAMPT","TRIM13","FLOT1","TNIP1","TLR6",
    "TFG","TRIM22","NOD1","TAB1","TRIM38","UBD","LAMTOR5","PRDX4","OLFM4","SIVA1",
    "RBCK1","NFAT5","TRAF3IP2","PLK2","ZMYND11","NEK6","USP20","MALT1","EDAR","LILRB4",
    "PIM2","MID2","FAF1","IRAK3","AKAP13","TMEM106A","CHRNA7","TIRAP","CHUK","CARD16",
    "CCR7","CANT1","IRAK1BP1","CREBBP","SIRPA","PARP1","MIB2","TMC8","PPM1N","TICAM1",
    "CTH","CX3CR1","DAB2IP","CYLD","DDX1","DHX15","DHX36","NLRP6","ZNF675","ECM1",
    "ECT2","EDA","LPAR1","NLRC3","EP300","ESR1","F2R","F2RL1","TMED4","UNC5CL",
    "FKBP1A","CARD8","TRIM32","MAPKBP1","ERC1","TAB2","FLNA","FLOT2","FBXW11","UFL1",
    "ZDHHC17","SIRT1","BRD4","NUP62","IFIT5","DEFB124","ABL1","TAB3","FBXO7","GAPDH",
    "PYDC1","ANKRD17","GJA1","ATP2C1","SLCO3A1","ANGPT1","NKIRAS2","NKIRAS1","LURAP1L","TRIM59",
    "CCDC22","PYCARD","TBK1","GSTP1","CARD10","HDAC1","ANXA4","HMOX1","PRMT2","BIRC2",
    "BIRC3","XIAP","HSPB1","HTR2B","CLEC4D","JMJD8","IRGM","SLC35B2","TICAM2","IKBKB",
    "IL1A","IL1B","IL1R1","IL1RAP","FASLG","ILK","INS","INSR","IRAK1","IRAK2",
    "IRF3","RHOA","SUMO4","GBP7","RHOC","LGALS1","LGALS9","LIMS1","RHOH","MYO18A",
    "LTA","LTB","LTBR","LTF","MIR138-1","MIR140","MIR146A","MIR15A","MIR15B","MIR16-1",
    "MIR195","MIR199A1","MIR21","MIR27A","MIR30C2","ARRB2","MAS1","MAP3K3","CD200","MYD88",
    "NFKB1","NFKB2","NFKBIA","NFKBIB","NFKBIL1","ROR1","TIFAB","C18orf32","PRKN","GOLT1B",
    "IRAK4","SHISA5","TLR7","TLR8","HACD3","CXXC5","PIAS4","PDPK1","PER1","PLCG2",
    "TLR9","LURAP1","TREM2","TERF2IP","ZFAND6","ZDHHC13","NDFIP2","UGT1A1","CC2D1A","LIME1",
    "PPM1A","PPM1B","RNF31","PPP2CB","SBNO1","TRIM62","PPP5C","PIDD1","TNFRSF19","INAVA",
    "PRKCB","PRKCE","PRKD1","MAP2K5","PRL","TRIM39","TMEM9B","ZC3HAV1","SPHK2","PSMA6",
    "C15orf39","OTUD7B","SLC44A2","PELI2","PELI1","MIR497","PLEKHG5","MAVS","IFT80","MIER1",
    "CACTIN","REL","RELA","RELB","BCL3","EDA2R","RORA","S100A4","S100A12","S100A13",
    "S100B","CCL19","CCL21","CX3CL1","SECTM1","PBLD","SLC39A8","NOD2","CARD9","AZI2",
    "BLVRA","CLEC7A","LRRC19","PINK1","GPR89A","SLC20A1","DDRGK1","SPI1","STAT1","STAT3",
    "BST2","MAP3K7","BTK","TFRC","TGFB1","TGFBR3","TLE1","TLR2","TLR3","TLR4",
    "TSPAN6","TMSB4X","TNF","TNFAIP3","TNFRSF1A","NR2C2","TRAF1","TRAF2","TRAF3","TRAF5",
    "TRAF6","UBE2I","UBE2N","UBE2V1","VEGFA","WNT5A","TRIM25","CARD14","TNIP2","MUL1",
    "NLRX1","SNIP1","TNIP3","WLS","ZC3H12A","ALPK1","NDFIP1","APOL3","TRIM8","SHARPIN",
    "CAPN3","CASP1","ITCH","CASP8","CARD19","WDR83","CASP10","TMEM101","CARD11","CUL1",
    "TRIM52","AJUBA","IKBKG","TRIM5","CAV1","TNFSF11","TRADD","RIPK1","TNFSF14","TNFSF10",
    "RIPK2","FADD","RIOK3","TNFRSF11A","TNFRSF10B","CFLAR","SQSTM1","TAX1BP1","CPNE1","BCL10",
    "BTRC","DNAJA3","USP10","NLRP12","DDX21","MTDH","VAPA","TIFA","KLF4","ADIPOQ",
    "CLEC6A","CD36","LITAF","NR1D1","CD40","CD40LG","TRAF4","IKBKE","CD74","RASSF2",
    "TNFSF15","NR1H4","RBX1"
]

sc.tl.score_genes(sdata["gex"], 
                      NFKB_sig,  
                      score_name='Cannonical NFkB signal transduction (GOBP)')

IFN_ab = [
    "TANK", "PSMD14", "ADAR", "IFITM3", "IFITM2", "ADRM1", "CDC37", "USP18",
    "TREX1", "TRIM6", "CR2", "DCST1", "TRIM65", "TTLL12", "YTHDF3", "SAMHD1",
    "SIN3A", "GIGYF2", "LSM14A", "SETD2", "TBK1", "CNOT7", "UBE2K", "IFNE",
    "STING1", "IFI27", "IFIT1", "IFNA1", "IFNA2", "IFNA4", "IFNA5", "IFNA6",
    "IFNA7", "IFNA8", "IFNA10", "IFNA14", "IFNA16", "IFNA17", "IFNA21",
    "IFNAR1", "IFNAR2", "IFNB1", "IFNW1", "IRAK1", "IRF3", "IRF7", "JAK1",
    "USP27X", "SMIM30", "MIR21", "MMP12", "MX1", "MYD88", "OAS1", "OAS2",
    "OAS3", "YTHDF2", "RBM47", "SHFL", "METTL3", "PSMA1", "PSMA2", "IFNK",
    "PSMA3", "PSMA4", "PSMA5", "PSMA6", "PSMA7", "PSMB1", "PSMB2", "PSMB3",
    "PSMB4", "GPR108", "PSMB5", "PSMB6", "PSMB7", "PSMC1", "PSMC2", "PSMC3",
    "PSMC4", "PSMC5", "PSMC6", "PSMD1", "PSMD2", "PSMD3", "PSMD4", "PSMD7",
    "PSMD8", "PSMD11", "PSMD12", "PSMD13", "PSME1", "PSME2", "MAVS", "USP29",
    "PTPN1", "PTPN2", "PTPN6", "PTPN11", "CACTIN", "IFIH1", "AZI2", "SHMT2",
    "SMPD1", "SP100", "STAT1", "STAT2", "TRAF3", "TYK2", "WNT5A", "MUL1",
    "SEM1", "ZBP1", "TRIM56", "NLRC5", "IFITM1", "OASL", "FADD", "CH25H",
    "TRIM41", "RNF185", "EIF4E2", "ISG15", "IKBKE", "TBKBP1", "HDAC4", "PSMD6"
]

sc.tl.score_genes(sdata["gex"], 
                      IFN_ab,  
                      score_name='Response to type I IFN (GOBP)')



IFN_gamma = [
    "NR1H3", "ACTR3", "ACTR2", "CALCOCO2", "IFITM3", "CDC42EP2", "SYNCRIP",
    "UBD", "IFITM2", "ADAMTS13", "CDC37", "VPS26B", "GPR146", "GBP4", "GBP5",
    "OTOP1", "SIRPA", "IL23R", "CYP27B1", "DAPK1", "DAPK3", "GBP6",
    "SLC30A8", "EDN1", "AIF1", "PDE12", "EPRS1", "FCAR", "FLNB", "RPL13A",
    "CDC42EP4", "STXBP4", "GAPDH", "GBP1", "GBP2", "GBP3", "GCH1", "GSN",
    "SEC61A1", "HCK", "HLA-DPA1", "HPX", "RAB7B", "RAB43", "IRF8", "IRGM",
    "IFNG", "IFNGR1", "IFNGR2", "FASLG", "IL12B", "IL12RB1", "AQP4", "IRF1",
    "JAK1", "JAK2", "KIF5B", "ARG1", "GBP7", "LGALS9", "LCN10", "MEFV",
    "CIITA", "CD200", "MRC1", "ABCC1", "CITED1", "ASS1", "MYO1C", "ATF3",
    "NOS2", "NUB1", "PIM1", "PARP14", "PPARG", "MED1", "SHFL", "RAB20",
    "PSMA1", "PSMA2", "PSMA3", "PSMA4", "PSMA5", "PSMA6", "PSMA7",
    "PSMB1", "PSMB2", "PSMB3", "PSMB4", "PSMB8", "PSMB9", "PSMB10",
    "PTPN2", "CXCL16", "RAF1", "RPS6KB1", "CCL2", "CCL3", "CCL5",
    "SLC26A6", "SLC11A1", "SLC22A5", "SNCA", "SP100", "TRIM21",
    "STAT1", "STXBP1", "STXBP2", "STXBP3", "BST2", "CRIPTO", "TLR2",
    "TLR4", "ACTG1", "TNF", "TP53", "TXK", "TYK2", "ACOD1", "NR1H2",
    "VIM", "WAS", "WNT5A", "ZYX", "CALM1", "CAMK2A", "CASP1", "PARP9",
    "NLRC5", "IFITM1", "VAMP4", "KYNU", "CLDN1", "ZNF697", "DNAJA3",
    "VAMP3", "STX8", "CD40", "CD47", "CD58", "CD74", "CDC42"
]

sc.tl.score_genes(sdata["gex"], 
                      IFN_gamma,  
                      score_name='Response to type II IFN (GOBP)')

## Nonoverlapping
# Remove duplicates before computing set differences
IFN_ab_unique = list(set(IFN_ab))
IFN_Gamma_unique = list(set(IFN_gamma))

# Genes unique to each list
IFN_ab_only = sorted(list(set(IFN_ab_unique) - set(IFN_Gamma_unique)))
IFN_Gamma_only = sorted(list(set(IFN_Gamma_unique) - set(IFN_ab_unique)))

sc.tl.score_genes(sdata["gex"], 
                      IFN_ab_only,  
                      score_name='Exclusive to type I interferon signaling')
sc.tl.score_genes(sdata["gex"], 
                      IFN_Gamma_only,  
                      score_name='Exclusive to type II interferon signaling')


### Plot signatures

In [ ]:
IFN_list = ["Response to type I IFN (GOBP)", "Response to type II IFN (GOBP)", "Exclusive to type I interferon signaling", "Exclusive to type II interferon signaling"]

sc.settings.set_figure_params(figsize=(4.3, 4))
sc.pl.umap(sdata["gex"], color=IFN_list, size = 10, legend_fontweight= 'bold', frameon= False ,legend_fontoutline=2, show=True, legend_loc="on data", cmap="RdBu_r", return_fig=True)

In [ ]:
pathway_list = sdata["gex"].obs.columns[(sdata["gex"].obs.columns.str.contains("KEGG")) | sdata["gex"].obs.columns.str.contains("GOBP") | sdata["gex"].obs.columns.str.contains("Exclusive")].tolist()

sc.pl.matrixplot(
    sdata["gex"],
    var_names=pathway_list,   # works for obs keys too
    groupby="leiden_1.7",
    use_raw=False,
    return_fig=False,
    show=True,
    swap_axes=True,
    cmap="RdBu_r",
    standard_scale="var",   # z-score per score (row-wise)
)

plt.savefig("../../05_plots/15_CD4annotation/9_CD4_leiden_pathways.pdf", bbox_inches="tight",  dpi=400)

In [ ]:
sc.settings.set_figure_params(figsize=(4.3, 4))
sc.pl.umap(sdata["gex"], color=pathway_list, size = 10, legend_fontweight= 'bold', frameon= False ,legend_fontoutline=2, show=True, legend_loc="on data", cmap="RdBu_r", return_fig=True)
plt.savefig("../../05_plots/15_CD4annotation/10_CD4_umap_pathways.pdf", bbox_inches="tight",  dpi=400)

# Differential gene expression

## Step 3: Global differential gene expression analysis

In [ ]:
sc.tl.rank_genes_groups(sdata["gex"], 'leiden_1.7', method='wilcoxon', key_added="leiden_1.7_DEG")
Export_DEG = sc.get.rank_genes_groups_df(sdata["gex"], group = None, key = "leiden_1.7_DEG")
# Export_DEG = Export_DEG.groupby(['group']).head(50)
Export_DEG.to_csv("../../06_csv_outputs/DEG/1_CD4_DEGs_globally.csv")


In [ ]:
sc.pl.rank_genes_groups(sdata["gex"], key="leiden_1.7_DEG")

### Format the table for differentially expressed genes. 

In [ ]:
Export_DEG = sc.get.rank_genes_groups_df(sdata["gex"], group = None, key = "leiden_1.7_DEG")

DEGs_export = (
    Export_DEG[
        (Export_DEG["logfoldchanges"] > 1) &
        (Export_DEG["pvals_adj"] < 0.01)
    ]
    .sort_values("scores", ascending=False)
)

gene_lists = (
    DEGs_export
        # .groupby("group")
        # .head(20)
        .groupby("group")["names"]
        .apply(list)
)

gene_lists_df = pd.DataFrame({
    group: pd.Series(genes)
    for group, genes in gene_lists.items()
})

gene_lists_df.to_excel("../../06_csv_outputs/DEG/2_CD4_Leiden2_LFC_above_1_pval_below_0.01.xlsx")

In [ ]:
fig_deg = sc.pl.rank_genes_groups_dotplot(
    sdata["gex"], groupby="leiden_1.7", standard_scale="var", n_genes=10, key="leiden_1.7_DEG", cmap="RdBu_r", show=False, return_fig=True, dendrogram=True)

fig_deg.savefig("../../05_plots/15_CD4annotation/11_CD4_leiden_DEG_bubblechart.pdf", bbox_inches="tight",  dpi=400)
plt.show()
# fig_umap = sc.pl.umap(sdata["gex"], color=["leiden_1.7"], size = 5, legend_fontweight= 'bold', frameon= False ,legend_fontoutline=2, show=False, legend_loc="on data", cmap="RdBu_r", return_fig=True)


## Filter differentially expressed genes

In [ ]:
# # Run rank_genes_groups with filtered genes
# sc.tl.filter_rank_genes_groups(
#     sdata["gex"],
#     min_in_group_fraction=0.1,  #fraction positive in population of interest (i.e if I am looking at genes related to CSF-migration, then it should be expressed in X fraction of the target group
#     max_out_group_fraction=1.0,   #Max is 1 - no filtering will take place
#     min_fold_change=0,
#     compare_abs=False,       #Compare_abs=True retains negative log2fc genes.. We specifically want upregulated genes for the next analysis
#     key="leiden_1.7_DEG",
#     key_added="leiden_1.7_DEG (w. filtering)",
# )

## Takes a llong time to run

In [ ]:
# fig_deg = sc.pl.rank_genes_groups_dotplot(
#     sdata["gex"], groupby="leiden_1.7", standard_scale="var", n_genes=10, key="leiden_1.7_DEG (w. filtering)", cmap="RdBu_r", show=False, return_fig=True, dendrogram=True)

# fig_deg.savefig("../../05_plots/15_CD4annotation/11_CD4_leiden1.6_DEG_bubblechart_fiiltered.pdf", bbox_inches="tight",  dpi=400)
# plt.show()


# Differentiation gene expression without ribosomal genes

In [ ]:
# Remove TCR-related and ribosomal genes
# TCR genes often start with 'TRAV', 'TRBV', 'TRGV', etc.
# Ribosomal genes often start with 'RPS' or 'RPL'

sdata_filtered = sdata.copy()

# IMPORTANT TO SUBSET ONLY ON CELLS OF INTEREST- because the filter_rank_genes behaves odd if there are cells outside the groups (target and reference) that express a gene of interest. CCR7 kept getting filtered out because naive cells/other cells were part of the analysis
# mu.pp.filter_obs(sdata_pb_filtered["gex"], "top_vs_bot_clones", lambda x: (x != "unselected"))   #Going back to using memory cells as background for filtering

# Create a mask for genes to exclude
genes = sdata["gex"].var_names
exclude_mask = genes.str.startswith(("TRAV", "TRBV", "TRGV", "TRDV", "TRAJ", "TRBJ", "TRGJ", "TRDJ", "TRAC", "TRBC", "TRGC", "TRDC", "RPS", "RPL", "MT-"))

# Keep only genes not matching the exclusion
genes_to_keep = genes[~exclude_mask]
sdata_filtered = sdata_filtered["gex"][:, genes_to_keep]

#
sc.tl.rank_genes_groups(sdata_filtered, 'leiden_1.7', method='wilcoxon', key_added="leiden_1.7_DEG")
Export_DEG = sc.get.rank_genes_groups_df(sdata_filtered, group = None, key = "leiden_1.7_DEG")
# Export_DEG = Export_DEG.groupby(['group']).head(50)
# Export_DEG.to_csv("../../06_csv_outputs/DEG/1_CD4_DEGs_globally.csv")

Export_DEG.head(10)


In [ ]:
# # Run rank_genes_groups with filtered genes
# sc.tl.filter_rank_genes_groups(
#     sdata_filtered,
#     min_in_group_fraction=0.1,  #fraction positive in population of interest (i.e if I am looking at genes related to CSF-migration, then it should be expressed in X fraction of the target group
#     max_out_group_fraction=1.0,   #Max is 1 - no filtering will take place
#     min_fold_change=0,
#     compare_abs=False,       #Compare_abs=True retains negative log2fc genes.. We specifically want upregulated genes for the next analysis
#     key="leiden_1.7_DEG",
#     key_added="leiden_1.7_DEG (w. filtering)",
# )

# # Takes a llong time to run

In [ ]:
##### fig_deg = sc.pl.rank_genes_groups_dotplot(
# sdata_filtered, groupby="leiden_1.7", standard_scale="var", n_genes=10, key="leiden_1.7_DEG (w. filtering)", cmap="RdBu_r", show=False, return_fig=True, dendrogram=True)

# fig_deg.savefig("../../05_plots/15_CD4annotation/11_CD4_leiden1.6_DEG_bubblechart_fiiltered.pdf", bbox_inches="tight",  dpi=400)
# plt.show()


## Display hallmark genes

In [ ]:
Export_DEG = sc.get.rank_genes_groups_df(sdata_filtered, group = None, key = "leiden_1.7_DEG")



In [ ]:
### Cluster 0
Cluster_genes = Export_DEG[
    (Export_DEG["group"]== "0") & 
    (Export_DEG["logfoldchanges"] > 1) &
    (Export_DEG["pvals_adj"] < 0.01)].head(50)["names"].tolist()

sc.pl.dotplot(
    sdata["gex"],
    Cluster_genes,
    groupby="leiden_1.7",
    standard_scale="var",
    dendrogram=False,
    cmap="RdBu_r",
    swap_axes=False,
)


In [ ]:
### Cluster 1
Cluster_genes = Export_DEG[
    (Export_DEG["group"]== "1") & 
    (Export_DEG["logfoldchanges"] > 1) &
    (Export_DEG["pvals_adj"] < 0.01)].head(50)["names"].tolist()

sc.pl.dotplot(
    sdata["gex"],
    Cluster_genes,
    groupby="leiden_1.7",
    standard_scale="var",
    dendrogram=False,
    cmap="RdBu_r",
    swap_axes=False,
)



In [ ]:
### Cluster 2
Cluster_genes = Export_DEG[
    (Export_DEG["group"]== "2") & 
    (Export_DEG["logfoldchanges"] > 1) &
    (Export_DEG["pvals_adj"] < 0.01)].head(50)["names"].tolist()

sc.pl.dotplot(
    sdata["gex"],
    Cluster_genes,
    groupby="leiden_1.7",
    standard_scale="var",
    dendrogram=False,
    cmap="RdBu_r",
    swap_axes=False,
)


In [ ]:
### Cluster 3
Cluster_genes = Export_DEG[
    (Export_DEG["group"]== "3") & 
    (Export_DEG["logfoldchanges"] > 1) &
    (Export_DEG["pvals_adj"] < 0.01)].head(50)["names"].tolist()

sc.pl.dotplot(
    sdata["gex"],
    Cluster_genes,
    groupby="leiden_1.7",
    standard_scale="var",
    dendrogram=False,
    cmap="RdBu_r",
    swap_axes=False,
)


In [ ]:
### Cluster 4
Cluster_genes = Export_DEG[
    (Export_DEG["group"]== "4") & 
    (Export_DEG["logfoldchanges"] > 1) &
    (Export_DEG["pvals_adj"] < 0.01)].head(50)["names"].tolist()

sc.pl.dotplot(
    sdata["gex"],
    Cluster_genes,
    groupby="leiden_1.7",
    standard_scale="var",
    dendrogram=False,
    cmap="RdBu_r",
    swap_axes=False,
)


In [ ]:
### Cluster 5
Cluster_genes = Export_DEG[
    (Export_DEG["group"]== "5") & 
    (Export_DEG["logfoldchanges"] > 1) &
    (Export_DEG["pvals_adj"] < 0.01)].head(50)["names"].tolist()

sc.pl.dotplot(
    sdata["gex"],
    Cluster_genes,
    groupby="leiden_1.7",
    standard_scale="var",
    dendrogram=False,
    cmap="RdBu_r",
    swap_axes=False,
)



In [ ]:
### Cluster 6
Cluster_genes = Export_DEG[
    (Export_DEG["group"]== "6") & 
    (Export_DEG["logfoldchanges"] > 1) &
    (Export_DEG["pvals_adj"] < 0.01)].head(50)["names"].tolist()

sc.pl.dotplot(
    sdata["gex"],
    Cluster_genes,
    groupby="leiden_1.7",
    standard_scale="var",
    dendrogram=False,
    cmap="RdBu_r",
    swap_axes=False,
)


In [ ]:
### Cluster 7
Cluster_genes = Export_DEG[
    (Export_DEG["group"]== "7") & 
    (Export_DEG["logfoldchanges"] > 1) &
    (Export_DEG["pvals_adj"] < 0.01)].head(50)["names"].tolist()

sc.pl.dotplot(
    sdata["gex"],
    Cluster_genes,
    groupby="leiden_1.7",
    standard_scale="var",
    dendrogram=False,
    cmap="RdBu_r",
    swap_axes=False,
)



In [ ]:
### Cluster 8
Cluster_genes = Export_DEG[
    (Export_DEG["group"]== "8") & 
    (Export_DEG["logfoldchanges"] > 1) &
    (Export_DEG["pvals_adj"] < 0.01)].head(50)["names"].tolist()

sc.pl.dotplot(
    sdata["gex"],
    Cluster_genes,
    groupby="leiden_1.7",
    standard_scale="var",
    dendrogram=False,
    cmap="RdBu_r",
    swap_axes=False,
)



In [ ]:
### Cluster 9
Cluster_genes = Export_DEG[
    (Export_DEG["group"]== "9") & 
    (Export_DEG["logfoldchanges"] > 1) &
    (Export_DEG["pvals_adj"] < 0.01)].head(50)["names"].tolist()

sc.pl.dotplot(
    sdata["gex"],
    Cluster_genes,
    groupby="leiden_1.7",
    standard_scale="var",
    dendrogram=False,
    cmap="RdBu_r",
    swap_axes=False,
)



In [ ]:
### Cluster 10
Cluster_genes = Export_DEG[
    (Export_DEG["group"]== "10") & 
    (Export_DEG["logfoldchanges"] > 1) &
    (Export_DEG["pvals_adj"] < 0.01)].head(50)["names"].tolist()

sc.pl.dotplot(
    sdata["gex"],
    Cluster_genes,
    groupby="leiden_1.7",
    standard_scale="var",
    dendrogram=False,
    cmap="RdBu_r",
    swap_axes=False,
)


In [ ]:
### Cluster 11
Cluster_genes = Export_DEG[
    (Export_DEG["group"]== "11") & 
    (Export_DEG["logfoldchanges"] > 1) &
    (Export_DEG["pvals_adj"] < 0.01)].head(50)["names"].tolist()

sc.pl.dotplot(
    sdata["gex"],
    Cluster_genes,
    groupby="leiden_1.7",
    standard_scale="var",
    dendrogram=False,
    cmap="RdBu_r",
    swap_axes=False,
)


In [ ]:
### Cluster 12
Cluster_genes = Export_DEG[
    (Export_DEG["group"]== "12") & 
    (Export_DEG["logfoldchanges"] > 1) &
    (Export_DEG["pvals_adj"] < 0.01)].head(50)["names"].tolist()

sc.pl.dotplot(
    sdata["gex"],
    Cluster_genes,
    groupby="leiden_1.7",
    standard_scale="var",
    dendrogram=False,
    cmap="RdBu_r",
    swap_axes=False,
)


In [ ]:
### Cluster 13
Cluster_genes = Export_DEG[
    (Export_DEG["group"]== "13") & 
    (Export_DEG["logfoldchanges"] > 1) &
    (Export_DEG["pvals_adj"] < 0.01)].head(50)["names"].tolist()

sc.pl.dotplot(
    sdata["gex"],
    Cluster_genes,
    groupby="leiden_1.7",
    standard_scale="var",
    dendrogram=False,
    cmap="RdBu_r",
    swap_axes=False,
)



In [ ]:
### Cluster 14
Cluster_genes = Export_DEG[
    (Export_DEG["group"]== "14") & 
    (Export_DEG["logfoldchanges"] > 1) &
    (Export_DEG["pvals_adj"] < 0.01)].head(50)["names"].tolist()

sc.pl.dotplot(
    sdata["gex"],
    Cluster_genes,
    groupby="leiden_1.7",
    standard_scale="var",
    dendrogram=False,
    cmap="RdBu_r",
    swap_axes=False,
)



In [ ]:
### Cluster 15
Cluster_genes = Export_DEG[
    (Export_DEG["group"]== "15") & 
    (Export_DEG["logfoldchanges"] > 1) &
    (Export_DEG["pvals_adj"] < 0.01)].head(50)["names"].tolist()

sc.pl.dotplot(
    sdata["gex"],
    Cluster_genes,
    groupby="leiden_1.7",
    standard_scale="var",
    dendrogram=False,
    cmap="RdBu_r",
    swap_axes=False,
)



In [ ]:
sc.settings.set_figure_params(figsize=(6, 4))
sc.pl.violin(sdata["gex"], groupby="donor", keys=["AC004448.2"])

In [ ]:
### Cluster 16
Cluster_genes = Export_DEG[
    (Export_DEG["group"]== "16") & 
    (Export_DEG["logfoldchanges"] > 1) &
    (Export_DEG["pvals_adj"] < 0.01)].head(50)["names"].tolist()

sc.pl.dotplot(
    sdata["gex"],
    Cluster_genes,
    groupby="leiden_1.7",
    standard_scale="var",
    dendrogram=False,
    cmap="RdBu_r",
    swap_axes=False,
)


In [ ]:
### Cluster 17
Cluster_genes = Export_DEG[
    (Export_DEG["group"]== "17") & 
    (Export_DEG["logfoldchanges"] > 1) &
    (Export_DEG["pvals_adj"] < 0.01)].head(50)["names"].tolist()

sc.pl.dotplot(
    sdata["gex"],
    Cluster_genes,
    groupby="leiden_1.7",
    standard_scale="var",
    dendrogram=False,
    cmap="RdBu_r",
    swap_axes=False,
)


In [ ]:
### Cluster 18
Cluster_genes = Export_DEG[
    (Export_DEG["group"]== "18") & 
    (Export_DEG["logfoldchanges"] > 1) &
    (Export_DEG["pvals_adj"] < 0.01)].head(50)["names"].tolist()

sc.pl.dotplot(
    sdata["gex"],
    Cluster_genes,
    groupby="leiden_1.7",
    standard_scale="var",
    dendrogram=False,
    cmap="RdBu_r",
    swap_axes=False,
)



In [ ]:
### Cluster 19
Cluster_genes = Export_DEG[
    (Export_DEG["group"]== "19") & 
    (Export_DEG["logfoldchanges"] > 1) &
    (Export_DEG["pvals_adj"] < 0.01)].head(50)["names"].tolist()

sc.pl.dotplot(
    sdata["gex"],
    Cluster_genes,
    groupby="leiden_1.7",
    standard_scale="var",
    dendrogram=False,
    cmap="RdBu_r",
    swap_axes=False,
)


In [ ]:
### Cluster 20
Cluster_genes = Export_DEG[
    (Export_DEG["group"]== "20") & 
    (Export_DEG["logfoldchanges"] > 1) &
    (Export_DEG["pvals_adj"] < 0.01)].head(50)["names"].tolist()

sc.pl.dotplot(
    sdata["gex"],
    Cluster_genes,
    groupby="leiden_1.7",
    standard_scale="var",
    dendrogram=False,
    cmap="RdBu_r",
    swap_axes=False,
)


# Specific differential gene expression

In [ ]:
# Cluster X versus y
sc.tl.rank_genes_groups(sdata["gex"], groupby="leiden_1.7", groups=["1"], reference="3", key_added="upX")
sc.tl.rank_genes_groups(sdata["gex"], groupby="leiden_1.7",  groups=["3"], reference="1", key_added="upY")
sc.settings.set_figure_params(figsize=(3, 2.5))
sc.pl.rank_genes_groups(sdata["gex"], key="upX", n_genes=20)
sc.pl.rank_genes_groups(sdata["gex"], key="upY", n_genes=20)

In [ ]:
# Cluster X versus y
sc.tl.rank_genes_groups(sdata_filtered, groupby="leiden_1.7", groups=["15"], reference="8", key_added="upX")
sc.tl.rank_genes_groups(sdata_filtered, groupby="leiden_1.7",  groups=["8"], reference="15", key_added="upY")
sc.settings.set_figure_params(figsize=(3, 2.5))
sc.pl.rank_genes_groups(sdata_filtered, key="upX", n_genes=20)
sc.pl.rank_genes_groups(sdata_filtered, key="upY", n_genes=20)

In [ ]:
# Cluster X versus y
sc.tl.rank_genes_groups(sdata_filtered, groupby="leiden_1.7", groups=["10"], reference="9", key_added="upX")
sc.tl.rank_genes_groups(sdata_filtered, groupby="leiden_1.7",  groups=["9"], reference="10", key_added="upY")
sc.settings.set_figure_params(figsize=(3, 2.5))
sc.pl.rank_genes_groups(sdata_filtered, key="upX", n_genes=20)
sc.pl.rank_genes_groups(sdata_filtered, key="upY", n_genes=20)



In [ ]:
#CSF specific genes
sdata["gex"].obs["deg_grp"] = pd.Categorical(np.where(sdata["gex"].obs["Compartment"].isin(["CSF"]), "CSF", "PB"))
sdata_temp = sdata.copy()
# mu.pp.filter_obs(sdata_temp["gex"], "leiden_1.9", lambda x: (~x.isin(["0", "1", "2", "6", "15"])))
sc.tl.rank_genes_groups(sdata_temp["gex"], groupby="deg_grp", groups=["CSF"], reference="PB", key_added="upX")
sc.tl.rank_genes_groups(sdata_temp["gex"], groupby="deg_grp",  groups=["PB"], reference="CSF", key_added="upY")
sc.settings.set_figure_params(figsize=(3, 2.5))
sc.pl.rank_genes_groups(sdata_temp["gex"], key="upX", n_genes=20)

In [ ]:
# One cluster versus others of a similar type
sdata["gex"].obs["deg_grp"] = pd.Categorical(np.where((sdata["gex"].obs["leiden_1.7"].isin(["15"])) & (sdata["gex"].obs["Compartment"] == "CSF"), "C15", 
                                                      np.where((sdata["gex"].obs["leiden_1.7"].isin(["6", "7", "8", "9", "10", "16", "19"])) & (sdata["gex"].obs["Compartment"] == "CSF"), "Other_CSF_clusters", "error")))
sdata_temp = sdata.copy()
mu.pp.filter_obs(sdata_temp["gex"], "Compartment", lambda x: (x=="CSF"))
sc.tl.rank_genes_groups(sdata_temp["gex"], groupby="deg_grp", groups=["C15"], reference="Other_CSF_clusters", key_added="upX")
sc.tl.rank_genes_groups(sdata_temp["gex"], groupby="deg_grp",  groups=["Other_CSF_clusters"], reference="C15", key_added="upY")
sc.settings.set_figure_params(figsize=(5, 2.5))
sc.pl.rank_genes_groups(sdata_temp["gex"], key="upX", n_genes=30)
sc.pl.rank_genes_groups(sdata_temp["gex"], key="upY", n_genes=30)

In [ ]:
# One cluster versus others of a similar type
sdata["gex"].obs["deg_grp"] = pd.Categorical(np.where((sdata["gex"].obs["leiden_1.7"].isin(["15"])) & (sdata["gex"].obs["Compartment"] == "CSF"), "C10", 
                                                      np.where((sdata["gex"].obs["leiden_1.7"].isin(["6", "7", "8", "15", "16", "19"])) & (sdata["gex"].obs["Compartment"] == "CSF"), "Other_CSF_clusters", "error")))
sdata_temp = sdata.copy()
mu.pp.filter_obs(sdata_temp["gex"], "Compartment", lambda x: (x=="CSF"))
sc.tl.rank_genes_groups(sdata_temp["gex"], groupby="deg_grp", groups=["C10"], reference="Other_CSF_clusters", key_added="upX")
sc.tl.rank_genes_groups(sdata_temp["gex"], groupby="deg_grp",  groups=["Other_CSF_clusters"], reference="C10", key_added="upY")
sc.settings.set_figure_params(figsize=(5, 2.5))
sc.pl.rank_genes_groups(sdata_temp["gex"], key="upX", n_genes=30)
sc.pl.rank_genes_groups(sdata_temp["gex"], key="upY", n_genes=30)

# Uniquely expressed genes

In [ ]:
## Maximal
marker_dict = {"CITEseq" : ["prot_CD45RA_adt", "prot_CD62L_adt", "prot_CD95_adt"], 
               "TN" : ["CCR7", "SELL", "LEF1", "TCF7", "BACH2", "NOG"],
               "IFNsig" : ["STAT1", "IFI6", "ISG15"], 
               #Memory / Effector
               "C11" : ["CYLD", "PIM1"], 
               "C2" : ["IL7R", "NR3C1", "SESN3"], 
                "C4" : ["GPR171", "HERC5", "TNFSF13B"],
                "C12" : ["TNF", "TNFRSF4", "BHLHE40", "BATF3"], 
                "C18" : ["GZMB", "PRF1", "EOMES"], 
                "C14" : ["FOXP3", "IL2RA", "CTLA4"], 


               #Tact
               "C13" : ["NFKB2", "RELB", "CXCR4"], 
               "C17" : ["JUNB", "JUND", "FOS"], 
                "C20" : ["MALAT1", "RPS8","RPL30", "RPL19"],
    
               #CSF
               "C7" :  ["ITGA4", "AQP3", "LIMS1", "ID3"],
               "C6" : ["CXCR3", "CTSW", "KLRB1"],
               "C8" : [ "CRIP1", "LGALS1"], 
               "C15" : ["H1FX","JUNB"],
               "C16" : ["GZMK", "GZMA", "EOMES"], 
               "C19" :["CD74", "CD27", "IKZF2", "TOX"], 
                "C9" :  ["JUNB","CD69", "ZFP36", "EGR1"], 

                #Shared
                "Shared CSF"  : ["TXNIP", "S100A4", "CD52"]
              }

Cluster_order = [
    "0",  "5", "1", "3",       #naive + ACt naive
      "11",  "2",  "4", "12",  "18",  "14", 
      
      "13",  "17", "20", 
    "7", "6", "8",  "15", "16",  "19", "9", "10"]      #Tcsf]   

# # #Implement order
sdata["gex"].obs["leiden_1.7"] = sdata["gex"].obs["leiden_1.7"].cat.reorder_categories(
    Cluster_order, ordered=True
)

dp = sc.pl.dotplot(
    sdata["gex"],
    marker_dict,
    groupby="leiden_1.7",
    standard_scale="var",
    dendrogram=False,
    cmap="RdBu_r",
    swap_axes=False,
    show=False,
    return_fig=True,
)

dp.savefig(
    "../../05_plots/15_CD4annotation/2_CD4_bubblechart_maximal.pdf",
    bbox_inches="tight",
    dpi=400
)


In [ ]:
## Heatmap to cover lowly expressed transcripts too
markers = [ "IL7R", 'EOMES', 'TBX21', 'GATA3', 'RORC', 'FOXP3', 'BCL6',"CCR6", "GZMB", "GZMK", "GZMH", "GZMA", "CCL5", "IFNG", "IL10", "CHI3L2", "CCL4", "CCR5", "CST7", "PDCD1", "TOX", "LAG3"]
sc.pl.heatmap(sdata["gex"], markers, groupby='leiden_1.7', swap_axes=True, figsize=(20,5), cmap="RdBu_r")

## DICE gene sets

In [ ]:
ann_dat = pd.read_excel("../../03_annotation/01_cluster_annotation/Schmiedel_etal_DICE_CellTypeSpecificGenes_edit.xlsx")

#Filter
ann_dat = ann_dat[(ann_dat["Log2FC"] > 1) & (ann_dat["P_adj"] < 0.01)]
ann_dat["Cell_type"].drop_duplicates()

In [ ]:
gene_list = ann_dat[ann_dat["Cell_type"] == "TH1 cells"]["Gene ID"].tolist()
# Cacluate a score
sc.tl.score_genes(sdata["gex"], 
                      gene_list,  
                      score_name='TH1 cells (Schmiedel et al. 2018)')

In [ ]:
gene_list = ann_dat[ann_dat["Cell_type"] == "TH1/17 cells"]["Gene ID"].tolist()
# Cacluate a score
sc.tl.score_genes(sdata["gex"], 
                      gene_list,  
                      score_name='TH1_17 cells (Schmiedel et al. 2018)')

In [ ]:
gene_list = ann_dat[ann_dat["Cell_type"] == "Memory TREG cells"]["Gene ID"].tolist()
# Cacluate a score
sc.tl.score_genes(sdata["gex"], 
                      gene_list,  
                      score_name='Memory TREG cells (Schmiedel et al. 2018)')

In [ ]:
gene_list = ann_dat[ann_dat["Cell_type"] == "Naïve CD4+ T cells"]["Gene ID"].tolist()
# Cacluate a score
sc.tl.score_genes(sdata["gex"], 
                      gene_list,  
                      score_name='Naïve CD4+ T cells (Schmiedel et al. 2018)')

In [ ]:
gene_list = ann_dat[ann_dat["Cell_type"] == "Naïve TREG cells"]["Gene ID"].tolist()
# Cacluate a score
sc.tl.score_genes(sdata["gex"], 
                      gene_list,  
                      score_name='Naïve TREG cells (Schmiedel et al. 2018)')

In [ ]:
gene_list = ann_dat[ann_dat["Cell_type"] == "TH2 cells"]["Gene ID"].tolist()
# Cacluate a score
sc.tl.score_genes(sdata["gex"], 
                      gene_list,  
                      score_name='TH2 cells (Schmiedel et al. 2018)')

In [ ]:
gene_list = ann_dat[ann_dat["Cell_type"] == "TH17 cells"]["Gene ID"].tolist()
# Cacluate a score
sc.tl.score_genes(sdata["gex"], 
                      gene_list,  
                      score_name='TH17 cells (Schmiedel et al. 2018)')

In [ ]:
gene_list = ann_dat[ann_dat["Cell_type"] == "TFH cells"]["Gene ID"].tolist()
# Cacluate a score
sc.tl.score_genes(sdata["gex"], 
                      gene_list,  
                      score_name='TFH cells (Schmiedel et al. 2018)')

In [ ]:
list_of_sigs = [ "EOMES+ Treg (Bonnel et al 2021)", "Tr1+ cells (De Simone et al 2021)", "TH1 cells (Schmiedel et al. 2018)", "TH1_17 cells (Schmiedel et al. 2018)", "Memory TREG cells (Schmiedel et al. 2018)", 
                "Naïve CD4+ T cells (Schmiedel et al. 2018)", "Naïve TREG cells (Schmiedel et al. 2018)", "TH2 cells (Schmiedel et al. 2018)", 
                "TH17 cells (Schmiedel et al. 2018)", "TFH cells (Schmiedel et al. 2018)",]

In [ ]:
sc.settings.set_figure_params(figsize=(4.3, 4))
sc.pl.umap(sdata["gex"], color=list_of_sigs, size = 20, legend_fontweight= 'bold', frameon= False ,legend_fontoutline=2, show=True, legend_loc="on data", cmap="RdBu_r")

In [ ]:
sc.pl.matrixplot(sdata["gex"], list_of_sigs, groupby='leiden_1.7', swap_axes=True, figsize=(10,5), standard_scale="var", cmap="RdBu_r")

## Ostkampf signatures
These were derived from Monaco et al Cell Reports

In [ ]:
ann_dat = pd.read_excel("../../03_annotation/02_gene_sets/Ostkampf_all_gene_sets.xlsx")

#Filter
ann_dat.head(2)

In [ ]:
gene_list = ann_dat["Tfh"].drop_duplicates()
# Cacluate a score
sc.tl.score_genes(sdata["gex"], 
                      gene_list,  
                      score_name='Tfh (Ostkampf et al. 2022)')

In [ ]:
gene_list = ann_dat["Th17"].drop_duplicates()
# Cacluate a score
sc.tl.score_genes(sdata["gex"], 
                      gene_list,  
                      score_name='Th17 (Ostkampf et al. 2022)')

In [ ]:
gene_list = ann_dat["Th1"].drop_duplicates()
# Cacluate a score
sc.tl.score_genes(sdata["gex"], 
                      gene_list,  
                      score_name='Th1 (Ostkampf et al. 2022)')

In [ ]:
gene_list = ann_dat["Tregs"].drop_duplicates()
# Cacluate a score
sc.tl.score_genes(sdata["gex"], 
                      gene_list,  
                      score_name='Tregs (Ostkampf et al. 2022)')

In [ ]:
gene_list = ann_dat["Th2"].drop_duplicates()
# Cacluate a score
sc.tl.score_genes(sdata["gex"], 
                      gene_list,  
                      score_name='Th2 (Ostkampf et al. 2022)')

In [ ]:
gene_list = ann_dat["CD4 TEMRA"].drop_duplicates()
# Cacluate a score
sc.tl.score_genes(sdata["gex"], 
                      gene_list,  
                      score_name='CD4 TEMRA (Ostkampf et al. 2022)')

In [ ]:
gene_list = ann_dat["Th22"].drop_duplicates()
# Cacluate a score
sc.tl.score_genes(sdata["gex"], 
                      gene_list,  
                      score_name='Th22 (Ostkampf et al. 2022)')

In [ ]:
gene_list = ann_dat["TRM"].drop_duplicates()
# Cacluate a score
sc.tl.score_genes(sdata["gex"], 
                      gene_list,  
                      score_name='TRM (Ostkampf et al. 2022)')

In [ ]:
gene_list = ann_dat["TRM CD103+"].drop_duplicates()
# Cacluate a score
sc.tl.score_genes(sdata["gex"], 
                      gene_list,  
                      score_name='TRM CD103+ (Ostkampf et al. 2022)')

In [ ]:
list_of_sigs = ["Tfh (Ostkampf et al. 2022)", "Th17 (Ostkampf et al. 2022)", "Th1 (Ostkampf et al. 2022)", "Tregs (Ostkampf et al. 2022)", "Th2 (Ostkampf et al. 2022)", "CD4 TEMRA (Ostkampf et al. 2022)", "Th22 (Ostkampf et al. 2022)", "TRM (Ostkampf et al. 2022)","TRM CD103+ (Ostkampf et al. 2022)"]

In [ ]:
sc.settings.set_figure_params(figsize=(4.3, 4))
sc.pl.umap(sdata["gex"], color=list_of_sigs, size = 20, legend_fontweight= 'bold', frameon= False ,legend_fontoutline=2, show=True, legend_loc="on data", cmap="RdBu_r")

In [ ]:
sc.pl.matrixplot(sdata["gex"], list_of_sigs,groupby='leiden_1.7', swap_axes=True, figsize=(10,5), standard_scale="var", cmap="RdBu_r")

# Annotation

In [ ]:
#Kilian's suggestions
Cluster_order = [
    "0",  "5", "1", "3",       #naive + ACt naive
      "4", "12", "14",  "18",  "11",  "2",    
      
      "13",  "17", "20", 
    "7", "6", "8",  "15", "16",  "19", "9", "10"]      #Tcsf]   


leiden_annot = {
    "0": "0 TN (BACH2hi IFNlo)", 
    "5": "5 TN (BACH2hi IFNint)", 
    "1": "1 TN (BACH2lo IFNlo)", 
    "3": "3 TN (BACH2lo IFNhi)", 
    "11": "11 TCM (CYLDhi PIMhi)", 
    "2" : "2 TCM/EM (IL7Rhi NR3C1hi)", 
    "4": "4 TCM/TEM (TNFSF13Bhi IFNhi)", 
    "12": "12 TEM (TNFhi TNFRSF4hi)",
    "18" : "18 TC (GZMBhi PRF1hi)", 
    "14" : "14 Treg (FOXP3hi)", 
    "13" : "13 TAct (NFKB2hi CXCR4hi)", 
    "17" : "17 TAct (JUNBhi FOShi)", 
    "20" : "20 TAct (Ribolo)", 
    "7" : "7 TCSF (TCF7hi ITGA4hi)", 
    "6" : "6 TCSF (CXCR3hi KLRB1hi)", 
    "8" : "8 TCSF (CXCR3hi LGALS1hi)", 
    "15" : "15 TCSF (H1FXlo JUNBlo)", 
    "16" : "16 TCSF (GZMKhi EOMEShi)", 
    "19" : "19 TCSF (TOXhi FOXP3int)", 
    "9" : "9 TCSF (CD69hi JUNBhi)", 
    "10" : "10 TCSF (TCF7int Ribohi)",
}

# Add cell type column based on annotation
sdata["gex"].obs['L3_ann'] = [leiden_annot[sample] for sample in sdata["gex"].obs['leiden_1.7']]

In [ ]:
## Maximal
marker_dict = {"CITEseq" : ["prot_CD45RA_adt", "prot_CD62L_adt", "prot_CD95_adt"], 
               "TN" : ["CCR7", "SELL", "LEF1", "TCF7", "BACH2", "NOG"],
               "IFNsig" : ["STAT1", "IFI6", "ISG15"], 
               #Memory / Effector
               "C11" : ["CYLD", "PIM1"], 
               "C2" : ["IL7R", "NR3C1", "SESN3"], 
                "C4" : ["GPR171", "HERC5", "TNFSF13B"],
                "C12" : ["TNF", "TNFRSF4", "BHLHE40", "BATF3"], 
                "C18" : ["GZMB", "PRF1", "EOMES"], 
                "C14" : ["FOXP3", "IL2RA", "CTLA4"], 


               #Tact
               "C13" : ["NFKB2", "RELB", "CXCR4"], 
               "C17" : ["JUNB", "JUND", "FOS"], 
                "C20" : ["MALAT1", "RPS8","RPL30", "RPL19"],
    
               #CSF
               "C7" :  ["ITGA4", "AQP3", "LIMS1", "ID3"],
               "C6" : ["CXCR3", "CTSW", "KLRB1"],
               "C8" : [ "CRIP1", "LGALS1"], 
               "C15" : ["H1FX","JUNB"],
               "C16" : ["GZMK", "GZMA", "EOMES"], 
               "C19" :["CD74", "CD27", "IKZF2", "TOX"], 
                "C9" :  ["JUNB","CD69", "ZFP36", "EGR1"],   #Ostkamp makes claims regarding:  "ZFP36L2", "CXCR6", "DUSP1","ID2","ITGAE",

                #Shared
                "Shared CSF"  : ["TXNIP", "S100A4", "CD52"]
              }

# # Define a custom order by following the order of clusters in your annotation box above
ordered_categories = list(leiden_annot.values())

# #Implement order
# Convert to categorical
sdata["gex"].obs["L3_ann"] = pd.Categorical(
    sdata["gex"].obs["L3_ann"],
    categories=ordered_categories,
    ordered=True
)

sdata["gex"].obs["L3_ann"] = sdata["gex"].obs["L3_ann"].cat.reorder_categories(
    ordered_categories, ordered=True
)

dp = sc.pl.dotplot(
    sdata["gex"],
    marker_dict,
    groupby="L3_ann",
    standard_scale="var",
    dendrogram=False,
    cmap="RdBu_r",
    swap_axes=False,
    show=False,
    return_fig=True,
)

dp.savefig(
    "../../05_plots/15_CD4annotation/2_CD4_bubblechart_minimal_annotated.pdf",
    bbox_inches="tight",
    dpi=400
)


# statistics

In [ ]:
# --- Prepare obs_df ---
obs_df = sdata["gex"].obs.copy()
obs_df = obs_df[~obs_df["donor"].isin(["Pt4", "Pt5"])]
obs_df = obs_df[obs_df["clone_size"] > 5]
obs_df = obs_df[obs_df["Compartment"] == "CSF"]
obs_df["donor"] = obs_df["donor"].cat.remove_unused_categories() 


# Leiden cluster
cls = "L3_ann"

# Calculate counts per donor and cluster
obs_df = (
    obs_df[["donor", cls, "catpat"]]
    .groupby(["donor", cls])
    .size()
    .reset_index(name="count")
)

# Total per donor
obs_df["total"] = obs_df.groupby("donor")["count"].transform("sum")

# Percentage per donor
obs_df["cluster_pct"] = 100 * obs_df["count"] / obs_df["total"]

# Annotate catpat
patient_list = ["Pt4", "Pt11", "Pt12", "Pt14", "Pt17", "Pt20"]
control_list = ["Pt1", "Pt5", "Pt8", "Pt18", "Pt19", "Pt21"]
obs_df["catpat"] = pd.Categorical(
    np.where(
        obs_df["donor"].isin(patient_list), "RRMS",
        np.where(obs_df["donor"].isin(control_list), "Control", "Error")
    )
)

# --- Statistics: Mann–Whitney U per cluster ---
def p_to_star(p):
    if p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return "ns"

pvals = {}
clusters = obs_df[cls].cat.categories if hasattr(obs_df[cls], "cat") else obs_df[cls].unique()

for cl in clusters:
    sub = obs_df[obs_df[cls] == cl]
    rrms = sub.loc[sub["catpat"] == "RRMS", "cluster_pct"]
    ctrl = sub.loc[sub["catpat"] == "Control", "cluster_pct"]

    if (len(rrms) > 1) and (len(ctrl) > 1):
        _, p = mannwhitneyu(rrms, ctrl, alternative="two-sided")
        pvals[cl] = p
    else:
        pvals[cl] = np.nan

# --- Plot ---
plt.figure(figsize=(10, 6))

ax = sb.boxplot(
    data=obs_df,
    x=cls,
    y="cluster_pct",
    hue="catpat",
    dodge=True,
    showfliers=False,
    linewidth=1
)

sb.stripplot(
    data=obs_df,
    x=cls,
    y="cluster_pct",
    hue="catpat",
    dodge=True,
    jitter=0.2,
    alpha=0.5,
    size=5,
    linewidth=0,
    ax=ax
)

# Fix double legend
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[:2], labels[:2], title="catpat", frameon=False)

ax.set_xlabel("Leiden cluster")
ax.set_ylabel("Percentage")
plt.xticks(rotation=45, ha="right")

# --- Annotate p-values ---
ymax = obs_df.groupby(cls)["cluster_pct"].max()

for i, cl in enumerate(clusters):
    p = pvals.get(cl, np.nan)
    if np.isnan(p):
        continue

    y = ymax[cl] * 1.08
    ax.text(
        i, y,
        p_to_star(p),
        ha="center",
        va="bottom",
        fontsize=10,
        fontweight="bold"
    )
    ax.plot(
        [i - 0.2, i + 0.2],
        [y * 0.98, y * 0.98],
        lw=1,
        c="black"
    )

plt.tight_layout()
plt.show()

pvals_df = (
    pd.DataFrame.from_dict(pvals, orient="index", columns=["p_value"])
    .reset_index()
    .rename(columns={"index": cls})
)
print(pvals_df)


# Report UMAPs

In [ ]:
wishlistv2 = [
    "IFNG", "TNF", "CCL5", "TNFSF13B", 
    "IL2", "IL4", "IL21", "IL10", "IL17A", "CSF2", "PDCD1",
    "KLF2", "LAG3", "VSIR", "prot_CD4_adt", "prot_CD62L_adt",
    "prot_CD45RA_adt", "prot_CD95_adt",  "CCR2", "CCR4", 
    "CCR5", "CCR6", "CXCR3", "CXCR4"
]

sc.settings.set_figure_params(figsize=(4.3, 4))
sc.pl.umap(sdata["gex"], color=wishlistv2, size = 15, legend_fontweight= 'bold', frameon= False ,legend_fontoutline=2, show=True, legend_loc="on data", cmap="RdBu_r", return_fig=True)
plt.savefig("../../05_plots/15_CD4annotation/10_CD4_umap_wishlistV2_A.pdf", bbox_inches="tight",  dpi=400)

In [ ]:
wishlistv2 = [
        "CXCR5", "CXCR6", "CX3CR1", "GPR183", 
    "S1PR1",  "ITGA1", "ITGA4", "ITGB1",
    "ITGB2", "ITGB7", "ITGAE", "SIRPG", 
    "ICAM3", "CD99", "CD58", "LGALS1",
    "LGALS3",
]

sc.settings.set_figure_params(figsize=(4.3, 4))
sc.pl.umap(sdata["gex"], color=wishlistv2, size = 15, legend_fontweight= 'bold', frameon= False ,legend_fontoutline=2, show=True, legend_loc="on data", cmap="RdBu_r", return_fig=True)
plt.savefig("../../05_plots/15_CD4annotation/10_CD4_umap_wishlistV2_B.pdf", bbox_inches="tight",  dpi=400)

In [ ]:
wishlistv2 = [
    "THEMIS", "HCST", "LCK", "CD4",
    "EOMES", "TOX", "ZEB2",
    "TBX21", "GATA3", "RORC", "FOXP3", "BCL6",
    "CD27", "CD28", "ICOS",
    "GZMA", "GZMB", "GZMH", "GZMK", 
    "GZMM", "PRF1", "NKG7", "FASLG", "KLRB1"
]

sc.settings.set_figure_params(figsize=(4.3, 4))
sc.pl.umap(sdata["gex"], color=wishlistv2, size = 15, legend_fontweight= 'bold', frameon= False ,legend_fontoutline=2, show=True, legend_loc="on data", cmap="RdBu_r", return_fig=True)
plt.savefig("../../05_plots/15_CD4annotation/10_CD4_umap_wishlistV2_C.pdf", bbox_inches="tight",  dpi=400)

In [ ]:
wishlistv2 = [
    "TCF7", "CCR7", "SELL", "IL7R", 
    "LTB", "LEF1", "BACH2", "FAS",
    "TNFRSF4", "IL2RA", "NFKBIA", "NFKB2", "JUNB", 
    "FOS", "BATF3", "CD69", "NR4A1",
    "EGR1", "PASK", "ZFP36L2", "NR3C1"
]

sc.settings.set_figure_params(figsize=(4.3, 4))
sc.pl.umap(sdata["gex"], color=wishlistv2, size = 15, legend_fontweight= 'bold', frameon= False ,legend_fontoutline=2, show=True, legend_loc="on data", cmap="RdBu_r", return_fig=True)
plt.savefig("../../05_plots/15_CD4annotation/10_CD4_umap_wishlistV2_D.pdf", bbox_inches="tight",  dpi=400)

In [ ]:
wishlistv2 = [
    "H1FX", "TXNIP", "HCST", "S100A4", "CD52", "ZFP36", "prot_CD4_adt"
]

sc.settings.set_figure_params(figsize=(4.3, 4))
sc.pl.umap(sdata["gex"], color=wishlistv2, size = 15, legend_fontweight= 'bold', frameon= False ,legend_fontoutline=2, show=True, legend_loc="on data", cmap="RdBu_r", return_fig=True)
plt.savefig("../../05_plots/15_CD4annotation/10_CD4_umap_wishlistV2_E.pdf", bbox_inches="tight",  dpi=400)

In [ ]:
fig_markers = sc.pl.umap(sdata["gex"], color=["leiden_1.7", "chain_pairing"], size = 10, legend_fontweight= 'bold', frameon= False , legend_fontoutline=2, cmap="RdBu_r", return_fig=True)
fig_markers.savefig("../../05_plots/15_CD4annotation/14_Leiden_and_TCRinfo.pdf", bbox_inches="tight",  dpi=400)
plt.show()

In [ ]:
#color dictionary
Compartment_map = {
    "PB": "#C06A6A",   # red
    "CSF":  "#4B8AC1",   # blue
}

In [ ]:
sc.settings.set_figure_params(figsize=(4.3, 4))
# Create the plot but do NOT show it
sc.pl.umap(
    sdata["gex"],
    color="Compartment",
    palette=Compartment_map,
    size=10,
    legend_fontweight="bold",
    legend_fontsize=6,
    frameon=False,
    show=False,
    legend_loc="on data",
    title=""
)

# Save exactly like your example
plt.tight_layout()
plt.savefig(
    "../../05_plots/15_CD4annotation/15_Compartment_CSF_PB_umap.pdf",
    bbox_inches="tight",
    dpi=400
)
plt.show()

# Report Signatures

In [ ]:
list_of_sigs = ["Exclusive to type I interferon signaling", "Exclusive to type II interferon signaling", "Tr1+ cells (De Simone et al 2021)", "Tfh (Ostkampf et al. 2022)", "Th17 (Ostkampf et al. 2022)", "Th1 (Ostkampf et al. 2022)", "Tregs (Ostkampf et al. 2022)", "Th2 (Ostkampf et al. 2022)", "CD4 TEMRA (Ostkampf et al. 2022)", "Th22 (Ostkampf et al. 2022)", "TRM (Ostkampf et al. 2022)","TRM CD103+ (Ostkampf et al. 2022)"]


In [ ]:
sc.settings.set_figure_params(figsize=(4.3, 4))
sc.pl.umap(sdata["gex"], color=list_of_sigs, size = 15, legend_fontweight= 'bold', frameon= False ,legend_fontoutline=2, show=True, legend_loc="on data", cmap="RdBu_r", return_fig=True)
plt.savefig("../../05_plots/15_CD4annotation/17_Wishlist_of_signatures.pdf", bbox_inches="tight",  dpi=400)

# Repertoire information

### Step 1: Transfer important information and make sure the clone_handle does not contain artefacts

In [ ]:
#Make into categorical columns
sdata["gex"].obs["donor"] = sdata["gex"].obs["donor"].astype("category")

#Transfer important information
sdata["gex"].obs["IR_VDJ_1_junction_aa"] = sdata["airr"].obs["IR_VDJ_1_junction_aa"].copy()

# Transfer remaining important information
sdata["airr"].obs["donor"] = sdata["gex"].obs["donor"].copy()
sdata["airr"].obs["pool"] = sdata["gex"].obs["pool"].copy()
sdata["airr"].obs["Compartment"] = sdata["gex"].obs["Compartment"].copy()
sdata["airr"].obs["clone_handle"] = sdata["gex"].obs["clone_handle"].copy()

# If clone_id is nan then clone_handle is also na
sdata["gex"].obs["clone_handle"] = np.where(sdata["gex"].obs["clone_id"].isna(), None, sdata["gex"].obs["clone_handle"])

sdata.update()

### Step 2: Make sure the GEX, AIRR and PROT modalities represent the same cells found in GEX

In [ ]:
#Make sure only the indexes recorded in GEX are used in the o*ther modalities too
index_list = sdata["gex"].obs.index.tolist()

## define a cell index
sdata["airr"].obs["cell_index"] = sdata["airr"].obs.index
sdata["prot"].obs["cell_index"] = sdata["prot"].obs.index

## ".isin" using indexes does not seem to work on a global level, but if I annotate a column and then filter in place, maybe then it works. 
sdata["airr"].obs["filter_index"] = pd.Categorical(np.where(sdata["airr"].obs["cell_index"].isin(index_list), "GEX_present", "GEX_absent"))
sdata["prot"].obs["filter_index"] = pd.Categorical(np.where(sdata["prot"].obs["cell_index"].isin(index_list), "GEX_present", "GEX_absent"))

#Enforce that sdata["airr"].obs and sdata["prot"].obs only contain information on those indexes that exists in GEX
mu.pp.filter_obs(sdata["airr"], 'filter_index', lambda x: x == "GEX_present")
mu.pp.filter_obs(sdata["prot"], 'filter_index', lambda x: x == "GEX_present")

### Step 3: Introduce L3_ann_overview

In [ ]:
# Introduce L3_ann_overview globally: 
sdata["gex"].obs["L3_ann_overview"] = pd.Categorical(np.where(sdata["gex"].obs["leiden_1.7"].isin(["0", "1", "3", "5"]), "TN", 
                                                                       np.where(sdata["gex"].obs["leiden_1.7"].isin(["11"]), "TCM", 
                                                                             np.where(sdata["gex"].obs["leiden_1.7"].isin(["2", "4"]), "TCM/TEM", 
                                                                                      np.where(sdata["gex"].obs["leiden_1.7"].isin(["12"]), "TEM", 
                                                                                               np.where(sdata["gex"].obs["leiden_1.7"].isin(["18"]), "TC", 
                                                                                                         np.where(sdata["gex"].obs["leiden_1.7"].isin(["14"]), "Treg", 
                                                                                                                  np.where(sdata["gex"].obs["leiden_1.7"].isin(["13", "17", "20"]), "TAct", 
                                                                                                                      np.where(sdata["gex"].obs["leiden_1.7"].isin(["6", "7", "8", "9", "10", "15", "16", "19"]), "TCSF", "error")))))))))
sdata.update()

In [ ]:
sdata["gex"].obs["L3_ann_overview"].value_counts()

### Step 4: Introduce catpat

In [ ]:
#Group annotation
patient_list = ["Pt4", "Pt11", "Pt12", "Pt14", "Pt17", "Pt20"]
control_list = ["Pt1", "Pt5", "Pt8", "Pt18", "Pt19", "Pt21"]

#Match the lists defined above to define "RRMS" and "Control"
sdata["gex"].obs["catpat"] = pd.Categorical(np.where(sdata["gex"].obs["donor"].isin(patient_list), "RRMS", np.where(sdata["gex"].obs["donor"].isin(control_list), "Control", "Error")))


### Step 5: Define clonotypes and type_x_clone_count (compartment-wise per clonotype)

In [ ]:
#Transfer labels from the airr to gex modality
# sdata["gex"].obs["antigen.species"] = sdata["airr"].obs["antigen.species"].copy()
# sdata["gex"].obs["antigen.epitope"] = sdata["airr"].obs["antigen.epitope"].copy()

#CDR3aa
sdata["gex"].obs["IR_VDJ_1_junction_aa"] = sdata["airr"].obs["IR_VDJ_1_junction_aa"].copy()
sdata["gex"].obs["IR_VJ_1_junction_aa"] = sdata["airr"].obs["IR_VJ_1_junction_aa"].copy()

#TRAV and TRBV
sdata["gex"].obs["IR_VJ_1_v_call"] = sdata["airr"].obs["IR_VJ_1_v_call"].copy()
sdata["gex"].obs["IR_VDJ_1_v_call"] = sdata["airr"].obs["IR_VDJ_1_v_call"].copy()

#D regions
sdata["gex"].obs["IR_VDJ_1_d_call"] = sdata["airr"].obs["IR_VDJ_1_d_call"].copy()

#J regions
sdata["gex"].obs["IR_VJ_1_j_call"] = sdata["airr"].obs["IR_VJ_1_j_call"].copy()
sdata["gex"].obs["IR_VDJ_1_j_call"] = sdata["airr"].obs["IR_VDJ_1_j_call"].copy()


In [ ]:
sdata_backup = sdata.copy()
airr_export = sdata["airr"].obs[["pool","donor","Compartment", "clone_id", "clone_handle",
                   'IR_VJ_1_junction_aa', 'IR_VDJ_1_junction_aa',
                   'IR_VJ_1_v_call', 'IR_VDJ_1_v_call',
                   'IR_VJ_1_j_call', 'IR_VDJ_1_j_call',
                   'IR_VDJ_1_d_call',
                   'IR_VJ_2_junction_aa', 'IR_VDJ_2_junction_aa',
                   'IR_VJ_2_v_call',  'IR_VDJ_2_v_call', 
                   'IR_VJ_2_j_call', 'IR_VDJ_2_j_call',
                   'IR_VDJ_2_d_call']]

# Drop rows where 'clone_handle' is NaN
airr_export = airr_export.dropna(subset=['Compartment']) #Subset tells pandas to dropna only for Comparment
airr_export = airr_export.dropna(subset=['clone_id']) #Subset tells pandas to dropna only for clone_id

## First generate a new clone ID label that is a concatenation of donor and clone_id
airr_export["clone_handle"] #Already generated

## Second generate a summary table with counts per clone_handle per compartment - clone_handles are used here to avoid counting public shared by donors together
counts_per_id = airr_export.groupby(["Compartment", 'clone_handle']).size().reset_index(name='type_x_clone_count')

# Merge the count information back into the original DataFrame
airr_export = pd.merge(airr_export, counts_per_id, on=["Compartment", 'clone_handle'], how='left')

#### Clean the data frame

In [ ]:
airr_export_backup = airr_export.copy()
airr_export = airr_export[["pool", "donor", "Compartment", "clone_id", "clone_handle", "type_x_clone_count", "IR_VJ_1_junction_aa", "IR_VDJ_1_junction_aa"]]

### Step 6: A venn diagram -- non-naive clonotypes

In [ ]:
## Non naive clone handles
NonNaive_set = set(sdata["gex"].obs[~sdata["gex"].obs["leiden_1.7"].isin(["0", "1", "3", "5"])]["clone_handle"])

In [ ]:
from matplotlib_venn import venn2

# Create subplots
fig, axes = plt.subplots(1, 1, figsize=(5, 5))

# %%% Patient 1 %%%%
# Extract unique values from each column
set1 = set(airr_export[airr_export['pool'].isin(["Pt1_PBMC", "Pt4_PBMC", "Pt5_PBMC", "Pt8_PBMC", "Pt11_PBMC", "Pt12_PBMC", "Pt14_PBMC", "Pt17_PBMC", "Pt18_PBMC", "Pt19_PBMC", "Pt20_PBMC", "Pt21_PBMC"])]["clone_handle"])
set2 = set(airr_export[airr_export['pool'].isin(["Pt1", "Pt4", "Pt5", "Pt8", "Pt11", "Pt12", "Pt14", "Pt17", "Pt18", "Pt19", "Pt20", "Pt20+21"])]["clone_handle"])

#Non_naive set 1
set1_nonnaive = set1 & NonNaive_set

# Compute sizes for each section
only_set1 = len(set1_nonnaive - set2)
only_set2 = len(set2 - set1_nonnaive)
intersection = len(set1_nonnaive & set2)

total = only_set1 + only_set2 + intersection

# Compute percentages
pct1 = 100 * only_set1 / total
pct2 = 100 * only_set2 / total
pct_inter = 100 * intersection / total

# Create a Venn diagram
vennAll = venn2(subsets=(only_set1, only_set2, intersection), set_labels=('PB', 'CSF'))

# Modify labels to show percentages
if vennAll.get_label_by_id('10'):
    vennAll.get_label_by_id('10').set_text(f"{only_set1}\n({pct1:.1f}%)")
if vennAll.get_label_by_id('01'):
    vennAll.get_label_by_id('01').set_text(f"{only_set2}\n({pct2:.1f}%)")
if vennAll.get_label_by_id('11'):
    vennAll.get_label_by_id('11').set_text(f"{intersection}\n({pct_inter:.1f}%)")

# Set colors to a colorblind-friendly palette
vennAll.get_patch_by_id('10').set_color('#C06A6A')  # left = Muted red
vennAll.get_patch_by_id('01').set_color('#4B8AC1')  # right = Blue
vennAll.get_patch_by_id('11').set_color('#D89A52')  # overlap = orange

# Adjust transparency for all patches
for venn in [vennAll]:
    for patch in venn.patches:
        if patch:
            patch.set_alpha(1)

# Display the plot
plt.title("Clonotype overlap, CD4 T cells") 
plt.tight_layout()
plt.show()

# Save the figure 
fig.savefig("../../05_plots/15_CD4annotation/16_VennCompartmentOverlap.pdf", bbox_inches='tight', format="pdf", dpi=400)

### Step 7: Binomial test
https://www.statology.org/binomial-test-python/

Example 1: We roll a 6-sided die 24 times and it lands on the number “3” exactly 6 times. Perform a binomial test to determine if the die is biased towards the number “3.”

**binom_test(x=6, n=24, p=1/6, alternative='greater')**

0.1995295129479586

Because this p-value (0.1995) is not less than 0.05, we fail to reject the null hypothesis. We do not have sufficient evidence to say the die is biased towards the number “3.”

#### Can we do the same for compartment resampling
From Kilian'sa advice: 
The clone size corresponds to the probability of sampling a given clone, fx. if the clone size is 30 out of 3000 cells, then the probability is 30/3000 or 1%
For any given clone this corresponds to a binomial distribution with the probability being equal to the clone fraction in PB, and
n being the number of T cells surveyed in the CSF repertoire. A binomal test can be made. 

So: binom_test(
    x=    "how many X clones we saw", 
    n=    "how many cells we sampled in CSF", 
    p=    "a baseline distribution based on resampling PB", 
    alternative='two-sided')

Under the assumption of random clone sharing between PB and CSF, we can assume that CSF can be "created" from the binomal distribution observed in the PB samples. 
The clone size in the PB sample corresponds to the probability of sampling a given clone. For any given clone this corresponds to a binomial distribution with the probability being equal
to the clone fraction in PB and n being the number of T cells surveyed in CSF. Using a binomial test, we can test if the observed size of a given clone in the CSF 
corresponds to the expected under the assumption of random clone sharing (observed CSF fraction should under the null hypothesis be similar to fraction in CSF * N cells in CSF) 
This method only simulates the probabilities of clone sharing. 

(Nikolaj) Why not just use Fischer's exact test?


### Step 7: Binomial test
https://www.statology.org/binomial-test-python/

Example 1: We roll a 6-sided die 24 times and it lands on the number “3” exactly 6 times. Perform a binomial test to determine if the die is biased towards the number “3.”

**binom_test(x=6, n=24, p=1/6, alternative='greater')**

0.1995295129479586

Because this p-value (0.1995) is not less than 0.05, we fail to reject the null hypothesis. We do not have sufficient evidence to say the die is biased towards the number “3.”

#### Can we do the same for compartment resampling
From Kilian'sa advice: 
The clone size corresponds to the probability of sampling a given clone, fx. if the clone size is 30 out of 3000 cells, then the probability is 30/3000 or 1%
For any given clone this corresponds to a binomial distribution with the probability being equal to the clone fraction in PB, and
n being the number of T cells surveyed in the CSF repertoire. A binomal test can be made. 

So: binom_test(
    x=    "how many X clones we saw", 
    n=    "how many cells we sampled in CSF", 
    p=    "a baseline distribution based on resampling PB", 
    alternative='two-sided')

Under the assumption of random clone sharing between PB and CSF, we can assume that CSF can be "created" from the binomal distribution observed in the PB samples. 
The clone size in the PB sample corresponds to the probability of sampling a given clone. For any given clone this corresponds to a binomial distribution with the probability being equal
to the clone fraction in PB and n being the number of T cells surveyed in CSF. Using a binomial test, we can test if the observed size of a given clone in the CSF 
corresponds to the expected under the assumption of random clone sharing (observed CSF fraction should under the null hypothesis be similar to fraction in CSF * N cells in CSF) 
This method only simulates the probabilities of clone sharing. 

(Nikolaj) Why not just use Fischer's exact test?


In [ ]:
# Definition of cell total per sample
airr_export['cells_in_sample'] = airr_export.groupby(['pool'])['type_x_clone_count'].transform('size')

# %%%%%%%%%%%%
# Definition of clone_x_fra_PB (used as reference frequency later)
# %%%%%%%%%%%%

## For each clonotype calculate the fraction of cells out of all cells in PB. I.e. the fraction of the dataset made up by the given clone. 
PB_clones = airr_export[airr_export["Compartment"] == "PB"]
PB_clones = PB_clones[["donor", "clone_handle", "type_x_clone_count", "cells_in_sample"]]
PB_clones = PB_clones.drop_duplicates().reset_index()
PB_clones["clone_x_fra_PB"] = PB_clones["type_x_clone_count"] / PB_clones["cells_in_sample"]
PB_clones = PB_clones[["donor", "clone_handle", "clone_x_fra_PB"]]

## merge the summary information back
airr_export = pd.merge(airr_export, PB_clones, on=["donor", 'clone_handle'], how='left')

# Define minimum values as a lower obund too
dfc = PB_clones.groupby('donor')['clone_x_fra_PB']
Min_df = PB_clones.assign(min=dfc.transform(min))

Min_df = Min_df[["donor", "min"]].drop_duplicates()

## merge the min data information back
airr_export = pd.merge(airr_export, Min_df, on=["donor"], how='left')

## Now replace the NaN values in clone_x_fra_PB with the lower bound min value
airr_export['clone_x_fra_PB'] = airr_export['clone_x_fra_PB'].fillna(airr_export.pop('min'))

# %%%%%%%%%%%%
# Definition of clone_x_fra_CSF (used as reference frequency later)
# %%%%%%%%%%%%

## Get the fraction of cells for a given clone from the PB compartment per clone
CSF_clones = airr_export[airr_export["Compartment"] == "CSF"]
CSF_clones = CSF_clones[["donor", "clone_handle", "type_x_clone_count", "cells_in_sample"]]
CSF_clones = CSF_clones.drop_duplicates().reset_index()
CSF_clones["clone_x_fra_CSF"] = CSF_clones["type_x_clone_count"] / CSF_clones["cells_in_sample"]
CSF_clones = CSF_clones[["donor", "clone_handle", "clone_x_fra_CSF"]]

## merge the summary information back
airr_export = pd.merge(airr_export, CSF_clones, on=["donor", 'clone_handle'], how='left')

# Define minimum values as a lower obund too
dfc = CSF_clones.groupby('donor')['clone_x_fra_CSF']
Min_df = CSF_clones.assign(min=dfc.transform(min))

Min_df = Min_df[["donor", "min"]].drop_duplicates()

## merge the min data information back
airr_export = pd.merge(airr_export, Min_df, on=["donor"], how='left')

## Now replace the NaN values in clone_x_fra_PB with the lower bound min value
airr_export['clone_x_fra_CSF'] = airr_export['clone_x_fra_CSF'].fillna(airr_export.pop('min'))

In [ ]:
#How does the binom test work: Example: 
binomtest(6, 24, p=1/6)

In [ ]:
## For CSF cells compared to PB background distribution
airr_export["p_csf"] = airr_export.apply(lambda x: binomtest(x['type_x_clone_count'],   #k - number of successes (number of observed cells in the given compartment)
                                        x['cells_in_sample'],                       #n - number of trials (based on cells in the given compartment)
                                        x['clone_x_fra_PB'],                        #p - null expected likelihood of success (based on freq. in PB)
                                        alternative= "two-sided").pvalue, axis=1)

## For PB cells compared to CSF background distribution
airr_export["p_PB"] = airr_export.apply(lambda x: binomtest(x['type_x_clone_count'],   #k - number of successes (number of observed cells in the given compartment)
                                        x['cells_in_sample'],                       #n - number of trials (based on cells in the given compartment)
                                        x['clone_x_fra_CSF'],                        #p - null expected likelihood of success (based on freq. in PB)
                                        alternative= "two-sided").pvalue, axis=1)

## Merge the two p-values - take the smallest of the two first. 
airr_export["p"] = np.where(airr_export["p_csf"] < airr_export["p_PB"], airr_export["p_csf"], airr_export["p_PB"])

#### Bonferroni adj
The Bonferroni correction can also be applied as a p-value adjustment: Using that approach, instead of adjusting the alpha level, each p-value is multiplied by the number of tests (with adjusted p-values that exceed 1 then being reduced to 1), and the alpha level is left unchanged. The significance decisions using this approach will be the same as when using the alpha-level adjustment approach.

In [ ]:
## Correct the p-value by dividing with the number of test made for every donor
N_tests_df = airr_export[["donor", "Compartment", "clone_handle"]].drop_duplicates().reset_index()
N_tests_df["N_tests"] = N_tests_df.groupby(['donor', "Compartment"])['clone_handle'].transform('size')

# Merge the count information back into the original DataFrame
airr_export = pd.merge(airr_export, N_tests_df, on=["donor","Compartment",'clone_handle'], how='left')

In [ ]:
#Bonferroni p_adj calculation
airr_export["p_adj"] = airr_export["p"]*airr_export["N_tests"]  

#Mask p_adj-value to 1
airr_export['p_adj'] = airr_export['p_adj'].apply(lambda x: 1 if x > 1 else x)

#Generate a label if significant. 
airr_export["Sig_label"] = airr_export['p_adj'].apply(lambda x: "ns" if x >= 0.05 else "q < 0.05")

### Step 8: Biaxial plot w. cell counts

In [ ]:
### First you need the % values to later give color annotations correctly
# Add a column with the total number of observations for each combination
airr_export['cells_in_sample'] = airr_export.groupby(['pool'])['type_x_clone_count'].transform('size')

# Return unique values of two columns is in pandas defined as a "drop_duplicates()" problem
airr_export[["pool", "cells_in_sample"]].drop_duplicates() #Number of cells in sample seems realisitic - although 22 and 87 cells in CSF is going to be a problem...

#Define the percentage-wise observation from columns you have
airr_export['type_x_clone_pct'] = (airr_export['type_x_clone_count'] / airr_export['cells_in_sample']) * 100

# General columns to keep
cols = ["Compartment", "clone_handle", "type_x_clone_pct"] 

# %%% Patient samples %%%
PT_all = airr_export # All of it [airr_export["pool"].isin(["Pt1_PBMC", "Pt1", "Pt4_PBMC", "Pt4", "Pt5", "Pt5_PBMC", "Pt8", "Pt8_PBMC", "Pt11", "Pt11_PBMC", "Pt12", "Pt12_PBMC"])]
PT_all = PT_all[cols]
PT_all = PT_all.drop_duplicates(subset=cols)

#df.pivot(index='foo', columns='bar', values='baz')
PT_all = PT_all.pivot(index=["clone_handle"], columns=["Compartment"], values="type_x_clone_pct").fillna(0).reset_index()

#Rename to make merging easier later
Pct_df = PT_all.rename(columns={
    "CSF":"Pct_CSF", 
    "PB": "Pct_PB"
})

In [ ]:
# General columns to keep
cols = ["Compartment", "clone_handle", "type_x_clone_count", "Sig_label"]

# Assign colors based on categories
colors = {'ns': 'grey', 'q < 0.05': "red"}

# %%% Patient samples %%%
PT_all = airr_export
PT_all = PT_all[cols]
PT_all = PT_all.drop_duplicates(subset=cols)

# Pivot table
PT_all = PT_all.pivot(index=["clone_handle", "Sig_label"], columns=["Compartment"], values="type_x_clone_count").fillna(0).reset_index()

PT_all = PT_all.merge(Pct_df, how="left", on="clone_handle")

#Define color
PT_all["color"] = np.where((PT_all["Sig_label"] != "ns") & (PT_all["Pct_CSF"] > PT_all["Pct_PB"]), "CSF-enriched",
                          np.where((PT_all["Sig_label"] != "ns") & (PT_all["Pct_CSF"] < PT_all["Pct_PB"]), "PB-enriched", "ns"))

In [ ]:
# Create subplots
fig, axes = plt.subplots(1, 1, figsize=(6.75, 4))

#color dictionary
color_map = {
    "PB-enriched": "#C06A6A",   # red
    "CSF-enriched":  "#4B8AC1",   # blue
    "ns":           "lightgrey"
}


# Plotting
x = PT_all["CSF"]
y = PT_all["PB"]

# Rasterize scatter plot & place grid behind data
axes.set_axisbelow(True)
scatter = axes.scatter(
    x=PT_all["CSF"],
    y=PT_all["PB"],
    alpha=1,
    s=50,
    c=PT_all["color"].map(color_map),
    edgecolors="white",
    linewidths=0.2,   # slim white border
    rasterized=True
)

# Labels and grid
# axes.set_ylim(-55,1300)
# axes.set_xlim(-5, 90)
axes.set_xlabel(r'CSF - No. of cells', fontsize=15)
axes.set_ylabel(r'PB - No. of cells', fontsize=15)
axes.set_title('Clonotype overlap, CD4 T cells, CSF vs PB T cells')
axes.grid(False)
sb.despine()

# Legend (explicitly tied to color_map)
legend_handles = [
    mpatches.Patch(color=color_map[label], label=label)
    for label in ["CSF-enriched", "PB-enriched", "ns"]
    if label in color_map
]

axes.legend(
    handles=legend_handles,
    title="Significance",
    bbox_to_anchor=(1.05, 1),
    loc="upper left",
    borderaxespad=0.
)

# Layout and save
plt.tight_layout(rect=[0, 0, 0.85, 1])
fig.savefig("../../05_plots/16_CD4analysis/1_CD4_XY_expansion.pdf",
            bbox_inches='tight', format="pdf", dpi=400)
plt.show()

#### Save the statistical output

In [ ]:
#Is clone counts in airr_export the same as airr_backup - yes
len(airr_export[airr_export_backup["type_x_clone_count"] == airr_export["type_x_clone_count"]]) == len(airr_export)

### Step 9: Add back the clonal expansion information to the mudata object (sdata)

In [ ]:
#Generating a total count - adding back in sdata

# Create deduplicated DataFrame
airr_total_count = (
    airr_export[["clone_handle", "Compartment", "type_x_clone_count", "p_csf", "p_PB", "p_adj", "Sig_label"]]
    .drop_duplicates()
    .sort_values(by="clone_handle")
)

# Add total_clone_count column using transform to preserve row count
airr_total_count["total_clone_count"] = (
    airr_total_count
    .groupby("clone_handle")["type_x_clone_count"]
    .transform("sum")
)

# (Optional) Sort by total_clone_count
airr_total_count = airr_total_count.sort_values(by="total_clone_count", ascending=False)
airr_total_count

In [ ]:
## %%
# Add "type_x_clone_count" as well as "total_clone_count" back into the single-cell object
# %%

sdata["gex"].obs = sdata["gex"].obs.reset_index().merge(airr_total_count, on=["clone_handle", "Compartment"], how="left").set_index('index')  
  #Note: The reset_index to .set_index("index") works to save the current index of the left data frame, and then use that later in set_index() again. 
  #It's a common problem described here: https://stackoverflow.com/questions/11976503/how-to-keep-index-when-using-pandas-merge

sdata.update()

# Export L3_annotations

In [ ]:
sdata.write(filename="..\\..\\04_data_objects\\01_preprocessed_scseq_scanpy\\2025_12 - with EBV collected\\2026_4_CD4_annotated.h5mu")

# Graveyard

In [ ]:
sc.settings.set_figure_params(figsize=(4.3, 4))
fig_markers = sc.pl.umap(sdata["gex"], color=["CD4", "prot_CD4_adt", "CD8A", "CD8B", "prot_CD8_adt", "CD3E", "THEMIS", "HCST", "TYROBP", "LCK"], size = 10, legend_fontweight= 'bold', frameon= False , legend_fontoutline=2, cmap="RdBu_r", return_fig=True)
fig_markers.savefig("../../05_plots/15_CD4annotation/12_Cytokines_and_CD4CD8.pdf", bbox_inches="tight",  dpi=400)
plt.show()

In [ ]:
sc.settings.set_figure_params(figsize=(4.3, 4))
fig_markers = sc.pl.umap(sdata["gex"], color=["TCF7","CCR7", "SELL",  "prot_CD62L_adt", "LEF1", "CD27", "CD28", "IL7R", "LTB", "KLF2", "BACH2", "PTPRC", "prot_CD45RA_adt","FAS","prot_CD95_adt", "CD58", "ITGA4"], size = 10, legend_fontweight= 'bold', frameon= False , legend_fontoutline=2, cmap="RdBu_r", return_fig=True)
fig_markers.savefig("../../05_plots/15_CD4annotation/12_CD4_NaiveAndCM.pdf", bbox_inches="tight",  dpi=400)
plt.show()

In [ ]:
fig_markers = sc.pl.umap(sdata["gex"], color=["TNFRSF4","IL2RA", "NFKBIA", "NFKB2","TNF", "JUN", "FOS", "BATF3", "CD69", "NR4A1"], size = 10, legend_fontweight= 'bold', frameon= False , legend_fontoutline=2, cmap="RdBu_r", return_fig=True)
fig_markers.savefig("../../05_plots/15_CD4annotation/13_cannonical_activation.pdf", bbox_inches="tight",  dpi=400)
plt.show()

In [ ]:
fig_markers = sc.pl.umap(sdata["gex"], color=["GZMK","GZMA", "GZMH", "GZMB", "PRF1", "IFNG", "TNF", "IL2", "IL13", "IL17A", "IL10", "IL4", "IL21", "IL17A", "IL10", "TGFB1", "TGFB2", "TGFB3", "IL7", "IL15"], size = 10, legend_fontweight= 'bold', frameon= False , legend_fontoutline=2, cmap="RdBu_r", return_fig=True)
fig_markers.savefig("../../05_plots/15_CD4annotation/13_effector_genes.pdf", bbox_inches="tight",  dpi=400)
plt.show()

In [ ]:
fig_markers = sc.pl.umap(sdata["gex"], color=["CXCR3", "CXCR4", "CXCR5", "CXCR6", "CCR2", "CCR4", "CCR5", "CCR6", "CX3CR1", "GPR183", "CCR7", "S1PR1"], size = 10, legend_fontweight= 'bold', frameon= False , legend_fontoutline=2, cmap="RdBu_r", return_fig=True)
fig_markers.savefig("../../05_plots/15_CD4annotation/13_chemokinereceptors_genes.pdf", bbox_inches="tight",  dpi=400)
plt.show()

In [ ]:
fig_markers = sc.pl.umap(sdata["gex"], color=["ITGA4", "ITGB1", "ITGB2", "ITGB7", "ITGAE", "ITGAX", "ITGAM", "SIRPG", "ICAM3", "PECAM1", "CD99", "CD58", "LGALS1", "LGALS3",  "CD226"], size = 10, legend_fontweight= 'bold', frameon= False , legend_fontoutline=2, cmap="RdBu_r", return_fig=True)
fig_markers.savefig("../../05_plots/15_CD4annotation/13_intergrins_genes.pdf", bbox_inches="tight",  dpi=400)
plt.show()

In [ ]:
fig_markers = sc.pl.umap(sdata["gex"], color=["TBX21","GATA3", "RORC", "FOXP3", "BCL6", "EOMES", "TOX", "ZEB2"], size = 10, legend_fontweight= 'bold', frameon= False , legend_fontoutline=2, cmap="RdBu_r", return_fig=True)
fig_markers.savefig("../../05_plots/15_CD4annotation/13_MasterRegulators.pdf", bbox_inches="tight",  dpi=400)
plt.show()

In [ ]:
fig_markers = sc.pl.umap(sdata["gex"], color=["EGR1", "PASK", "AC004448.2", "TNFSF13B"], size = 10, legend_fontweight= 'bold', frameon= False , legend_fontoutline=2, cmap="RdBu_r", return_fig=True)
fig_markers.savefig("../../05_plots/15_CD4annotation/13_Other.pdf", bbox_inches="tight",  dpi=400, format="pdf")
plt.show()

In [ ]:
fig_markers = sc.pl.umap(sdata["gex"], color=["DKK3", "CD28", "CD27", "NELL2", "GPR183", "CCR7", "GZMK", "TNFSF8", "COTL1", "CLDND1", "CXCR4", "CXCR3", "PDCD1", "CCR5"], size = 10, legend_fontweight= 'bold', frameon= False , legend_fontoutline=2, cmap="RdBu_r", return_fig=True)
fig_markers.savefig("../../05_plots/15_CD4annotation/13_MarkersFromCD8CSFbias.pdf", bbox_inches="tight",  dpi=400, format="pdf")
plt.show()